# 🏆 Churn & Insurance Analytics — Master Notebook

**Template lengkap untuk case: customer churn / policy lapse di perusahaan asuransi.**
Notebook ini dirancang supaya **data apapun yang dikasih panitia** (asal ada kolom target churn) bisa langsung diolah dari awal sampai akhir:

```text
Load data (1 / banyak tabel)  →  Data Quality & Cleaning otomatis  →  Target parsing
→  Domain Feature Engineering (asuransi)  →  EDA + Driver Analysis (statistik + effect size + IV)
→  Segment / Interaction Analysis  →  Survival Analysis (Kaplan-Meier, log-rank, Cox)
→  Interpretable Model (Logistic Regression odds ratio)  →  Leakage & Drift Check
→  ML Modeling (LogReg, RF, ET, HGB, LightGBM, XGBoost, CatBoost) dengan CV OOF
→  Optuna Tuning  →  Multi-seed  →  Ensemble (hill-climbing, rank, stacking)
→  Threshold & Calibration  →  Debugging (learning curve, overfit, shuffled target, error analysis)
→  Feature Importance (gain, permutation, SHAP, PDP)  →  Feature Selection
→  Business Analytics (lift, gains, risk tier, CLV, risk×value, campaign ROI, what-if, personas)
→  Rekomendasi otomatis  →  Submission + Validator  →  Export tabel/figure/insight untuk laporan
```

## Prinsip desain
1. **Tidak gampang error.** Semua analisis opsional dibungkus `safe_section(...)` → kalau satu section gagal, notebook tetap jalan, error dicatat di `SECTION_ERRORS` dan dirangkum di akhir.
2. **Semua dievaluasi dengan Out-Of-Fold (OOF).** Angka score yang lu lihat = estimasi jujur performa di data baru.
3. **Tidak ada leakage preprocessing.** Encoding yang pakai target (target encoding) selalu di-fit di dalam fold.
4. **Report-ready.** Semua figure disimpan otomatis (bernomor) di `outputs/.../figures`, semua tabel di `tables/` + 1 file Excel, dan insight teks otomatis di `insights.md`.
5. **Panduan lengkap.** Setiap section ada: tujuan, cara baca output, aturan **"Jika X → lakukan Y"**, dan catatan **"Untuk laporan"**.

## Cara pakai cepat (5 langkah)
1. Taruh data di folder `./data/` (atau ubah path di cell **USER OVERRIDE**).
2. Di cell **USER OVERRIDE**: isi `TRAIN_PATH`, `TEST_PATH`, `SAMPLE_SUB_PATH`. `TARGET_COL` dan `ID_COL` boleh `None` (auto-detect), tapi **lebih aman diisi manual**.
3. Set `CFG.METRIC` sesuai metrik lomba (misal `"roc_auc"`, `"f1"`, `"accuracy"`).
4. Jalankan dulu dengan `CFG.RUN_MODE = "fast"` (± beberapa menit) → cek semua jalan & baca hasil EDA.
5. Ganti ke `CFG.RUN_MODE = "full"` → **Kernel → Restart & Run All** → ambil submission + figure + tabel untuk laporan.

## Strategi waktu saat final (saran)
| Fase | Waktu | Aktivitas |
|---|---|---|
| 1 | 0:00–0:20 | Baca soal, set config, run **fast mode**, pastikan tidak error |
| 2 | 0:20–1:00 | Baca EDA, driver table, survival, segment rules → tulis draft insight |
| 3 | 1:00–2:00 | Run **full mode** (modeling + tuning jalan di background) sambil mulai tulis laporan |
| 4 | 2:00–... | Ambil SHAP, business analytics, rekomendasi → finalisasi laporan & submission |

> Jika waktu sangat mepet: set `CFG.USE_OPTUNA = False`, `CFG.MODELS = ["lgbm", "catboost", "logreg"]`, `CFG.N_FOLDS = 5` → sudah cukup kuat.

## 📑 Daftar Isi
- **0. Setup** — install, import, helper
- **1. Config** — `CFG` + **USER OVERRIDE**
- **2. Load Data** — single / multi-table, merge & agregasi
- **3. Data Quality & Cleaning** — tipe kolom, angka-dalam-string, tanggal, kategori typo, nilai mustahil, duplikat
- **4. Target** — parsing label churn, churn rate, baseline
- **5. Column Roles & Domain Feature Engineering (asuransi)**
- **6. EDA Univariate**
- **7. Driver Analysis** — churn rate per kategori/bin + uji statistik + effect size + IV + ranking driver
- **8. Korelasi & Multikolinearitas** — Spearman, Cramér's V, VIF
- **9. Segment & Interaction Analysis** — heatmap 2-arah, decision-tree rules, top risky segments
- **10. Survival Analysis** — Kaplan-Meier, log-rank, hazard per tenure, Cox PH
- **11. Interpretable Model** — Logistic Regression (odds ratio + CI)
- **12. Leakage & Drift Check** — single-feature AUC, adversarial validation
- **13. Modeling Setup** — matriks fitur, CV folds, metric
- **14. Model Training (CV OOF)**
- **15. Hyperparameter Tuning (Optuna)**
- **16. Multi-seed Final Models + Feature Selection**
- **17. Ensemble**
- **18. Threshold & Calibration**
- **19. Evaluation & Debugging** — ROC/PR, confusion matrix, learning curve, shuffled-target test, error analysis
- **20. Feature Importance & Explainability** — gain, permutation, SHAP, dependence, waterfall, PDP
- **21. Business Analytics** — lift/gains, risk tier, CLV, risk×value, campaign ROI, what-if, persona
- **22. Recommendations (otomatis)**
- **23. Submission + Validator**
- **24. Export untuk Laporan** — Excel, insights.md, executive summary figure, **auto REPORT_DRAFT.docx + REPORT_TODO.md**, ZIP deliverables
- **25. Troubleshooting, Recipes, Checklist**

# 0. Setup

## 0.1 Install package yang belum ada
Laptop panitia belum tentu punya semua library. Cell ini cek package; kalau `CFG_AUTO_INSTALL = True` dan ada yang hilang → `pip install` otomatis.

- **Wajib**: numpy, pandas, scikit-learn, scipy, matplotlib, seaborn.
- **Sangat disarankan**: lightgbm, xgboost, catboost, statsmodels (statistik & survival), optuna (tuning), shap (plot), openpyxl (export Excel).
- **Opsional**: lifelines (survival lebih lengkap), imbalanced-learn (SMOTE).

**Jika install gagal** (tidak ada internet / permission): tidak apa-apa, notebook tetap jalan; model/analisis yang butuh library itu otomatis di-skip.
Kalau mau install manual di terminal: `pip install lightgbm xgboost catboost optuna shap statsmodels openpyxl`.

> Setelah install package baru, kadang perlu **Kernel → Restart** lalu run ulang dari atas.

In [ ]:
CFG_AUTO_INSTALL = True   # ubah ke False kalau tidak mau notebook menginstall apapun

import importlib
import importlib.util
import subprocess
import sys

REQUIRED = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "scipy": "scipy",
            "matplotlib": "matplotlib", "seaborn": "seaborn"}
RECOMMENDED = {"lightgbm": "lightgbm", "xgboost": "xgboost", "catboost": "catboost",
               "statsmodels": "statsmodels", "optuna": "optuna", "shap": "shap", "openpyxl": "openpyxl"}
OPTIONAL = {"lifelines": "lifelines", "imblearn": "imbalanced-learn", "docx": "python-docx"}


def ensure_packages(pkgs, install=True):
    """Cek package via find_spec; install yang belum ada (kalau install=True). Tidak pernah raise."""
    missing = [imp for imp in pkgs if importlib.util.find_spec(imp) is None]
    for imp in missing:
        if not install:
            print(f"  - {imp}: TIDAK ADA (auto-install off)")
            continue
        try:
            print(f"  - installing {pkgs[imp]} ...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkgs[imp]])
            importlib.invalidate_caches()
        except Exception as e:  # noqa
            print(f"    gagal install {pkgs[imp]}: {e}")
    return [imp for imp in pkgs if importlib.util.find_spec(imp) is None]


print("Cek package wajib...")
still_missing = ensure_packages(REQUIRED, CFG_AUTO_INSTALL)
if still_missing:
    raise ImportError(f"Package wajib tidak tersedia: {still_missing}. Install manual dulu.")
print("Cek package disarankan...")
ensure_packages(RECOMMENDED, CFG_AUTO_INSTALL)
print("Cek package opsional (boleh gagal)...")
ensure_packages(OPTIONAL, CFG_AUTO_INSTALL)
print("Selesai.")

## 0.2 Import & availability flags
Setiap library opsional punya flag `HAS_xxx`. Kalau `False`, bagian yang butuh library itu di-skip dengan pesan jelas — **bukan error**.

In [ ]:
import os
import re
import gc
import json
import math
import time
import random
import warnings
import traceback
import itertools
from contextlib import contextmanager
from collections import OrderedDict

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.dummy import DummyClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (roc_auc_score, average_precision_score, log_loss, brier_score_loss,
                             accuracy_score, f1_score, precision_score, recall_score, fbeta_score,
                             balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score,
                             confusion_matrix, classification_report, roc_curve, precision_recall_curve,
                             silhouette_score, mutual_info_score)
from sklearn.feature_selection import mutual_info_classif
from sklearn.isotonic import IsotonicRegression

try:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_HGB = True
except Exception:
    HAS_HGB = False

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except Exception:
    HAS_SGKF = False

try:
    from IPython.display import display, Markdown
except Exception:  # di luar Jupyter
    display = print
    Markdown = str


def _try_import(name):
    try:
        return importlib.import_module(name)
    except Exception:
        return None


lgb = _try_import("lightgbm")
xgb = _try_import("xgboost")
cb = _try_import("catboost")
optuna = _try_import("optuna")
shap = _try_import("shap")
sm = _try_import("statsmodels.api")
lifelines = _try_import("lifelines")
imblearn = _try_import("imblearn")

HAS_LGB, HAS_XGB, HAS_CB = lgb is not None, xgb is not None, cb is not None
HAS_OPTUNA, HAS_SHAP, HAS_SM = optuna is not None, shap is not None, sm is not None
HAS_LIFELINES, HAS_IMBLEARN = lifelines is not None, imblearn is not None
HAS_OPENPYXL = importlib.util.find_spec("openpyxl") is not None

if HAS_SM:
    try:
        from statsmodels.duration.survfunc import SurvfuncRight, survdiff
        from statsmodels.duration.hazard_regression import PHReg
        from statsmodels.stats.multitest import multipletests
        from statsmodels.stats.outliers_influence import variance_inflation_factor
        HAS_SM_SURV = True
    except Exception:
        HAS_SM_SURV = False
else:
    HAS_SM_SURV = False

if HAS_OPTUNA:
    optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

print("Python      :", sys.version.split()[0])
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("sklearn     :", sklearn.__version__)
print("scipy       :", scipy.__version__)
for _name, _mod in [("lightgbm", lgb), ("xgboost", xgb), ("catboost", cb), ("optuna", optuna),
                    ("shap", shap), ("statsmodels", _try_import("statsmodels")), ("lifelines", lifelines),
                    ("imblearn", imblearn)]:
    print(f"{_name:<12}:", getattr(_mod, "__version__", "OK") if _mod is not None else "— tidak tersedia (bagian terkait di-skip)")

## 0.3 Helper functions umum
Isi cell ini: reproducibility, `safe_section` (anti-crash), penyimpan figure/tabel, pencatat insight, format angka, Wilson CI, dan compat shim untuk beda versi library.

**Cara kerja `safe_section`:**
```python
with safe_section("Nama section"):
    ...kode...
```
Kalau kode di dalamnya error → error dicetak + disimpan di `SECTION_ERRORS`, lalu notebook lanjut. Set `CFG.STRICT = True` kalau lagi debugging dan mau error langsung berhenti.

In [ ]:
SECTION_ERRORS = OrderedDict()
INSIGHTS = []          # list of (section, text) → ditulis ke insights.md
FIG_COUNTER = [0]
FIGURE_INDEX = []      # (file, title) → index figure untuk laporan
TABLES = OrderedDict() # name → DataFrame → diexport ke Excel
FEATURE_FAMILY = {}    # diisi di Section 5
LEAKY_COLS = []


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
    except Exception:
        pass


@contextmanager
def safe_section(name, critical=False):
    """Jalankan blok kode; kalau error → catat & lanjut (kecuali critical / CFG.STRICT)."""
    t0 = time.time()
    try:
        yield
        print(f"\n✅ [{name}] selesai dalam {time.time() - t0:.1f}s")
    except Exception as e:
        SECTION_ERRORS[name] = f"{type(e).__name__}: {e}"
        print(f"\n❌ [{name}] ERROR → {type(e).__name__}: {e}")
        traceback.print_exc(limit=4)
        print("   ↳ Notebook tetap lanjut. Lihat ringkasan error di Section 24 / bagian Troubleshooting.")
        if critical or getattr(globals().get("CFG", object), "STRICT", False):
            raise


def save_fig(name, fig=None, show=True):
    """Simpan figure ke figures/ dengan nomor urut, lalu tampilkan & tutup."""
    fig = fig or plt.gcf()
    FIG_COUNTER[0] += 1
    safe = re.sub(r"[^A-Za-z0-9_\-]+", "_", name).strip("_")[:80]
    path = os.path.join(CFG.FIG_DIR, f"fig{FIG_COUNTER[0]:02d}_{safe}.png")
    try:
        fig.savefig(path, dpi=CFG.FIG_DPI, bbox_inches="tight", facecolor="white")
        title = fig._suptitle.get_text() if fig._suptitle is not None else (fig.axes[0].get_title() if fig.axes else name)
        FIGURE_INDEX.append((os.path.basename(path), title or name))
    except Exception as e:
        print("  (gagal simpan figure:", e, ")")
    if show:
        plt.show()
    plt.close(fig)
    return path


def save_table(df, name, show=False, index=False):
    """Simpan tabel ke tables/<name>.csv dan daftarkan untuk export Excel."""
    if df is None or len(df) == 0:
        return None
    path = os.path.join(CFG.TABLE_DIR, f"{name}.csv")
    try:
        df.to_csv(path, index=index)
        TABLES[name] = df.reset_index() if index else df
    except Exception as e:
        print("  (gagal simpan tabel:", e, ")")
    if show:
        display(df)
    return path


def add_insight(section, text, show=True):
    """Catat 1 kalimat insight (English, siap tempel ke laporan)."""
    INSIGHTS.append((section, text))
    if show:
        print("💡", text)


def pct(x, d=1):
    try:
        return f"{100 * float(x):.{d}f}%"
    except Exception:
        return str(x)


def fmt_num(x, d=2):
    try:
        x = float(x)
    except Exception:
        return str(x)
    if abs(x) >= 1e9:
        return f"{x / 1e9:,.{d}f}B"
    if abs(x) >= 1e6:
        return f"{x / 1e6:,.{d}f}M"
    if abs(x) >= 1e3:
        return f"{x / 1e3:,.{d}f}K"
    return f"{x:,.{d}f}"


def money_axis(ax, axis="y"):
    """Format sumbu angka besar jadi 1.2K / 3.4M / 5.6B (bukan notasi 1e8)."""
    f = matplotlib.ticker.FuncFormatter(lambda v, _: fmt_num(v, 1))
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(f)


def wilson_ci(k, n, z=1.96):
    """95% Wilson confidence interval untuk proporsi k/n (lebih stabil dari normal approx untuk n kecil)."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z ** 2 / n
    centre = (p + z ** 2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))


def p_fmt(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def make_ohe(min_frequency=None):
    """OneHotEncoder yang kompatibel lintas versi sklearn (sparse vs sparse_output, min_frequency)."""
    kwargs = {"handle_unknown": "ignore"}
    try:
        return OneHotEncoder(sparse_output=False, min_frequency=min_frequency, **kwargs)
    except TypeError:
        try:
            return OneHotEncoder(sparse=False, min_frequency=min_frequency, **kwargs)
        except TypeError:
            return OneHotEncoder(sparse=False, **kwargs)


def sanitize_name(c):
    """Nama kolom aman untuk LightGBM/XGBoost (tanpa karakter JSON khusus)."""
    c = re.sub(r"[^0-9A-Za-z_]+", "_", str(c)).strip("_")
    return c if c else "col"


def sanitize_columns(df):
    new, seen = [], {}
    for c in df.columns:
        s = sanitize_name(c)
        if s in seen:
            seen[s] += 1
            s = f"{s}_{seen[s]}"
        else:
            seen[s] = 0
        new.append(s)
    out = df.copy()
    out.columns = new
    return out, dict(zip(df.columns, new))


print("Helper siap.")

# 1. Config

`CFG` adalah **pusat kontrol**. Jangan edit cell `CFG` ini saat lomba — edit di cell **USER OVERRIDE** di bawahnya supaya default tetap utuh.

### Knob paling penting
| Knob | Isi | Catatan |
|---|---|---|
| `TRAIN_PATH`, `TEST_PATH`, `SAMPLE_SUB_PATH` | path file | csv/xlsx/parquet/json didukung. Test boleh tidak ada. |
| `TARGET_COL` | nama kolom churn | `None` = auto-detect (kolom yang ada di train tapi tidak di test / nama mengandung churn/exited/lapse/attrition/status) |
| `POSITIVE_LABEL` | nilai yang berarti churn | `None` = auto (Yes/1/True/Churn/Exited/...) |
| `ID_COL` | kolom ID | `None` = auto |
| `METRIC` | metrik lomba | `roc_auc`, `pr_auc`, `logloss`, `brier`, `f1`, `f2`, `accuracy`, `balanced_accuracy`, `precision`, `recall`, `mcc`, `kappa`, `gini` |
| `SUBMISSION_MODE` | `"probability"` / `"label"` | Lihat format sample_submission. AUC/logloss → probability; F1/accuracy → label. |
| `RUN_MODE` | `"fast"` / `"full"` | fast = cek cepat; full = final |
| `COLUMN_ROLES` | mapping role → kolom | auto-detect by keyword (EN + ID). Override kalau salah. |
| `ACTIONABLE_FEATURES` | fitur yang bisa diintervensi perusahaan | dipakai untuk what-if & rekomendasi |
| Parameter bisnis | margin, biaya retensi, success rate | untuk simulasi ROI campaign |

In [ ]:
class CFG:
    # ===================== RUN =====================
    RUN_MODE = "full"          # "fast" (debug cepat: subsample, 3 fold, model sedikit, tanpa Optuna) | "full"
    STRICT = False             # True → error di section manapun langsung stop (untuk debugging)
    SEED = 42
    VERBOSE = True

    # ===================== PATHS =====================
    TRAIN_PATH = "./data/train.csv"
    TEST_PATH = "./data/test.csv"                  # boleh tidak ada → hanya analisis + CV
    SAMPLE_SUB_PATH = "./data/sample_submission.csv"
    OUTPUT_DIR = "./outputs/churn_insurance"
    # Tabel tambahan one-to-many (misal claims, payments, complaints) → diagregasi per customer lalu di-merge.
    # Format: {"claims": {"path": "./data/claims.csv", "key": "customer_id", "date_col": "claim_date"}}
    EXTRA_TABLES = {}
    # Tabel tambahan one-to-one (misal demografi terpisah) → langsung di-merge.
    # Format: {"demo": {"path": "./data/demographics.csv", "key": "customer_id"}}
    EXTRA_ONE_TO_ONE = {}

    # ===================== COLUMNS =====================
    ID_COL = None              # None = auto-detect
    TARGET_COL = None          # None = auto-detect
    POSITIVE_LABEL = None      # None = auto (Yes/1/True/Churn/...). Contoh: "Attrited Customer"
    DROP_COLS = []             # kolom yang pasti dibuang (misal nama, nomor telp, alamat)
    FORCE_CATEGORICAL = []     # kolom angka yang sebenarnya kategori (misal kode wilayah, kode produk)
    FORCE_NUMERIC = []         # kolom string yang sebenarnya angka
    DATE_COLS = None           # None = auto-detect kolom tanggal
    REFERENCE_DATE = None      # tanggal acuan untuk "days since"; None = tanggal maksimum di data
    GROUP_COL = None           # misal "agent_id"/"household_id" kalau satu grup tidak boleh bocor antar fold

    # Role kolom (auto-detect by keyword). Override contoh: {"tenure": "lama_polis_bulan", "premium": "premi"}
    COLUMN_ROLES = {}
    TENURE_UNIT = "auto"       # "months" | "years" | "days" | "auto" (dipakai untuk band & survival)
    ACTIONABLE_FEATURES = None # None = auto by keyword (payment, renew, complaint, claim, premium, channel, ...)

    # ===================== CLEANING =====================
    NA_TOKENS = ["", " ", "?", "NA", "N/A", "n/a", "na", "null", "NULL", "Null", "nan", "NaN", "-", "--", "#N/A", "missing", "Missing", "unknown_value"]
    DROP_DUPLICATES = True
    QUASI_CONSTANT_THRESHOLD = 0.995   # kolom dengan 1 nilai ≥ 99.5% → drop
    HIGH_CARD_THRESHOLD = 50           # kategori dengan > 50 level → high-cardinality (freq + target encoding)
    RARE_MIN_FREQ = 0.005              # level kategori < 0.5% digabung jadi "__RARE__" (untuk model linear & EDA)
    FIX_IMPOSSIBLE_VALUES = True       # age<0/>110, nilai negatif di premium/income/tenure → NaN

    # ===================== LEAKAGE =====================
    LEAKAGE_KEYWORDS = ["cancel", "terminat", "churn_date", "end_date", "exit_date", "lapse_date", "reason",
                        "alasan", "status_after", "closed", "close_date", "surrender", "refund", "batal", "tutup"]
    LEAKAGE_AUC_THRESHOLD = 0.97       # single-feature AUC ≥ ini → dicurigai leakage
    AUTO_DROP_LEAKY = True             # True → buang fitur dengan BUKTI statistik leakage (missingness/single-feature AUC).
                                       # Fitur yang hanya cocok kata kunci nama → warning saja. False → semua hanya warning.

    # ===================== FEATURE ENGINEERING =====================
    USE_DOMAIN_FE = True       # fitur domain asuransi (rasio klaim, band tenure, dsb)
    USE_FREQ_ENCODING = True   # frequency encoding untuk kategori
    USE_TARGET_ENCODING = True # target encoding in-fold untuk kategori high-cardinality
    USE_ROW_STATS = True       # jumlah missing per baris, dsb

    # ===================== VALIDATION & METRIC =====================
    METRIC = "roc_auc"
    THRESHOLD_METRIC = "auto"  # metrik untuk tuning threshold; "auto" = METRIC kalau label-based, else "f1"
    N_FOLDS = 5
    N_REPEATS = 1              # >1 = repeated CV (lebih stabil, lebih lama)
    CV_STRATEGY = "auto"       # "auto" | "stratified" | "group"

    # ===================== MODELS =====================
    MODELS = ["logreg", "rf", "et", "hgb", "lgbm", "xgb", "catboost"]
    IMBALANCE = "auto"         # "auto" | "none" | "class_weight" | "smote"
    EARLY_STOPPING_ROUNDS = 200
    USE_GPU = False            # True → coba GPU untuk xgb/catboost/lgbm (fallback ke CPU kalau gagal)
    N_JOBS = -1

    USE_OPTUNA = True
    OPTUNA_MODELS = ["lgbm", "catboost", "xgb"]
    OPTUNA_TRIALS = 40
    OPTUNA_TIMEOUT = 900       # detik per model
    OPTUNA_FOLDS = 3           # fold saat tuning (lebih cepat); final tetap pakai N_FOLDS

    FINAL_SEEDS = [42, 2024, 7]  # multi-seed averaging untuk model terbaik
    USE_ENSEMBLE = True
    CALIBRATION = "auto"       # "auto" | "none" | "isotonic" | "sigmoid"
    USE_FEATURE_SELECTION = True

    # ===================== EXPLAINABILITY =====================
    SHAP_SAMPLE = 3000
    PERM_IMPORTANCE_REPEATS = 3
    TOP_N_FEATURES_PLOT = 20

    # ===================== EDA =====================
    MAX_EDA_PLOTS = 24
    MAX_CAT_LEVELS_PLOT = 15
    N_BINS_NUMERIC = 10

    # ===================== BUSINESS =====================
    CURRENCY = "IDR"
    VALUE_COL = None                    # kolom nilai customer (premi). None = auto dari role "premium"
    VALUE_PERIOD = "annual"             # "annual" | "monthly" (kalau monthly → dikali 12 untuk CLV tahunan)
    PROFIT_MARGIN = 0.20                # margin keuntungan dari premi
    RETENTION_COST = 150_000            # biaya kontak/insentif per customer yang di-target
    RETENTION_SUCCESS_RATE = 0.30       # % churner yang berhasil dipertahankan jika di-target
    DISCOUNT_RATE = 0.10                # discount rate untuk CLV
    CLV_HORIZON_YEARS = 5
    RISK_TIER_CUTS = (0.5, 0.8)         # quantile → Low (<50%), Medium (50–80%), High (≥80%)
    WHAT_IF_SCENARIOS = None            # None = auto-generate dari actionable features
    WHAT_IF_ADOPTION = 0.30             # asumsi % customer terdampak yang benar-benar berubah perilakunya

    # ===================== SUBMISSION =====================
    SUBMISSION_MODE = "probability"     # "probability" | "label"
    SUBMISSION_LABEL_FORMAT = "auto"    # "auto" | "01" | "original" (Yes/No dsb)
    SUBMISSION_TARGET_COL = None        # None = kolom ke-2 sample_submission / TARGET_COL

    # ===================== REPORT & DELIVERABLES =====================
    TEAM_NAME = "TeamName"              # dipakai untuk nama file laporan & ZIP: "<TEAM_NAME>_Final Stage 1"
    COMPANY_NAME = "the insurer"        # nama perusahaan di soal (untuk draft laporan)
    REPORT_TITLE = "Understanding and Reducing Customer Churn: A Data-Driven Retention Strategy"
    MAKE_REPORT_DRAFT = True            # buat REPORT_DRAFT.docx + .md otomatis dengan angka hasil run
    MAKE_ZIP = False                    # True setelah artikel PDF selesai → buat ZIP deliverables
    ARTICLE_PDF_PATH = None             # path PDF artikel final (untuk ZIP)
    NOTEBOOK_PATH = None                # path .ipynb ini (untuk ZIP); None = cari *.ipynb di folder kerja
    N_BOOTSTRAP = 1000                  # bootstrap CI untuk skor model
    FEATURE_LABELS = {}                 # nama bisnis untuk laporan, misal {"n_complaints_12m": "complaints in last 12 months"}

    # ===================== FIGURES =====================
    FIG_DPI = 200                       # 300 untuk kualitas cetak laporan
    PALETTE = ["#2E86AB", "#E4572E", "#76B041", "#F3A712", "#6C4AB6", "#17BEBB", "#8C8C8C"]


print("CFG default loaded. Edit nilai di cell USER OVERRIDE berikutnya.")

In [ ]:
# >>> USER OVERRIDE
# =====================================================================
# EDIT DI SINI SAAT LOMBA. Semua yang tidak diubah memakai default CFG.
# =====================================================================
# CFG.RUN_MODE = "fast"
# CFG.TRAIN_PATH = "./data/train.csv"
# CFG.TEST_PATH = "./data/test.csv"
# CFG.SAMPLE_SUB_PATH = "./data/sample_submission.csv"
# CFG.TARGET_COL = "churn"
# CFG.ID_COL = "customer_id"
# CFG.POSITIVE_LABEL = "Yes"
# CFG.METRIC = "roc_auc"
# CFG.SUBMISSION_MODE = "probability"
# CFG.DROP_COLS = []
# CFG.EXTRA_TABLES = {"claims": {"path": "./data/claims.csv", "key": "customer_id", "date_col": "claim_date"}}
# CFG.COLUMN_ROLES = {"tenure": "tenure_months", "premium": "annual_premium"}
# CFG.ACTIONABLE_FEATURES = ["payment_method", "auto_renew", "n_complaints_12m"]
# CFG.PROFIT_MARGIN = 0.2; CFG.RETENTION_COST = 150_000; CFG.RETENTION_SUCCESS_RATE = 0.3
# CFG.FIG_DPI = 300
# CFG.TEAM_NAME = "NamaTim"; CFG.COMPANY_NAME = "PT Asuransi X"; CFG.REPORT_TITLE = "..."
# CFG.ARTICLE_PDF_PATH = "./NamaTim_Final Stage 1.pdf"; CFG.MAKE_ZIP = True   # setelah artikel selesai

### Apply run mode & buat folder output
`fast` mode mengubah beberapa knob supaya cepat: subsample besar data, 3 fold, model sedikit, tanpa Optuna, 1 seed.
**Jangan pakai hasil fast mode untuk submission final.**

In [ ]:
def apply_run_mode():
    CFG.METRIC = str(CFG.METRIC).lower().strip()
    if CFG.RUN_MODE == "fast":
        CFG.N_FOLDS = min(CFG.N_FOLDS, 3)
        CFG.N_REPEATS = 1
        CFG.MODELS = [m for m in CFG.MODELS if m in ("logreg", "hgb", "lgbm", "catboost")] or ["logreg", "hgb"]
        CFG.USE_OPTUNA = False
        CFG.FINAL_SEEDS = CFG.FINAL_SEEDS[:1]
        CFG.SHAP_SAMPLE = min(CFG.SHAP_SAMPLE, 1000)
        CFG.PERM_IMPORTANCE_REPEATS = 1
        CFG.MAX_EDA_PLOTS = min(CFG.MAX_EDA_PLOTS, 12)
        CFG.EARLY_STOPPING_ROUNDS = min(CFG.EARLY_STOPPING_ROUNDS, 100)
        CFG.FAST_SAMPLE_ROWS = 20000
        CFG.N_BOOTSTRAP = min(CFG.N_BOOTSTRAP, 200)
    else:
        CFG.FAST_SAMPLE_ROWS = None
    CFG.FIG_DIR = os.path.join(CFG.OUTPUT_DIR, "figures")
    CFG.TABLE_DIR = os.path.join(CFG.OUTPUT_DIR, "tables")
    CFG.MODEL_DIR = os.path.join(CFG.OUTPUT_DIR, "models")
    CFG.SUB_DIR = os.path.join(CFG.OUTPUT_DIR, "submissions")
    for d in [CFG.OUTPUT_DIR, CFG.FIG_DIR, CFG.TABLE_DIR, CFG.MODEL_DIR, CFG.SUB_DIR]:
        os.makedirs(d, exist_ok=True)


apply_run_mode()
seed_everything(CFG.SEED)

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 12, "axes.labelsize": 10, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
sns.set_palette(CFG.PALETTE)

print("RUN_MODE    :", CFG.RUN_MODE)
print("METRIC      :", CFG.METRIC)
print("MODELS      :", CFG.MODELS)
print("N_FOLDS     :", CFG.N_FOLDS, "| repeats:", CFG.N_REPEATS)
print("OPTUNA      :", CFG.USE_OPTUNA and HAS_OPTUNA)
print("OUTPUT_DIR  :", os.path.abspath(CFG.OUTPUT_DIR))

# 2. Load Data

`smart_read()` membaca **csv / tsv / txt / xlsx / xls / parquet / json / feather / pickle**, otomatis mencoba beberapa encoding (utf-8 → utf-8-sig → latin-1) dan auto-detect separator (`,` `;` `|` tab).

**Kasus multi-tabel (umum di data asuransi):** data bisa dipecah jadi `customers`, `policies`, `claims`, `payments`, `complaints`.
- Tabel **one-to-one** (1 baris per customer) → isi `CFG.EXTRA_ONE_TO_ONE` → langsung di-merge.
- Tabel **one-to-many** (banyak baris per customer, misal riwayat klaim) → isi `CFG.EXTRA_TABLES` → diagregasi per customer (count, sum, mean, max, min, std, jumlah per kategori, recency dari tanggal) lalu di-merge.

**Jika X → Y:**
- Error `FileNotFoundError` → cek path; jalankan `os.listdir("./data")` untuk lihat nama file sebenarnya.
- Jumlah kolom cuma 1 padahal file punya banyak kolom → separator salah; `smart_read(path, sep=";")`.
- Huruf aneh (`Ã©`) → encoding; `smart_read(path, encoding="latin-1")`.
- Setelah merge jumlah baris bertambah → key tidak unik di tabel kanan (seharusnya one-to-many → pakai `EXTRA_TABLES`).
- Data cuma 1 file (tanpa test) → tidak apa-apa; notebook tetap jalan untuk analisis + CV, submission di-skip.

In [ ]:
def smart_read(path, **kwargs):
    """Baca file tabular apapun dengan fallback encoding & separator."""
    if path is None or not os.path.exists(str(path)):
        raise FileNotFoundError(f"File tidak ditemukan: {path}")
    ext = os.path.splitext(str(path))[1].lower()
    if ext in (".xlsx", ".xlsm", ".xls"):
        return pd.read_excel(path, **kwargs)
    if ext == ".parquet":
        return pd.read_parquet(path, **kwargs)
    if ext == ".feather":
        return pd.read_feather(path, **kwargs)
    if ext in (".pkl", ".pickle"):
        return pd.read_pickle(path, **kwargs)
    if ext == ".json":
        try:
            return pd.read_json(path, **kwargs)
        except ValueError:
            return pd.read_json(path, lines=True, **kwargs)
    last_err = None
    for enc in [kwargs.pop("encoding", None), "utf-8", "utf-8-sig", "latin-1"]:
        if enc is None:
            continue
        try:
            if "sep" in kwargs:
                return pd.read_csv(path, encoding=enc, **kwargs)
            df = pd.read_csv(path, encoding=enc, low_memory=False, **kwargs)
            if df.shape[1] == 1 and any(s in str(df.columns[0]) for s in [";", "|", "\t"]):
                df = pd.read_csv(path, encoding=enc, sep=None, engine="python", **kwargs)
            return df
        except UnicodeDecodeError as e:
            last_err = e
            continue
        except pd.errors.ParserError:
            return pd.read_csv(path, encoding=enc, sep=None, engine="python", **kwargs)
    raise last_err


def aggregate_child_table(child, key, prefix, date_col=None, ref_date=None, max_levels=10):
    """Agregasi tabel one-to-many per key: numeric stats, count per kategori, recency tanggal."""
    child = child.copy()
    aggs = [child.groupby(key).size().rename(f"{prefix}_count")]
    num_cols = [c for c in child.select_dtypes(include=np.number).columns if c != key]
    if num_cols:
        g = child.groupby(key)[num_cols].agg(["sum", "mean", "max", "min", "std"])
        g.columns = [f"{prefix}_{a}_{b}" for a, b in g.columns]
        aggs.append(g)
    cat_cols = [c for c in child.columns if c not in num_cols + [key, date_col] and child[c].dtype == object]
    for c in cat_cols:
        top = child[c].value_counts().index[:max_levels]
        sub = child[child[c].isin(top)]
        ct = pd.crosstab(sub[key], sub[c])
        ct.columns = [f"{prefix}_{c}_{sanitize_name(v)}_count" for v in ct.columns]
        aggs.append(ct)
        aggs.append(child.groupby(key)[c].nunique().rename(f"{prefix}_{c}_nunique"))
    if date_col and date_col in child.columns:
        d = pd.to_datetime(child[date_col], errors="coerce")
        ref = pd.Timestamp(ref_date) if ref_date is not None else d.max()
        child["_days_ago"] = (ref - d).dt.days
        g = child.groupby(key)["_days_ago"].agg(["min", "max"])
        g.columns = [f"{prefix}_days_since_last", f"{prefix}_days_since_first"]
        aggs.append(g)
    out = pd.concat(aggs, axis=1).reset_index()
    return out


def merge_safely(left, right, key, how="left", name="table"):
    n0 = len(left)
    if right[key].duplicated().any():
        raise ValueError(f"[{name}] key '{key}' tidak unik di tabel kanan → gunakan EXTRA_TABLES (agregasi) bukan one-to-one")
    out = left.merge(right, on=key, how=how)
    assert len(out) == n0, f"[{name}] jumlah baris berubah setelah merge ({n0} → {len(out)})"
    return out


with safe_section("Load data", critical=True):
    raw_train = smart_read(CFG.TRAIN_PATH)
    HAS_TEST = CFG.TEST_PATH is not None and os.path.exists(str(CFG.TEST_PATH))
    raw_test = smart_read(CFG.TEST_PATH) if HAS_TEST else None
    try:
        sample_sub = smart_read(CFG.SAMPLE_SUB_PATH) if CFG.SAMPLE_SUB_PATH and os.path.exists(str(CFG.SAMPLE_SUB_PATH)) else None
    except Exception as e:
        print("Sample submission gagal dibaca:", e)
        sample_sub = None

    # strip whitespace di nama kolom (sering terjadi: " Churn")
    raw_train.columns = [str(c).strip() for c in raw_train.columns]
    ORIGINAL_COLS = list(raw_train.columns)  # kolom asli sebelum merge tabel tambahan
    if HAS_TEST:
        raw_test.columns = [str(c).strip() for c in raw_test.columns]

    if CFG.FAST_SAMPLE_ROWS and len(raw_train) > CFG.FAST_SAMPLE_ROWS:
        raw_train = raw_train.sample(CFG.FAST_SAMPLE_ROWS, random_state=CFG.SEED).reset_index(drop=True)
        print(f"FAST MODE: train di-subsample ke {CFG.FAST_SAMPLE_ROWS} baris")

    print("Train shape :", raw_train.shape)
    print("Test shape  :", raw_test.shape if HAS_TEST else "— tidak ada test file (submission akan di-skip)")
    print("Sample sub  :", sample_sub.shape if sample_sub is not None else "— tidak ada")
    display(raw_train.head())

In [ ]:
with safe_section("Merge extra tables"):
    for name, spec in (CFG.EXTRA_ONE_TO_ONE or {}).items():
        t = smart_read(spec["path"])
        t.columns = [str(c).strip() for c in t.columns]
        raw_train = merge_safely(raw_train, t, spec["key"], name=name)
        if HAS_TEST:
            raw_test = merge_safely(raw_test, t, spec["key"], name=name)
        print(f"[{name}] merged one-to-one: +{t.shape[1] - 1} kolom")
    for name, spec in (CFG.EXTRA_TABLES or {}).items():
        t = smart_read(spec["path"])
        t.columns = [str(c).strip() for c in t.columns]
        agg = aggregate_child_table(t, spec["key"], prefix=sanitize_name(name), date_col=spec.get("date_col"),
                                    ref_date=CFG.REFERENCE_DATE)
        raw_train = merge_safely(raw_train, agg, spec["key"], name=name)
        if HAS_TEST:
            raw_test = merge_safely(raw_test, agg, spec["key"], name=name)
        cnt_cols = [c for c in agg.columns if c.endswith("_count") or c.endswith("_sum")]
        raw_train[cnt_cols] = raw_train[cnt_cols].fillna(0)   # customer tanpa record → 0
        if HAS_TEST:
            raw_test[cnt_cols] = raw_test[cnt_cols].fillna(0)
        print(f"[{name}] one-to-many diagregasi: {t.shape} → +{agg.shape[1] - 1} fitur")
    if not CFG.EXTRA_TABLES and not CFG.EXTRA_ONE_TO_ONE:
        print("Tidak ada tabel tambahan (CFG.EXTRA_TABLES kosong).")

# 3. Data Quality & Cleaning

## 3.1 Deteksi target & ID
**Target auto-detect** (urutan): `CFG.TARGET_COL` → kolom yang ada di train tapi tidak ada di test → nama kolom mengandung kata kunci churn (`churn, exited, attrition, lapse, lapsed, left, cancel, status, is_active, retained, renew, berhenti, putus`).

**ID auto-detect**: nama seperti `id`, `customer_id`, `policy_no`, `nomor_polis` dan nilainya hampir unik per baris.

**Jika auto-detect salah** → set manual `CFG.TARGET_COL` / `CFG.ID_COL` di USER OVERRIDE lalu run ulang dari Section 2.

In [ ]:
TARGET_KEYWORDS = ["churn", "exited", "attrition", "attrited", "lapse", "lapsed", "left", "cancel", "is_active",
                   "retained", "retention", "renew", "berhenti", "putus", "status", "target", "label", "flag"]
ID_PATTERN = re.compile(r"(^id$|^id_|_id$|customer.?id|cust.?id|client.?id|member.?id|policy.?(no|num|number|id)|"
                        r"^no_?polis|nomor|^row.?num|^index$|^unnamed)", re.I)


def detect_target(train_df, test_df):
    if CFG.TARGET_COL:
        if CFG.TARGET_COL not in train_df.columns:
            raise KeyError(f"TARGET_COL '{CFG.TARGET_COL}' tidak ada di train. Kolom: {list(train_df.columns)}")
        return CFG.TARGET_COL
    if test_df is not None:
        only_train = [c for c in train_df.columns if c not in test_df.columns]
        if len(only_train) == 1:
            return only_train[0]
        cand = [c for c in only_train if any(k in c.lower() for k in TARGET_KEYWORDS)]
        if cand:
            return cand[0]
    for k in TARGET_KEYWORDS:
        for c in train_df.columns:
            if k in c.lower() and train_df[c].nunique(dropna=True) == 2:
                return c
    raise ValueError("Target tidak bisa dideteksi otomatis. Set CFG.TARGET_COL di USER OVERRIDE.")


def detect_id(df, target):
    if CFG.ID_COL:
        return CFG.ID_COL if CFG.ID_COL in df.columns else None
    n = len(df)
    for c in df.columns:
        if c == target:
            continue
        if ID_PATTERN.search(str(c)) and df[c].nunique(dropna=True) >= 0.95 * n:
            return c
    for c in df.columns:
        if c != target and df[c].dtype == object and df[c].nunique(dropna=True) == n:
            return c
    return None


with safe_section("Detect target & ID", critical=True):
    TARGET = detect_target(raw_train, raw_test)
    ID_COL = detect_id(raw_train, TARGET)
    print("TARGET :", TARGET)
    print("ID_COL :", ID_COL)
    if ID_COL is None:
        print("⚠️ Tidak ada ID column terdeteksi → submission akan memakai urutan baris test / sample_submission.")

## 3.2 Parsing label churn → 0/1
Positive class (=1) adalah **churn**. Auto-mapping untuk nilai seperti `Yes / True / 1 / Churn / Churned / Exited / Left / Lapsed / Cancelled / Attrited Customer / Ya`.

**Hati-hati kasus terbalik**: kolom `is_active`, `retained`, `renewed`, `status = Active` → nilai 1/Yes justru berarti **tidak churn**. Notebook mendeteksi nama kolom seperti ini dan membalik label otomatis (dicetak jelas). Kalau salah, set `CFG.POSITIVE_LABEL` = nilai yang berarti churn.

**Jika target punya > 2 kelas** (misal `Active / Lapsed / Surrendered`) → set `CFG.POSITIVE_LABEL` ke salah satu kelas (atau list) → dibinarisasi "kelas itu vs lainnya". Untuk multiclass murni pakai `02_Tabular/tabular_master.ipynb`.

In [ ]:
POSITIVE_TOKENS = {"yes", "y", "true", "t", "1", "1.0", "churn", "churned", "exited", "exit", "left", "leave",
                   "lapsed", "lapse", "cancelled", "canceled", "cancel", "attrited customer", "attrited", "inactive",
                   "terminated", "closed", "ya", "berhenti", "keluar", "putus", "lost"}
INVERSE_NAME_HINTS = ["active", "retain", "renew", "stay", "aktif", "loyal", "persist"]


def parse_target(series):
    s = series.copy()
    if s.isna().any():
        print(f"⚠️ Target punya {s.isna().sum()} NaN → baris tersebut dibuang dari train.")
    vals = s.dropna()
    uniq = pd.Series(vals.unique())
    norm = uniq.astype(str).str.strip().str.lower()

    if CFG.POSITIVE_LABEL is not None:
        pos = CFG.POSITIVE_LABEL if isinstance(CFG.POSITIVE_LABEL, (list, tuple, set)) else [CFG.POSITIVE_LABEL]
        pos_norm = {str(p).strip().lower() for p in pos}
        mapping = {u: int(n in pos_norm) for u, n in zip(uniq, norm)}
        how = f"POSITIVE_LABEL={list(pos)}"
    elif len(uniq) != 2:
        raise ValueError(f"Target punya {len(uniq)} nilai unik {list(uniq)[:10]}. Set CFG.POSITIVE_LABEL (nilai churn).")
    else:
        hits = [n in POSITIVE_TOKENS for n in norm]
        if sum(hits) == 1:
            mapping = {u: int(h) for u, h in zip(uniq, hits)}
            how = "token churn (Yes/1/True/Churn/...)"
        else:
            vc = vals.value_counts()
            minority = vc.index[-1]
            mapping = {u: int(u == minority) for u in uniq}
            how = f"minority class ('{minority}') dianggap churn (tidak ada token yang jelas)"
        if any(h in TARGET.lower() for h in INVERSE_NAME_HINTS) and not any(k in TARGET.lower() for k in ["churn", "exit", "lapse", "cancel"]):
            mapping = {k: 1 - v for k, v in mapping.items()}
            how += " → DIBALIK karena nama kolom menunjukkan status aktif/retain (1 = churn sekarang)"
    y_ = s.map(mapping)
    return y_, mapping, how


with safe_section("Parse target", critical=True):
    raw_train = raw_train[raw_train[TARGET].notna()].reset_index(drop=True)
    y_series, TARGET_MAPPING, how = parse_target(raw_train[TARGET])
    POS_ORIGINAL = [k for k, v in TARGET_MAPPING.items() if v == 1]
    NEG_ORIGINAL = [k for k, v in TARGET_MAPPING.items() if v == 0]
    print("Mapping target :", TARGET_MAPPING)
    print("Metode         :", how)
    assert y_series.notna().all(), "Ada nilai target yang tidak ter-mapping"
    y = y_series.astype(int).values

## 3.3 Cleaning otomatis & inferensi tipe kolom
Yang dilakukan (train & test diproses **bersamaan** dengan aturan yang sama → konsisten):
1. String "kosong" (`"", "?", "NA", "null", "-", ...`) → `NaN`. Whitespace di-strip.
2. **Angka dalam string** dikonversi: `"12,500,000"`, `"Rp 5.000.000"`, `"45%"`, `"$1,234.50"`, `" "` (contoh Telco `TotalCharges`). Deteksi otomatis format ribuan titik (Indonesia) vs koma.
3. **Tanggal** dideteksi (nama kolom / format) → di-parse → nanti dijadikan fitur "days since".
4. **Kategori typo/case** disatukan: `"Male"`, `"male "`, `"MALE"` → satu ejaan (yang paling sering).
5. **Boolean-like** (`Yes/No`, `True/False`, `Y/N`) → 1/0.
6. Tipe kolom: `numeric`, `binary`, `categorical`, `high_card` (kategori > `HIGH_CARD_THRESHOLD` level), `date`, `id`, `text`, `constant`.
7. Kolom **konstan / quasi-konstan** → dibuang. Duplikat baris → dibuang (train saja).
8. **Nilai mustahil** (umur < 0 atau > 110, premi/pendapatan/tenure negatif) → `NaN`.

**Cara baca `quality_report`:** cek kolom `missing_%` (> 40% → pertimbangkan drop / jadikan indikator), `n_unique` (1 → konstan), `inferred_type` (pastikan masuk akal!).

**Jika tipe salah:** kolom kode (misal `region_code = 31, 32`) terbaca numeric → masukkan ke `CFG.FORCE_CATEGORICAL`. Kolom angka terbaca kategori → `CFG.FORCE_NUMERIC`.

In [ ]:
DATE_NAME_HINT = re.compile(r"(date|tanggal|tgl|_dt$|^dt_|time|waktu|periode|period|_at$|dob|birth|lahir|start|mulai|join)", re.I)
BOOL_MAP = {"yes": 1, "no": 0, "y": 1, "n": 0, "true": 1, "false": 0, "t": 1, "f": 0, "ya": 1, "tidak": 0,
            "iya": 1, "ga": 0, "gak": 0, "enggak": 0}


def try_parse_numeric(s):
    """Coba konversi series string → numeric. Return (series_numeric, success_ratio)."""
    x = s.dropna().astype(str).str.strip()
    if len(x) == 0:
        return None, 0.0
    x2 = x.str.replace(r"(?i)(rp\.?|idr|usd|\$|€|£|%|\s)", "", regex=True)
    dot_thousand = x2.str.match(r"^-?\d{1,3}(\.\d{3})+(,\d+)?$").mean()
    comma_thousand = x2.str.match(r"^-?\d{1,3}(,\d{3})+(\.\d+)?$").mean()
    decimal_comma = x2.str.match(r"^-?\d+,\d+$").mean()
    if dot_thousand > 0.5:
        x2 = x2.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    elif comma_thousand > 0.3 or (comma_thousand > 0 and decimal_comma == 0):
        x2 = x2.str.replace(",", "", regex=False)
    elif decimal_comma > 0.5:
        x2 = x2.str.replace(",", ".", regex=False)
    num = pd.to_numeric(x2, errors="coerce")
    ratio = num.notna().mean()
    if ratio == 0:
        return None, 0.0
    full = pd.Series(np.nan, index=s.index)
    full.loc[num.index] = num.values
    return full, ratio


def try_parse_date(s, name):
    x = s.dropna().astype(str)
    if len(x) == 0:
        return None, 0.0
    looks = x.str.contains(r"\d{1,4}[-/\.]\d{1,2}[-/\.]\d{1,4}").mean()
    if looks < 0.8 and not DATE_NAME_HINT.search(name):
        return None, 0.0
    if looks < 0.5:
        return None, 0.0
    parsed = pd.to_datetime(s, errors="coerce")
    if parsed.notna().sum() < 0.9 * s.notna().sum():
        parsed2 = pd.to_datetime(s, errors="coerce", dayfirst=True)
        if parsed2.notna().sum() > parsed.notna().sum():
            parsed = parsed2
    ratio = parsed.notna().sum() / max(1, s.notna().sum())
    return parsed, ratio


def unify_category_case(s):
    """Satukan ejaan kategori yang beda kapitalisasi/spasi → ejaan paling sering."""
    x = s.astype("object").where(s.isna(), s.astype(str).str.strip().str.replace(r"\s+", " ", regex=True))
    key = x.where(x.isna(), x.astype(str).str.lower())
    canon = {}
    for k, grp in x.groupby(key):
        canon[k] = grp.value_counts().index[0]
    return x.where(x.isna(), key.map(canon))


def clean_and_infer(train_df, test_df):
    """Bersihkan train+test dengan aturan sama. Return train_c, test_c, col_types, log."""
    n_tr = len(train_df)
    full = pd.concat([train_df.drop(columns=[TARGET]), test_df] if test_df is not None else [train_df.drop(columns=[TARGET])],
                     axis=0, ignore_index=True)
    log, col_types = [], {}
    full = full.replace(CFG.NA_TOKENS, np.nan)

    for c in list(full.columns):
        if c == ID_COL:
            col_types[c] = "id"
            continue
        if c in CFG.DROP_COLS:
            col_types[c] = "dropped"
            continue
        s = full[c]
        if s.dtype == object or str(s.dtype).startswith("string") or str(s.dtype) == "category":
            s = s.astype("object")
            s = s.where(s.isna(), s.astype(str).str.strip())
            s = s.replace(CFG.NA_TOKENS, np.nan)
            full[c] = s
            if c not in CFG.FORCE_CATEGORICAL:
                # date?
                if CFG.DATE_COLS is None or c in (CFG.DATE_COLS or []):
                    parsed, r = try_parse_date(s, c)
                    if parsed is not None and r >= 0.9:
                        full[c] = parsed
                        col_types[c] = "date"
                        log.append(f"{c}: parsed sebagai tanggal ({pct(r)})")
                        continue
                # numeric in string?
                num, r = try_parse_numeric(s)
                if num is not None and (r >= 0.95 or c in CFG.FORCE_NUMERIC):
                    full[c] = num
                    log.append(f"{c}: string → numeric ({pct(r)} berhasil)")
                    s = full[c]
                else:
                    # bool-like?
                    low = s.dropna().astype(str).str.lower()
                    if len(low) and low.isin(BOOL_MAP.keys()).all() and low.nunique() == 2:
                        full[c] = s.astype(str).str.lower().map(BOOL_MAP).where(s.notna())
                        col_types[c] = "binary"
                        log.append(f"{c}: boolean-like → 1/0")
                        continue
                    before = s.nunique()
                    full[c] = unify_category_case(s)
                    after = full[c].nunique()
                    if after < before:
                        log.append(f"{c}: {before - after} variasi ejaan/case disatukan ({before} → {after} level)")
        elif np.issubdtype(s.dtype, np.datetime64):
            col_types[c] = "date"
            continue
        elif s.dtype == bool:
            full[c] = s.astype(int)

        s = full[c]
        nun = s.nunique(dropna=True)
        if nun <= 1:
            col_types[c] = "constant"
            continue
        top_freq = s.value_counts(normalize=True, dropna=False).iloc[0]
        if top_freq >= CFG.QUASI_CONSTANT_THRESHOLD:
            col_types[c] = "constant"
            log.append(f"{c}: quasi-constant (nilai teratas {pct(top_freq)})")
            continue
        if c in CFG.FORCE_CATEGORICAL:
            full[c] = s.astype("object").where(s.isna(), s.astype(str))
            col_types[c] = "categorical" if nun <= CFG.HIGH_CARD_THRESHOLD else "high_card"
            continue
        if pd.api.types.is_numeric_dtype(s):
            col_types[c] = "binary" if nun == 2 else "numeric"
            if nun == 2 and set(s.dropna().unique()) - {0, 1}:
                vals = sorted(s.dropna().unique())
                full[c] = (s == vals[1]).astype(float).where(s.notna())
                log.append(f"{c}: binary numeric {vals} → 0/1")
        else:
            avg_len = s.dropna().astype(str).str.len().mean()
            if nun > 0.5 * len(s) and avg_len > 30:
                col_types[c] = "text"
            elif nun > 0.95 * len(s):
                col_types[c] = "id"
                log.append(f"{c}: hampir unik per baris → dianggap ID-like (dibuang dari model)")
            elif nun == 2:
                col_types[c] = "binary_cat"
            elif nun > CFG.HIGH_CARD_THRESHOLD:
                col_types[c] = "high_card"
            else:
                col_types[c] = "categorical"

    train_c = full.iloc[:n_tr].reset_index(drop=True)
    test_c = full.iloc[n_tr:].reset_index(drop=True) if test_df is not None else None
    return train_c, test_c, col_types, log


with safe_section("Cleaning & type inference", critical=True):
    train, test, COL_TYPES, clean_log = clean_and_infer(raw_train, raw_test)
    print("Log cleaning:")
    for line in clean_log:
        print("  •", line)
    type_summary = pd.Series(COL_TYPES).value_counts()
    print("\nRingkasan tipe kolom:")
    display(type_summary.to_frame("n_columns"))

## 3.4 Nilai mustahil, duplikat, dan Data Quality Report
Role kolom (umur, premi, tenure, dll.) dideteksi dulu secara kasar di sini supaya pengecekan nilai mustahil tepat sasaran. Detail role ada di Section 5.

In [ ]:
ROLE_KEYWORDS = OrderedDict([
    ("tenure", ["tenure", "lama", "masa", "months_active", "month_with", "duration", "durasi", "membership_length", "policy_age", "customer_since_months", "months_on_book", "seniority"]),
    ("age", ["age", "usia", "umur"]),
    ("premium", ["premium", "premi", "monthlycharges", "monthly_charge", "charges", "fee", "price", "iuran", "tagihan", "billing_amount"]),
    ("total_paid", ["totalcharges", "total_charge", "total_paid", "total_premium", "lifetime_value", "ltv", "total_amount"]),
    ("income", ["income", "pendapatan", "salary", "gaji", "penghasilan", "estimatedsalary"]),
    ("sum_insured", ["sum_insured", "sum_assured", "uang_pertanggungan", "coverage_amount", "insured_amount", "face_amount", "benefit"]),
    ("claims", ["claim", "klaim"]),
    ("complaints", ["complain", "keluhan", "komplain", "ticket", "support_call", "customer_service_calls", "service_calls"]),
    ("late_payment", ["late", "telat", "terlambat", "overdue", "arrear", "missed_payment", "tunggakan", "delinq", "grace"]),
    ("products", ["n_products", "num_products", "numofproducts", "product_count", "jumlah_produk", "products", "riders", "num_policies", "policies"]),
    ("premium_change", ["premium_change", "price_change", "increase", "kenaikan", "rate_change", "renewal_change"]),
    ("engagement", ["engagement", "login", "app_usage", "visit", "activity", "digital", "isactivemember", "usage"]),
    ("channel", ["channel", "saluran", "source", "acquisition"]),
    ("payment_method", ["payment_method", "paymentmethod", "metode_bayar", "payment_type", "cara_bayar", "payment_mode"]),
    ("payment_frequency", ["frequency", "frekuensi", "payment_freq", "billing_cycle", "contract", "term"]),
    ("auto_renew", ["auto_renew", "autorenew", "auto-renew", "auto_debit", "autodebit", "paperless"]),
    ("gender", ["gender", "sex", "jenis_kelamin", "kelamin"]),
    ("region", ["region", "wilayah", "provinsi", "province", "city", "kota", "geography", "state", "area", "branch", "cabang"]),
    ("policy_type", ["policy_type", "product_type", "plan", "jenis_polis", "tipe_polis", "coverage", "line_of_business", "lob"]),
    ("agent", ["agent", "agen", "broker", "advisor"]),
    ("dependents", ["dependent", "tanggungan", "children", "anak", "family_size", "household"]),
])


def detect_roles(df, col_types):
    """Cari kolom untuk tiap role berdasarkan keyword nama kolom. Override via CFG.COLUMN_ROLES."""
    roles = {}
    used = set()
    numeric_roles = {"tenure", "age", "premium", "total_paid", "income", "sum_insured", "claims", "complaints",
                     "late_payment", "products", "premium_change", "engagement", "dependents"}
    for role, kws in ROLE_KEYWORDS.items():
        if role in (CFG.COLUMN_ROLES or {}):
            roles[role] = CFG.COLUMN_ROLES[role]
            used.add(CFG.COLUMN_ROLES[role])
            continue
        for c in df.columns:
            if c in used or col_types.get(c) in ("id", "dropped", "constant", "text"):
                continue
            cl = c.lower()
            if any(k in cl for k in kws):
                is_num = col_types.get(c) in ("numeric", "binary")
                if role in numeric_roles and not is_num:
                    continue
                if role == "claims" and any(k in cl for k in ["reject", "tolak", "denied", "settle", "days", "amount", "ratio"]):
                    continue  # klaim ditolak/lama settle → simpan sebagai fitur biasa
                if role == "tenure" and col_types.get(c) == "date":
                    continue
                roles[role] = c
                used.add(c)
                break
    return roles


with safe_section("Impossible values, duplicates & quality report"):
    ROLES = detect_roles(train, COL_TYPES)
    if CFG.FIX_IMPOSSIBLE_VALUES:
        rules = {"age": (0, 110), "tenure": (0, None), "premium": (0, None), "income": (0, None),
                 "sum_insured": (0, None), "total_paid": (0, None), "claims": (0, None), "complaints": (0, None),
                 "late_payment": (0, None), "products": (0, None), "dependents": (0, 30)}
        for role, (lo, hi) in rules.items():
            c = ROLES.get(role)
            if c is None or c not in train.columns or not pd.api.types.is_numeric_dtype(train[c]):
                continue
            for name, d in [("train", train), ("test", test)]:
                if d is None:
                    continue
                bad = pd.Series(False, index=d.index)
                if lo is not None:
                    bad |= d[c] < lo
                if hi is not None:
                    bad |= d[c] > hi
                if bad.sum():
                    print(f"  • {name}.{c} ({role}): {bad.sum()} nilai mustahil (di luar [{lo}, {hi}]) → NaN")
                    d.loc[bad, c] = np.nan

    # duplikat
    feat_cols_for_dup = [c for c in train.columns if c != ID_COL]
    dup_mask = train.duplicated(subset=feat_cols_for_dup, keep="first")
    tmp = train[feat_cols_for_dup].copy()
    tmp["__y"] = y
    conflict = tmp[tmp.duplicated(subset=feat_cols_for_dup, keep=False)].groupby(feat_cols_for_dup, dropna=False)["__y"].nunique()
    n_conflict = int((conflict > 1).sum()) if len(conflict) else 0
    N_DUP_REMOVED = int(dup_mask.sum()) if CFG.DROP_DUPLICATES else 0
    print(f"\nDuplikat baris (fitur identik, abaikan ID): {dup_mask.sum()} | grup duplikat dengan label berbeda: {n_conflict}")
    if CFG.DROP_DUPLICATES and dup_mask.sum() > 0:
        keep = ~dup_mask.values
        train = train[keep].reset_index(drop=True)
        raw_train = raw_train[keep].reset_index(drop=True)
        y = y[keep]
        print(f"  → {dup_mask.sum()} duplikat dibuang. Train sekarang {train.shape}")
    if n_conflict:
        print("  ⚠️ Ada data identik dengan label berbeda → noise label / fitur kurang informatif. Wajar dalam jumlah kecil.")

    # quality report
    rows = []
    for c in train.columns:
        s = train[c]
        r = {"column": c, "inferred_type": COL_TYPES.get(c), "dtype": str(s.dtype),
             "missing_%": round(100 * s.isna().mean(), 2),
             "missing_test_%": round(100 * test[c].isna().mean(), 2) if test is not None and c in test else np.nan,
             "n_unique": s.nunique(dropna=True),
             "top_value": s.value_counts(dropna=True).index[0] if s.notna().any() else None,
             "top_freq_%": round(100 * s.value_counts(normalize=True, dropna=True).iloc[0], 2) if s.notna().any() else None}
        if pd.api.types.is_numeric_dtype(s):
            r.update({"min": s.min(), "median": s.median(), "max": s.max(), "skew": round(s.skew(), 2) if s.nunique() > 2 else np.nan})
        rows.append(r)
    quality_report = pd.DataFrame(rows)
    save_table(quality_report, "data_quality_report")
    display(quality_report)

    high_missing = quality_report[quality_report["missing_%"] > 40]["column"].tolist()
    if high_missing:
        print("⚠️ Kolom dengan missing > 40%:", high_missing, "→ missingness-nya sendiri bisa jadi sinyal (indikator dibuat otomatis).")
    if test is not None:
        miss_tr = set(raw_train.columns) - {TARGET} - set(raw_test.columns)
        miss_te = set(raw_test.columns) - set(raw_train.columns)
        if miss_tr or miss_te:
            print("⚠️ Kolom beda antara train & test → hanya train:", miss_tr, "| hanya test:", miss_te)

# 4. Target Analysis

**Cara baca:**
- **Churn rate** = proporsi churn. Ini juga = presisi kalau kita menebak "semua churn".
- **Imbalance**: churn rate < 20% → imbalanced. Accuracy jadi menyesatkan (tebak "tidak churn" semua sudah dapat accuracy tinggi). Fokus ke ROC-AUC, PR-AUC, F1, recall.
- **Baseline**: score dummy (tanpa model). Model kita **harus** jauh di atas ini.

**Untuk laporan:** "The dataset contains N customers with an overall churn rate of X%, indicating [moderate/severe] class imbalance."

In [ ]:
with safe_section("Target analysis"):
    n_pos, n_tot = int(y.sum()), len(y)
    CHURN_RATE = n_pos / n_tot
    print(f"Jumlah customer : {n_tot:,}")
    print(f"Churn (1)       : {n_pos:,} ({pct(CHURN_RATE)})")
    print(f"Tidak churn (0) : {n_tot - n_pos:,} ({pct(1 - CHURN_RATE)})")
    imb = "severe" if CHURN_RATE < 0.1 else "moderate" if CHURN_RATE < 0.25 else "mild/none"
    print("Tingkat imbalance:", imb)
    if CHURN_RATE < 0.1:
        print("  → Jika metrik = accuracy, waspada: model dummy sudah dapat", pct(1 - CHURN_RATE))
        print("  → Pertimbangkan PR-AUC/F1 untuk evaluasi internal & threshold tuning.")

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    counts = pd.Series(y).map({0: "Retained (0)", 1: "Churned (1)"}).value_counts()
    ax[0].bar(counts.index, counts.values, color=[CFG.PALETTE[0], CFG.PALETTE[1]])
    for i, v in enumerate(counts.values):
        ax[0].text(i, v, f"{v:,}\n({v / n_tot:.1%})", ha="center", va="bottom")
    ax[0].set_title("Customer Churn Distribution")
    ax[0].set_ylabel("Number of customers")
    ax[0].set_ylim(0, counts.max() * 1.25)
    ax[1].pie([n_tot - n_pos, n_pos], labels=["Retained", "Churned"], autopct="%1.1f%%",
              colors=[CFG.PALETTE[0], CFG.PALETTE[1]], startangle=90, wedgeprops={"width": 0.45})
    ax[1].set_title("Churn Rate")
    save_fig("target_distribution")

    # baseline
    base_prob = np.full(n_tot, CHURN_RATE)
    print("\nBaseline (tanpa model):")
    print(f"  ROC-AUC  = 0.5000 | PR-AUC = {CHURN_RATE:.4f} | LogLoss = {log_loss(y, base_prob):.4f} | "
          f"Accuracy (tebak mayoritas) = {max(CHURN_RATE, 1 - CHURN_RATE):.4f}")
    add_insight("Data", f"The training data contains {n_tot:,} customers with an overall churn rate of {pct(CHURN_RATE)} ({imb} class imbalance).")

# 5. Column Roles & Domain Feature Engineering (Asuransi)

## 5.1 Column roles
Role = arti bisnis kolom (tenure, premi, umur, klaim, keluhan, metode bayar, channel, ...). Dipakai untuk:
- membuat **fitur domain** yang masuk akal,
- **survival analysis** (butuh `tenure`),
- **nilai customer / CLV** (butuh `premium`),
- **rekomendasi** (teks rekomendasi disesuaikan role).

**Cek output di bawah!** Kalau ada role salah/kurang → `CFG.COLUMN_ROLES = {"tenure": "nama_kolom", ...}` lalu run ulang dari Section 3.4.

## 5.2 Fitur domain asuransi yang dibuat (kalau kolomnya ada)
| Fitur | Rumus | Intuisi bisnis |
|---|---|---|
| `fe_tenure_years`, `fe_tenure_band` | tenure → tahun & band (0-6m, 6-12m, 1-2y, 2-5y, 5y+) | churn biasanya tertinggi di tahun pertama |
| `fe_is_new_customer` | tenure < 12 bulan | fase onboarding kritis |
| `fe_age_band` | <25, 25-34, ..., 65+ | life stage berbeda kebutuhan |
| `fe_premium_to_income` | premi / pendapatan | **affordability** — beban premi tinggi → rawan lapse |
| `fe_sum_insured_to_premium` | UP / premi | value-for-money |
| `fe_claims_per_year` | klaim / tenure (tahun) | intensitas pemakaian produk |
| `fe_has_claim` | klaim > 0 | pernah merasakan layanan klaim |
| `fe_claim_reject_rate` | klaim ditolak / klaim | pengalaman klaim buruk |
| `fe_has_complaint`, `fe_has_late_payment` | > 0 | sinyal ketidakpuasan / kesulitan bayar |
| `fe_premium_increase`, `fe_big_premium_increase` | perubahan premi > 0 / > 10% | **price shock** saat renewal |
| `fe_multi_product` | produk > 1 | bundling → switching cost lebih tinggi |
| `fe_avg_paid_per_month` | total dibayar / tenure | konsistensi pembayaran |
| `<date>_days_since`, `<date>_year`, `<date>_month` | dari kolom tanggal | recency & seasonality |
| `fe_n_missing` | jumlah missing per baris | data tidak lengkap sering = customer kurang engaged |

Tambahkan fitur sendiri di fungsi `custom_feature_engineering(df)`.

**Peringatan leakage tanggal:** kolom seperti `cancel_date`, `end_date`, `termination_date` hanya terisi untuk yang churn → **LEAKAGE**. Notebook menandainya (Section 12). Jangan dipakai sebagai fitur.

In [ ]:
def infer_tenure_unit(s):
    if CFG.TENURE_UNIT != "auto":
        return CFG.TENURE_UNIT
    mx = s.quantile(0.99)
    if mx > 400:
        return "days"
    if mx <= 40 and s.max() <= 60:
        return "years"
    return "months"


def custom_feature_engineering(df):
    """HOOK: tambahkan fitur spesifik dataset lomba di sini. df = train atau test (sudah bersih).
    Contoh:
        if {"annual_premium", "n_products"} <= set(df.columns):
            df["premium_per_product"] = df["annual_premium"] / df["n_products"].clip(lower=1)
    """
    return df


def domain_feature_engineering(df, roles, ref_date):
    df = df.copy()
    new = []
    r = roles

    def has(role):
        return r.get(role) in df.columns and pd.api.types.is_numeric_dtype(df[r[role]])

    if has("tenure"):
        t = df[r["tenure"]].astype(float)
        unit = TENURE_UNIT
        months = t if unit == "months" else t * 12 if unit == "years" else t / 30.44
        df["fe_tenure_years"] = months / 12
        df["fe_tenure_band"] = pd.cut(months, [-np.inf, 6, 12, 24, 60, np.inf],
                                      labels=["0-6m", "6-12m", "1-2y", "2-5y", "5y+"]).astype(str).replace("nan", np.nan)
        df["fe_is_new_customer"] = (months < 12).astype(float).where(t.notna())
        new += ["fe_tenure_years", "fe_tenure_band", "fe_is_new_customer"]
    if has("age"):
        df["fe_age_band"] = pd.cut(df[r["age"]], [-np.inf, 25, 35, 45, 55, 65, np.inf],
                                   labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"]).astype(str).replace("nan", np.nan)
        new.append("fe_age_band")
    if has("premium") and has("income"):
        prem = df[r["premium"]] * (12 if CFG.VALUE_PERIOD == "monthly" else 1)
        df["fe_premium_to_income"] = prem / df[r["income"]].replace(0, np.nan)
        new.append("fe_premium_to_income")
    if has("premium") and has("sum_insured"):
        df["fe_sum_insured_to_premium"] = df[r["sum_insured"]] / df[r["premium"]].replace(0, np.nan)
        new.append("fe_sum_insured_to_premium")
    if has("claims"):
        df["fe_has_claim"] = (df[r["claims"]] > 0).astype(float).where(df[r["claims"]].notna())
        new.append("fe_has_claim")
        if "fe_tenure_years" in df:
            df["fe_claims_per_year"] = df[r["claims"]] / df["fe_tenure_years"].clip(lower=0.25)
            new.append("fe_claims_per_year")
        rej = [c for c in df.columns if "claim" in c.lower() and any(k in c.lower() for k in ["reject", "tolak", "denied", "declin"])
               and pd.api.types.is_numeric_dtype(df[c])]
        if rej:
            df["fe_claim_reject_rate"] = (df[rej[0]] / df[r["claims"]].replace(0, np.nan)).fillna(0)
            new.append("fe_claim_reject_rate")
    if has("complaints"):
        df["fe_has_complaint"] = (df[r["complaints"]] > 0).astype(float).where(df[r["complaints"]].notna())
        new.append("fe_has_complaint")
    if has("late_payment"):
        df["fe_has_late_payment"] = (df[r["late_payment"]] > 0).astype(float).where(df[r["late_payment"]].notna())
        new.append("fe_has_late_payment")
    if has("premium_change"):
        pc = df[r["premium_change"]]
        df["fe_premium_increase"] = (pc > 0).astype(float).where(pc.notna())
        big = 10 if pc.abs().quantile(0.95) > 1 else 0.10   # persen vs fraksi
        df["fe_big_premium_increase"] = (pc > big).astype(float).where(pc.notna())
        new += ["fe_premium_increase", "fe_big_premium_increase"]
    if has("products"):
        df["fe_multi_product"] = (df[r["products"]] > 1).astype(float).where(df[r["products"]].notna())
        new.append("fe_multi_product")
    if has("total_paid") and has("tenure"):
        df["fe_avg_paid_per_period"] = df[r["total_paid"]] / df[r["tenure"]].replace(0, np.nan)
        new.append("fe_avg_paid_per_period")
        if has("premium"):
            df["fe_paid_vs_expected"] = df[r["total_paid"]] / (df[r["premium"]] * df[r["tenure"]]).replace(0, np.nan)
            new.append("fe_paid_vs_expected")
    # tanggal → days since + year + month
    for c in [c for c, t in COL_TYPES.items() if t == "date" and c in df.columns]:
        d = pd.to_datetime(df[c], errors="coerce")
        df[f"{c}_days_since"] = (ref_date - d).dt.days
        df[f"{c}_year"] = d.dt.year
        df[f"{c}_month"] = d.dt.month
        new += [f"{c}_days_since", f"{c}_year", f"{c}_month"]
    if CFG.USE_ROW_STATS:
        base_cols = [c for c, t in COL_TYPES.items() if t not in ("id", "dropped", "constant") and c in df.columns
                     and c in ORIGINAL_COLS and c not in LEAK_NAME_COLS]
        df["fe_n_missing"] = df[base_cols].isna().sum(axis=1)
        new.append("fe_n_missing")
    df = custom_feature_engineering(df)
    return df, new


FE_NEW, FEATURE_FAMILY, TENURE_UNIT, REF_DATE = [], {}, None, pd.Timestamp.today().normalize()
with safe_section("Column roles & domain FE"):
    LEAK_NAME_COLS = [c for c in train.columns if any(k in c.lower() for k in CFG.LEAKAGE_KEYWORDS)]
    ROLES = detect_roles(train, COL_TYPES)
    print("Role yang terdeteksi (cek & override via CFG.COLUMN_ROLES kalau salah):")
    for k, v in ROLES.items():
        print(f"  {k:<18} → {v}")
    missing_roles = [k for k in ["tenure", "premium", "age", "claims", "complaints", "payment_method", "channel"] if k not in ROLES]
    if missing_roles:
        print("  (tidak ditemukan:", ", ".join(missing_roles), ")")

    TENURE_UNIT = infer_tenure_unit(train[ROLES["tenure"]].dropna()) if "tenure" in ROLES else None
    if TENURE_UNIT:
        print(f"Satuan tenure: {TENURE_UNIT}")

    date_cols = [c for c, t in COL_TYPES.items() if t == "date"]
    date_series = [pd.to_datetime(train[c], errors="coerce") for c in date_cols]
    if test is not None:
        date_series += [pd.to_datetime(test[c], errors="coerce") for c in date_cols if c in test.columns]
    all_dates = pd.concat(date_series) if date_series else pd.Series([], dtype="datetime64[ns]")
    REF_DATE = pd.Timestamp(CFG.REFERENCE_DATE) if CFG.REFERENCE_DATE else (all_dates.max() if all_dates.notna().any() else pd.Timestamp.today().normalize())
    if date_cols:
        print("Kolom tanggal:", date_cols, "| reference date:", REF_DATE.date())

    if CFG.USE_DOMAIN_FE:
        train, FE_NEW = domain_feature_engineering(train, ROLES, REF_DATE)
        if test is not None:
            test, _ = domain_feature_engineering(test, ROLES, REF_DATE)
        for c in FE_NEW:
            s = train[c]
            if s.dtype == object:
                COL_TYPES[c] = "categorical"
            else:
                COL_TYPES[c] = "binary" if s.nunique() == 2 else "numeric"
        print(f"\n{len(FE_NEW)} fitur domain dibuat:", FE_NEW)
    else:
        FE_NEW = []

    # Feature family: fitur yang membawa informasi sama (raw vs turunan) → dipakai untuk menghindari laporan dobel
    FEATURE_FAMILY = {}
    fam_members = {
        "tenure": [ROLES.get("tenure"), "fe_tenure_years", "fe_tenure_band", "fe_is_new_customer"]
                  + [f"{d}_{s}" for d in [c for c, t in COL_TYPES.items() if t == "date"] for s in ("days_since", "year")],
        "age": [ROLES.get("age"), "fe_age_band"],
        "claims": [ROLES.get("claims"), "fe_has_claim"],
        "complaints": [ROLES.get("complaints"), "fe_has_complaint"],
        "late_payment": [ROLES.get("late_payment"), "fe_has_late_payment"],
        "products": [ROLES.get("products"), "fe_multi_product"],
        "premium_change": [ROLES.get("premium_change"), "fe_premium_increase", "fe_big_premium_increase"],
    }
    for fam, members in fam_members.items():
        for m in members:
            if m and m in train.columns:
                FEATURE_FAMILY[m] = fam


with safe_section("Final feature lists", critical=True):
    # Daftar fitur final untuk analisis (section terpisah supaya tetap terbentuk walau FE gagal)
    NUM_COLS = [c for c, t in COL_TYPES.items() if t in ("numeric", "binary") and c in train.columns]
    CAT_COLS = [c for c, t in COL_TYPES.items() if t in ("categorical", "binary_cat") and c in train.columns]
    HIGH_CARD_COLS = [c for c, t in COL_TYPES.items() if t == "high_card" and c in train.columns]
    EXCLUDED = [c for c, t in COL_TYPES.items() if t in ("id", "dropped", "constant", "date", "text")]
    for c in CAT_COLS + HIGH_CARD_COLS:
        train[c] = train[c].astype("object")
        if test is not None:
            test[c] = test[c].astype("object")
    print(f"\nNumeric/binary : {len(NUM_COLS)}")
    print(f"Categorical    : {len(CAT_COLS)}")
    print(f"High-card cat  : {len(HIGH_CARD_COLS)} {HIGH_CARD_COLS}")
    print(f"Excluded       : {EXCLUDED}")

## 5.3 Early Leakage Screen (WAJIB dicek)

**Leakage** = fitur yang "bocor" informasi target, biasanya karena baru tercatat **setelah** customer churn (misal `cancellation_reason`, `termination_date`, `refund_amount`, `status_akhir`). Model jadi kelihatan super akurat (AUC 0.99) tapi **tidak berguna** dan insight-nya salah total.

Tiga cara deteksi otomatis:
1. **Nama kolom** mengandung kata kunci (`cancel, terminat, reason, end_date, refund, batal, ...`).
2. **Missingness = target**: kolom yang kosong hampir hanya untuk non-churner (atau sebaliknya). Contoh klasik: `cancellation_reason` hanya terisi kalau churn.
3. **Single-feature AUC** ≥ `CFG.LEAKAGE_AUC_THRESHOLD` (default 0.97).

**Jika X → Y:**
- Ada fitur ter-flag & secara logika bisnis memang terjadi setelah churn → **buang** (`CFG.DROP_COLS` atau `CFG.AUTO_DROP_LEAKY = True`).
- Ter-flag tapi secara bisnis valid diketahui sebelum churn (jarang) → boleh dipertahankan, tapi jelaskan di laporan.
- Kalau test set juga punya kolom leak itu dan metrik lomba murni prediksi → diskusikan dengan tim; untuk **analisis & rekomendasi** tetap jangan pakai.

**Untuk laporan:** sebutkan bahwa fitur post-event dikeluarkan untuk mencegah data leakage → menunjukkan kematangan metodologi (juri suka ini).

In [ ]:
def quick_auc(x, y_arr):
    """AUC univariate cepat: numeric → rank AUC; kategori → churn rate per level (out-of-fold)."""
    try:
        if pd.api.types.is_numeric_dtype(x) and x.nunique() > 2:
            v = x.fillna(x.median() if x.notna().any() else 0)
            a = roc_auc_score(y_arr, v)
            return max(a, 1 - a)
        lab = x.astype(str).where(x.notna(), "Missing").values
        pred = np.zeros(len(y_arr))
        prior = y_arr.mean()
        for tr, va in StratifiedKFold(5, shuffle=True, random_state=CFG.SEED).split(lab, y_arr):
            m = pd.DataFrame({"l": lab[tr], "y": y_arr[tr]}).groupby("l")["y"].agg(["sum", "count"])
            rate = (m["sum"] + 10 * prior) / (m["count"] + 10)
            pred[va] = pd.Series(lab[va]).map(rate).fillna(prior).values
        a = roc_auc_score(y_arr, pred)
        return max(a, 1 - a)
    except Exception:
        return np.nan


LEAKY_COLS = []

with safe_section("Early leakage screen"):
    rows = []
    for c in NUM_COLS + CAT_COLS + HIGH_CARD_COLS + [c for c, t in COL_TYPES.items() if t == "date" and c in train.columns]:
        s = train[c]
        name_hit = any(k in c.lower() for k in CFG.LEAKAGE_KEYWORDS)
        miss_auc = np.nan
        if s.isna().any() and s.notna().any():
            try:
                miss_auc = roc_auc_score(y, s.isna().astype(int))
                miss_auc = max(miss_auc, 1 - miss_auc)
            except Exception:
                pass
        auc = quick_auc(s, y) if COL_TYPES.get(c) != "date" else np.nan
        stat_hit = (miss_auc >= 0.95) or (auc >= CFG.LEAKAGE_AUC_THRESHOLD)
        flag = name_hit or stat_hit
        rows.append({"feature": c, "name_keyword": name_hit, "missingness_auc": miss_auc, "single_feature_auc": auc,
                     "statistical_evidence": stat_hit, "flagged": flag})
    leak_df = pd.DataFrame(rows).sort_values("single_feature_auc", ascending=False)
    save_table(leak_df, "leakage_screen")
    display(leak_df.head(15))
    LEAKY_COLS = leak_df[leak_df["flagged"]]["feature"].tolist()
    if LEAKY_COLS:
        print("🚨 Fitur dicurigai LEAKAGE:", LEAKY_COLS)
        for c in LEAKY_COLS:
            r = leak_df.set_index("feature").loc[c]
            why = []
            if r["name_keyword"]:
                why.append("nama kolom mengandung kata kunci post-event")
            if r["missingness_auc"] >= 0.95:
                why.append(f"pola missing hampir identik dengan target (AUC {r['missingness_auc']:.3f})")
            if r["single_feature_auc"] >= CFG.LEAKAGE_AUC_THRESHOLD:
                why.append(f"single-feature AUC {r['single_feature_auc']:.3f}")
            print(f"   - {c}: " + "; ".join(why))
        NAME_ONLY = leak_df[leak_df["flagged"] & ~leak_df["statistical_evidence"]]["feature"].tolist()
        if NAME_ONLY:
            print(f"   ℹ️ Hanya cocok nama (tanpa bukti statistik) → TIDAK dibuang otomatis, cek manual: {NAME_ONLY}")
        LEAKY_COLS = [c for c in LEAKY_COLS if c not in NAME_ONLY] if CFG.AUTO_DROP_LEAKY else LEAKY_COLS
        if CFG.AUTO_DROP_LEAKY and LEAKY_COLS:
            for c in LEAKY_COLS:
                COL_TYPES[c] = "dropped"
            NUM_COLS = [c for c in NUM_COLS if c not in LEAKY_COLS]
            CAT_COLS = [c for c in CAT_COLS if c not in LEAKY_COLS]
            HIGH_CARD_COLS = [c for c in HIGH_CARD_COLS if c not in LEAKY_COLS]
            EXCLUDED = EXCLUDED + LEAKY_COLS
            print(f"   → AUTO_DROP_LEAKY=True: {LEAKY_COLS} DIBUANG dari model & analisis driver (tetap dipakai deskriptif di 7.3 kalau kolom alasan).")
            add_insight("Data", f"Post-event variables ({', '.join(LEAKY_COLS)}) were excluded to prevent target leakage.")
        elif not CFG.AUTO_DROP_LEAKY:
            print("   → AUTO_DROP_LEAKY=False: fitur TIDAK dibuang. Kalau setuju itu leak, tambahkan ke CFG.DROP_COLS atau set CFG.AUTO_DROP_LEAKY=True lalu run ulang.")
    else:
        print("✅ Tidak ada fitur yang dicurigai leakage.")

# 6. EDA Univariate

Tujuan: kenal bentuk data **sebelum** melihat hubungan dengan churn.

**Cara baca & Jika X → Y:**
- **Missing bar**: kolom dengan missing tinggi → cek apakah missing-nya bermakna (misal `avg_claim_settlement_days` kosong karena tidak pernah klaim → bukan error, itu informasi!).
- **Histogram numeric**: skew tinggi (> 1) → untuk model linear perlu log-transform (dilakukan otomatis di LogReg pipeline via scaler; tree model tidak peduli). Nilai ekstrem → cek apakah salah input.
- **Bar kategori**: level dengan sangat sedikit data → churn rate-nya tidak stabil (lihat CI di section 7), di model linear digabung ke `__RARE__`.

**Untuk laporan:** cukup 1 kalimat profil data (misal "Most customers hold Health policies (35%) and pay monthly (50%)") — jangan habiskan halaman untuk univariate.

In [ ]:
def grid_axes(n, ncols=3, w=5, h=3.4):
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows))
    axes = np.array(axes).reshape(-1)
    for a in axes[n:]:
        a.axis("off")
    return fig, axes[:n]


with safe_section("EDA univariate"):
    miss = train[NUM_COLS + CAT_COLS + HIGH_CARD_COLS].isna().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    if len(miss):
        fig, ax = plt.subplots(figsize=(8, max(2.5, 0.3 * len(miss[:30]))))
        ax.barh(miss.index[:30][::-1], 100 * miss.values[:30][::-1], color=CFG.PALETTE[3])
        ax.set_xlabel("Missing (%)")
        ax.set_title("Missing Values by Feature (train)")
        save_fig("missing_values")
    else:
        print("Tidak ada missing value di fitur. ✅")

    desc = train[NUM_COLS].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
    desc["skew"] = train[NUM_COLS].skew()
    desc["missing_%"] = 100 * train[NUM_COLS].isna().mean()
    save_table(desc.reset_index().rename(columns={"index": "feature"}), "numeric_summary")
    display(desc)

    cont = [c for c in NUM_COLS if train[c].nunique() > 2][: CFG.MAX_EDA_PLOTS]
    if cont:
        fig, axes = grid_axes(len(cont), ncols=4, w=4, h=3)
        for a, c in zip(axes, cont):
            v = train[c].dropna()
            lo, hi = v.quantile([0.005, 0.995])
            a.hist(v.clip(lo, hi), bins=40, color=CFG.PALETTE[0], alpha=0.85)
            a.set_title(c[:35], fontsize=9)
        fig.suptitle("Numeric Feature Distributions (clipped at 0.5–99.5 pct)", y=1.01)
        plt.tight_layout()
        save_fig("numeric_distributions")

    cats = (CAT_COLS + [c for c in NUM_COLS if train[c].nunique() == 2])[: CFG.MAX_EDA_PLOTS]
    if cats:
        fig, axes = grid_axes(len(cats), ncols=4, w=4, h=3)
        for a, c in zip(axes, cats):
            vc = train[c].astype(str).replace("nan", "Missing").value_counts().head(CFG.MAX_CAT_LEVELS_PLOT)
            a.barh(vc.index[::-1].astype(str), vc.values[::-1], color=CFG.PALETTE[4])
            a.set_title(c[:35], fontsize=9)
            a.tick_params(axis="y", labelsize=7)
        fig.suptitle("Categorical Feature Frequencies", y=1.01)
        plt.tight_layout()
        save_fig("categorical_frequencies")

# 7. Driver Analysis — faktor apa yang berhubungan dengan churn?

**Ini section paling penting untuk laporan.** Untuk setiap fitur dihitung:

| Ukuran | Untuk | Arti | Cara baca |
|---|---|---|---|
| **Churn rate per level/bin** + 95% Wilson CI | semua | churn rate tiap kategori / tiap kuantil | CI tidak overlap dengan garis overall → beda nyata |
| **Lift** | semua | churn rate level ÷ churn rate overall | 2.0 = dua kali lebih rawan dari rata-rata |
| **Chi-square test** | kategori | ada asosiasi atau tidak | p < 0.05 → ada asosiasi |
| **Cramér's V** | kategori | **besar** asosiasi (effect size) | <0.1 negligible, 0.1–0.3 small, 0.3–0.5 medium, >0.5 large |
| **Mann-Whitney U** | numeric | distribusi churner vs non-churner beda? | p < 0.05 → beda |
| **Rank-biserial r** | numeric | effect size + **arah** (+ = nilai tinggi → lebih churn) | \|r\| <0.1 negligible, 0.1–0.3 small, 0.3–0.5 medium, >0.5 large |
| **Information Value (IV)** | semua | kekuatan prediktif (standar credit/insurance scoring) | <0.02 useless, 0.02–0.1 weak, 0.1–0.3 medium, 0.3–0.5 strong, **>0.5 suspicious (cek leakage!)** |
| **Univariate AUC** | semua | seberapa baik fitur ini sendirian memisahkan churn | 0.5 = acak; >0.7 kuat; **>0.95 hampir pasti leakage** |
| **q-value (Benjamini-Hochberg)** | semua | p-value terkoreksi multiple testing | pakai q < 0.05 untuk klaim "signifikan" |

**Kenapa effect size penting?** Dengan data ribuan baris, hampir semua p-value < 0.05. Yang membedakan driver penting vs remeh adalah **effect size & IV**, bukan p-value. Di laporan tulis keduanya: *"Payment method is strongly associated with churn (χ² p < 0.001, Cramér's V = 0.32)."*

**Jika X → Y:**
- IV > 0.5 atau AUC > 0.95 → **curigai leakage** (variabel yang hanya ada setelah churn terjadi). Cek Section 12.
- Fitur numeric dengan hubungan U-shape (churn tinggi di dua ujung, misal umur) → rank-biserial bisa kecil padahal IV besar → lihat plot churn-rate-per-bin, jangan hanya r.
- Level dengan n kecil (< 30) dan churn rate ekstrem → jangan dijadikan klaim utama (CI lebar).

**Untuk laporan:** tampilkan **tabel top 10 driver** (feature, IV, effect size, churn rate level berisiko vs overall) + 1 figure panel churn-rate per driver. Ini inti bagian "Findings".

In [ ]:
BAND_ORDERS = [["0-6m", "6-12m", "1-2y", "2-5y", "5y+"], ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]]


def fmt_edge(v):
    if not np.isfinite(v):
        return "∞" if v > 0 else "-∞"
    a = abs(v)
    if a >= 1e9:
        return f"{v / 1e9:.3g}B"
    if a >= 1e6:
        return f"{v / 1e6:.3g}M"
    if a >= 1e4:
        return f"{v / 1e3:.3g}K"
    if a >= 100 or float(v).is_integer():
        return f"{v:,.0f}"
    return f"{v:.3g}"


def interval_labels(cats):
    return [f"{fmt_edge(iv.left)}–{fmt_edge(iv.right)}" for iv in cats]


def bin_feature(x, numeric, n_bins=None, max_levels=None):
    """Kembalikan series label bin/level (string) dengan NaN → 'Missing'."""
    n_bins = n_bins or CFG.N_BINS_NUMERIC
    max_levels = max_levels or CFG.MAX_CAT_LEVELS_PLOT
    if numeric and x.nunique() > 10:
        try:
            b = pd.qcut(x, q=n_bins, duplicates="drop")
        except Exception:
            b = pd.cut(x, bins=n_bins)
        nice = interval_labels(b.cat.categories)
        if len(set(nice)) < len(nice):  # label bentrok setelah pembulatan → pakai label asli
            nice = [str(i) for i in b.cat.categories]
        b = b.cat.rename_categories(nice)
        labels = b.astype(str)
        order = list(nice)
        labels = labels.where(x.notna(), "Missing")
        return labels, order + (["Missing"] if x.isna().any() else [])
    lab = x.astype(str).where(x.notna(), "Missing")
    if numeric:
        try:
            u = sorted(x.dropna().unique())
            fmt_v = lambda v: f"{v:g}" if isinstance(v, (int, float, np.integer, np.floating)) else str(v)
            min_n = max(30, int(0.01 * len(x)))
            cap = None
            for i, v in enumerate(u):           # gabung ekor yang jarang (misal 4, 5, 6 keluhan) jadi "≥k"
                if i > 0 and (x >= v).sum() < min_n:
                    cap = u[i - 1]
                    break
            if cap is not None and cap != u[0]:
                lab = x.map(lambda v: "Missing" if pd.isna(v) else (f"≥{fmt_v(cap)}" if v >= cap else fmt_v(v)))
                order = [fmt_v(v) for v in u if v < cap] + [f"≥{fmt_v(cap)}"]
            else:
                order = [fmt_v(v) for v in u]
                lab = x.map(lambda v: fmt_v(v) if pd.notna(v) else "Missing")
        except Exception:
            order = list(lab.value_counts().index)
    else:
        vc = lab.value_counts()
        if len(vc) > max_levels:
            keep = set(vc.index[: max_levels - 1])
            lab = lab.where(lab.isin(keep), "Other")
        order = list(lab.value_counts().index)
        for bo in BAND_ORDERS:
            if set(order) - {"Missing", "Other"} <= set(bo):
                order = [o for o in bo if o in set(order)] + [o for o in order if o in ("Other", "Missing")]
                break
    if x.isna().any() and "Missing" not in order:
        order.append("Missing")
    return lab, order


def rate_table(labels, order, y_arr):
    df = pd.DataFrame({"level": labels.values, "y": y_arr})
    g = df.groupby("level")["y"].agg(["count", "sum"]).reindex([o for o in order if o in set(df["level"])])
    g = g.rename(columns={"count": "n", "sum": "churners"})
    g["churn_rate"] = g["churners"] / g["n"]
    ci = [wilson_ci(k, n) for k, n in zip(g["churners"], g["n"])]
    g["ci_low"], g["ci_high"] = [c[0] for c in ci], [c[1] for c in ci]
    g["lift"] = g["churn_rate"] / max(y_arr.mean(), 1e-9)
    g["share_%"] = 100 * g["n"] / g["n"].sum()
    return g.reset_index()


def information_value(tbl):
    good = (tbl["n"] - tbl["churners"]).astype(float) + 0.5
    bad = tbl["churners"].astype(float) + 0.5
    dg, db = good / good.sum(), bad / bad.sum()
    woe = np.log(dg / db)
    return float(((dg - db) * woe).sum()), woe.values


def iv_label(iv):
    return ("useless" if iv < 0.02 else "weak" if iv < 0.1 else "medium" if iv < 0.3 else "strong" if iv < 0.5 else "suspicious(>0.5)")


def effect_label(v):
    v = abs(v)
    return "negligible" if v < 0.1 else "small" if v < 0.3 else "medium" if v < 0.5 else "large"


def cramers_v_from_ct(ct):
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return 0.0, 1.0
    chi2, p, dof, _ = stats.chi2_contingency(ct, correction=False)
    n = ct.values.sum()
    v = math.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    return v, p


def oof_rate_auc(labels, y_arr, n_splits=5):
    """AUC univariate jujur untuk kategori: churn rate per level dihitung out-of-fold."""
    pred = np.zeros(len(y_arr))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=CFG.SEED)
    lab = labels.values
    prior = y_arr.mean()
    for tr, va in skf.split(lab, y_arr):
        m = pd.DataFrame({"l": lab[tr], "y": y_arr[tr]}).groupby("l")["y"].agg(["sum", "count"])
        rate = (m["sum"] + 10 * prior) / (m["count"] + 10)
        pred[va] = pd.Series(lab[va]).map(rate).fillna(prior).values
    try:
        return roc_auc_score(y_arr, pred)
    except Exception:
        return np.nan


def analyze_feature(c, x, y_arr):
    numeric = pd.api.types.is_numeric_dtype(x)
    labels, order = bin_feature(x, numeric)
    tbl = rate_table(labels, order, y_arr)
    iv, woe = information_value(tbl)
    tbl["woe"] = woe
    res = {"feature": c, "type": "numeric" if numeric and x.nunique() > 2 else "categorical",
           "n_levels": int(x.nunique()), "missing_%": round(100 * x.isna().mean(), 2), "iv": iv}
    if numeric and x.nunique() > 2:
        a, b = x[y_arr == 1].dropna(), x[y_arr == 0].dropna()
        try:
            u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
            auc = u / (len(a) * len(b))
        except Exception:
            p, auc = np.nan, 0.5
        r = 2 * auc - 1
        res.update({"test": "Mann-Whitney U", "p_value": p, "effect_size": r, "effect_metric": "rank-biserial r",
                    "univariate_auc": max(auc, 1 - auc),
                    "mean_churn": a.mean(), "mean_retained": b.mean(), "median_churn": a.median(), "median_retained": b.median(),
                    "direction": "higher → more churn" if r > 0.05 else "higher → less churn" if r < -0.05 else "weak/non-monotonic"})
    else:
        ct = pd.crosstab(labels, y_arr)
        v, p = cramers_v_from_ct(ct)
        res.update({"test": "Chi-square", "p_value": p, "effect_size": v, "effect_metric": "Cramér's V",
                    "univariate_auc": oof_rate_auc(labels, y_arr), "direction": "see level table"})
    valid = tbl[tbl["n"] >= max(30, 0.01 * len(y_arr))]
    if len(valid) == 0:
        valid = tbl
    worst = valid.sort_values("churn_rate", ascending=False).iloc[0]
    best = valid.sort_values("churn_rate", ascending=True).iloc[0]
    res.update({"highest_risk_level": worst["level"], "highest_risk_rate": worst["churn_rate"], "highest_risk_lift": worst["lift"],
                "highest_risk_n": int(worst["n"]), "lowest_risk_level": best["level"], "lowest_risk_rate": best["churn_rate"]})
    return res, tbl


def dedupe_features(feats, max_corr=0.95):
    """Buang fitur yang informasinya duplikat: satu family yang sama, atau |Spearman| > max_corr dengan fitur terpilih."""
    chosen, fams = [], set()
    for f in feats:
        fam = FEATURE_FAMILY.get(f)
        if fam is not None and fam in fams:
            continue
        dup = False
        if f in train.columns and pd.api.types.is_numeric_dtype(train[f]):
            for g in chosen:
                if pd.api.types.is_numeric_dtype(train[g]):
                    try:
                        if abs(train[[f, g]].corr(method="spearman").iloc[0, 1]) > max_corr:
                            dup = True
                            break
                    except Exception:
                        pass
        if dup:
            continue
        chosen.append(f)
        if fam is not None:
            fams.add(fam)
    return chosen


DRIVER_TABLE = pd.DataFrame()
LEVEL_TABLES = {}

with safe_section("Driver analysis"):
    ANALYSIS_FEATURES = NUM_COLS + CAT_COLS
    rows = []
    for c in ANALYSIS_FEATURES:
        try:
            res, tbl = analyze_feature(c, train[c], y)
            rows.append(res)
            LEVEL_TABLES[c] = tbl
        except Exception as e:
            print(f"  skip {c}: {e}")
    DRIVER_TABLE = pd.DataFrame(rows)
    pv = DRIVER_TABLE["p_value"].fillna(1).values
    if HAS_SM_SURV:
        DRIVER_TABLE["q_value_BH"] = multipletests(pv, method="fdr_bh")[1]
    else:
        order_ = np.argsort(pv)
        q = pv[order_] * len(pv) / (np.arange(len(pv)) + 1)
        q = np.minimum.accumulate(q[::-1])[::-1]
        DRIVER_TABLE["q_value_BH"] = np.empty_like(q)
        DRIVER_TABLE.loc[order_, "q_value_BH"] = np.clip(q, 0, 1)
    DRIVER_TABLE["iv_strength"] = DRIVER_TABLE["iv"].apply(iv_label)
    DRIVER_TABLE["effect_label"] = DRIVER_TABLE["effect_size"].apply(effect_label)
    DRIVER_TABLE["significant"] = DRIVER_TABLE["q_value_BH"] < 0.05
    DRIVER_TABLE = DRIVER_TABLE.sort_values("iv", ascending=False).reset_index(drop=True)

    # Mutual information (encoded)
    try:
        Xmi = train[ANALYSIS_FEATURES].copy()
        disc = []
        for c in ANALYSIS_FEATURES:
            if not pd.api.types.is_numeric_dtype(Xmi[c]):
                Xmi[c] = Xmi[c].astype(str).astype("category").cat.codes
                disc.append(True)
            else:
                Xmi[c] = Xmi[c].fillna(Xmi[c].median())
                disc.append(Xmi[c].nunique() <= 10)
        mi = mutual_info_classif(Xmi.fillna(-1), y, discrete_features=np.array(disc), random_state=CFG.SEED)
        DRIVER_TABLE["mutual_info"] = DRIVER_TABLE["feature"].map(dict(zip(ANALYSIS_FEATURES, mi)))
    except Exception as e:
        print("MI gagal:", e)

    show_cols = ["feature", "type", "iv", "iv_strength", "effect_metric", "effect_size", "effect_label", "p_value", "q_value_BH",
                 "univariate_auc", "mutual_info", "direction", "highest_risk_level", "highest_risk_rate", "highest_risk_lift",
                 "lowest_risk_level", "lowest_risk_rate"]
    DRIVER_TABLE = DRIVER_TABLE[[c for c in show_cols if c in DRIVER_TABLE] + [c for c in DRIVER_TABLE if c not in show_cols]]
    save_table(DRIVER_TABLE, "driver_table")
    display(DRIVER_TABLE.head(30))

    sus = DRIVER_TABLE[(DRIVER_TABLE["iv"] > 0.5) | (DRIVER_TABLE["univariate_auc"] > CFG.LEAKAGE_AUC_THRESHOLD)]
    if len(sus):
        print("⚠️ Fitur dengan IV > 0.5 atau AUC sangat tinggi (cek leakage di Section 12):", sus["feature"].tolist())
    print(f"\nFitur signifikan (q<0.05): {DRIVER_TABLE['significant'].sum()} dari {len(DRIVER_TABLE)}")

### 7.1 Visualisasi driver: churn rate per level/bin
Garis putus-putus merah = churn rate overall. Error bar = 95% CI. Angka di bar = jumlah customer.
**Cara baca:** bar jauh di atas garis merah dengan CI pendek → segmen berisiko tinggi yang **bisa dipercaya**.

In [ ]:
def plot_rate(ax, tbl, title, overall):
    t = tbl.copy()
    xlab = t["level"].astype(str).str.slice(0, 22)
    colors = [CFG.PALETTE[1] if r > overall else CFG.PALETTE[0] for r in t["churn_rate"]]
    ax.bar(range(len(t)), t["churn_rate"], color=colors, alpha=0.85,
           yerr=[(t["churn_rate"] - t["ci_low"]).clip(lower=0), (t["ci_high"] - t["churn_rate"]).clip(lower=0)], capsize=2, ecolor="#555")
    ax.axhline(overall, color="red", ls="--", lw=1)
    ax.set_xticks(range(len(t)))
    ax.set_xticklabels(xlab, rotation=45, ha="right", fontsize=7)
    ax.set_title(title[:45], fontsize=9)
    ax.set_ylabel("Churn rate")
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
    for i, n in enumerate(t["n"]):
        ax.text(i, 0, f"{n}", ha="center", va="bottom", fontsize=6, color="white" if t["churn_rate"].iloc[i] > 0.05 else "black")


with safe_section("Driver plots"):
    top_feats = dedupe_features([f for f in DRIVER_TABLE["feature"].tolist() if f in LEVEL_TABLES])[: CFG.MAX_EDA_PLOTS]
    print("(Panel memakai 1 fitur per 'family' informasi — duplikat seperti tenure vs start_date tidak diulang)")
    for start in range(0, len(top_feats), 6):
        chunk = top_feats[start:start + 6]
        fig, axes = grid_axes(len(chunk), ncols=3, w=5, h=3.6)
        for a, f in zip(axes, chunk):
            iv = DRIVER_TABLE.set_index("feature").loc[f, "iv"]
            plot_rate(a, LEVEL_TABLES[f], f"{f} (IV={iv:.2f})", CHURN_RATE)
        fig.suptitle(f"Churn Rate by Feature — top drivers #{start + 1}–{start + len(chunk)} (ranked by IV)", y=1.02)
        plt.tight_layout()
        save_fig(f"driver_churn_rate_{start // 6 + 1}")

    # distribusi numeric: churn vs retained
    num_top = [f for f in top_feats if f in NUM_COLS and train[f].nunique() > 10][:12]
    if num_top:
        fig, axes = grid_axes(len(num_top), ncols=4, w=4, h=3)
        for a, f in zip(axes, num_top):
            v = train[f]
            lo, hi = v.quantile([0.01, 0.99])
            for cls, col, lab in [(0, CFG.PALETTE[0], "Retained"), (1, CFG.PALETTE[1], "Churned")]:
                a.hist(v[y == cls].clip(lo, hi).dropna(), bins=30, alpha=0.55, density=True, color=col, label=lab)
            a.set_title(f[:35], fontsize=9)
            a.legend(fontsize=7)
        fig.suptitle("Numeric Distributions: Churned vs Retained", y=1.01)
        plt.tight_layout()
        save_fig("numeric_by_churn")

    # IV bar chart
    top_iv = DRIVER_TABLE.head(CFG.TOP_N_FEATURES_PLOT)
    fig, ax = plt.subplots(figsize=(8, max(3, 0.32 * len(top_iv))))
    colors = [CFG.PALETTE[1] if v > 0.5 else CFG.PALETTE[0] if v >= 0.1 else "#AAAAAA" for v in top_iv["iv"]]
    ax.barh(top_iv["feature"][::-1], top_iv["iv"][::-1], color=colors[::-1])
    for xline, lab in [(0.02, "weak"), (0.1, "medium"), (0.3, "strong"), (0.5, "suspicious")]:
        ax.axvline(xline, ls=":", color="gray", lw=0.8)
        ax.text(xline, len(top_iv) - 0.5, lab, fontsize=7, color="gray", rotation=90, va="top")
    ax.set_xlabel("Information Value (IV)")
    ax.set_title("Predictive Strength of Each Feature (Information Value)")
    save_fig("information_value_ranking")

### 7.2 Insight otomatis dari driver analysis
Kalimat di bawah (English) siap dipakai/diparafrase di laporan. Semua juga disimpan ke `insights.md`.

In [ ]:
with safe_section("Driver insights"):
    strong = DRIVER_TABLE[(DRIVER_TABLE["significant"]) & (DRIVER_TABLE["iv"] >= 0.02) & (DRIVER_TABLE["iv"] <= 0.5)
                          & (DRIVER_TABLE["univariate_auc"] < CFG.LEAKAGE_AUC_THRESHOLD)]
    keep_f = dedupe_features(strong["feature"].tolist())[:10]
    strong = strong[strong["feature"].isin(keep_f)]
    print("(Fitur duplikat informasi — satu family / |ρ|>0.95 — tidak diulang di insight)")
    for _, r in strong.iterrows():
        if r["type"] == "numeric":
            txt = (f"{r['feature']}: churned customers have a median of {fmt_num(r.get('median_churn'))} vs "
                   f"{fmt_num(r.get('median_retained'))} for retained customers ({r['direction']}; rank-biserial r = {r['effect_size']:.2f}, "
                   f"{r['effect_label']} effect, IV = {r['iv']:.2f}). Highest-risk bin {r['highest_risk_level']} churns at "
                   f"{pct(r['highest_risk_rate'])} ({r['highest_risk_lift']:.1f}x the average).")
        else:
            txt = (f"{r['feature']}: customers in '{r['highest_risk_level']}' churn at {pct(r['highest_risk_rate'])} "
                   f"({r['highest_risk_lift']:.1f}x the overall rate of {pct(CHURN_RATE)}) versus {pct(r['lowest_risk_rate'])} for "
                   f"'{r['lowest_risk_level']}' (χ² p {p_fmt(r['p_value'])}, Cramér's V = {r['effect_size']:.2f}, IV = {r['iv']:.2f}).")
        add_insight("Drivers", txt)
    weak = DRIVER_TABLE[~DRIVER_TABLE["significant"]]["feature"].tolist()
    if weak:
        add_insight("Drivers", f"No statistically significant association with churn after FDR correction for: {', '.join(weak[:12])}.")

### 7.3 Churn reasons (kolom post-event, HANYA deskriptif)
Kolom yang ter-flag leakage (misal `cancellation_reason`, `alasan_berhenti`) **tidak boleh** jadi fitur model, tapi **sangat berharga secara deskriptif**: "kenapa customer bilang mereka pergi?".
Section ini menampilkan distribusi alasan di antara churner dan menyilangkannya dengan driver actionable teratas.

**Untuk laporan:** "Among churned customers, the most frequently stated reasons were Price (34%) and Service (22%), consistent with the drivers identified by the model." → validasi silang yang kuat antara *stated* vs *revealed* drivers.
**Jika tidak ada kolom alasan** → section di-skip (normal).

In [ ]:
REASON_TABLE = pd.DataFrame()
with safe_section("Churn reasons (descriptive)"):
    reason_cols = [c for c in LEAKY_COLS if c in raw_train.columns and raw_train[c].nunique() <= 25
                   and not pd.api.types.is_numeric_dtype(raw_train[c])]
    if not reason_cols:
        print("⏭️ Tidak ada kolom alasan churn (post-event kategori) → skip.")
    for rc in reason_cols[:2]:
        src = raw_train.loc[raw_train[TARGET].map(TARGET_MAPPING) == 1, rc] if TARGET in raw_train.columns else raw_train[rc]
        vc = src.astype(str).replace({"nan": "Not stated", "None": "Not stated"}).value_counts()
        REASON_TABLE = pd.DataFrame({"reason": vc.index, "n_churners": vc.values, "share_%": 100 * vc.values / vc.sum()})
        save_table(REASON_TABLE, f"churn_reasons_{sanitize_name(rc)}")
        display(REASON_TABLE)
        fig, ax = plt.subplots(figsize=(7, max(2.5, 0.4 * len(vc))))
        ax.barh(vc.index[::-1].astype(str), 100 * vc.values[::-1] / vc.sum(), color=CFG.PALETTE[1])
        ax.set_xlabel("% of churned customers")
        ax.set_title(f"Stated churn reasons ({rc})")
        save_fig(f"churn_reasons_{rc}")
        top3 = ", ".join(f"{r} ({s:.0f}%)" for r, s in zip(REASON_TABLE["reason"][:3], REASON_TABLE["share_%"][:3]))
        add_insight("Drivers", f"Among churned customers, the most frequently stated reasons ({rc}) were {top3}.")
        cat_driver = [f for f in DRIVER_TABLE["feature"] if f in CAT_COLS and f not in LEAKY_COLS][:1]
        if cat_driver:
            f = cat_driver[0]
            mask = (y == 1)
            ct = pd.crosstab(train.loc[mask, f].astype(str), raw_train.loc[mask, rc].astype(str), normalize="index") * 100
            fig, ax = plt.subplots(figsize=(min(12, 1.2 * ct.shape[1] + 3), 0.5 * ct.shape[0] + 2))
            sns.heatmap(ct, annot=True, fmt=".0f", cmap="Reds", ax=ax, cbar_kws={"label": "% of churners in row"})
            ax.set_title(f"Stated churn reason by {f} (churners only, row %)")
            save_fig(f"churn_reasons_by_{f}")

# 8. Korelasi & Multikolinearitas

- **Spearman correlation** (numeric–numeric, tahan outlier & hubungan monoton non-linear).
- **Cramér's V matrix** (kategori–kategori).
- **VIF** (Variance Inflation Factor) untuk numeric: VIF > 10 → multikolinearitas serius (penting untuk **interpretasi Logistic Regression / Cox**, tidak masalah untuk tree model).

**Jika X → Y:**
- Pasangan |ρ| > 0.85 (misal `tenure` & `total_charges`) → untuk model interpretable (Section 11) cukup pakai salah satu; untuk GBDT biarkan.
- Jangan simpulkan "X menyebabkan churn" hanya dari korelasi — tulis "associated with".

**Untuk laporan:** biasanya cukup 1 kalimat ("tenure and total premium paid are highly correlated (ρ = 0.83), so only tenure is used in the interpretable model"). Heatmap bisa masuk lampiran.

In [ ]:
HIGH_CORR_PAIRS = pd.DataFrame()

with safe_section("Correlation & VIF"):
    num_for_corr = [c for c in NUM_COLS if train[c].nunique() > 1]
    if len(num_for_corr) > 30:
        num_for_corr = [f for f in DRIVER_TABLE["feature"] if f in num_for_corr][:30]
    if len(num_for_corr) >= 2:
        corr = train[num_for_corr].corr(method="spearman")
        fig, ax = plt.subplots(figsize=(min(16, 0.5 * len(num_for_corr) + 4), min(14, 0.45 * len(num_for_corr) + 3)))
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1, annot=len(num_for_corr) <= 15, fmt=".2f",
                    annot_kws={"size": 7}, ax=ax, cbar_kws={"shrink": 0.7})
        ax.set_title("Spearman Correlation (numeric features)")
        save_fig("spearman_correlation")
        pairs = []
        for i, a in enumerate(num_for_corr):
            for b in num_for_corr[i + 1:]:
                if abs(corr.loc[a, b]) >= 0.8:
                    pairs.append({"feature_a": a, "feature_b": b, "spearman": corr.loc[a, b]})
        HIGH_CORR_PAIRS = pd.DataFrame(pairs)
        if len(HIGH_CORR_PAIRS):
            print("Pasangan sangat berkorelasi (|ρ| ≥ 0.8):")
            display(HIGH_CORR_PAIRS.sort_values("spearman", key=abs, ascending=False))

    cat_for_v = [f for f in DRIVER_TABLE["feature"] if f in CAT_COLS][:20]
    if len(cat_for_v) >= 2:
        V = pd.DataFrame(np.eye(len(cat_for_v)), index=cat_for_v, columns=cat_for_v)
        for i, a in enumerate(cat_for_v):
            for b in cat_for_v[i + 1:]:
                ct = pd.crosstab(train[a].astype(str), train[b].astype(str))
                V.loc[a, b] = V.loc[b, a] = cramers_v_from_ct(ct)[0]
        fig, ax = plt.subplots(figsize=(min(14, 0.55 * len(cat_for_v) + 4), min(12, 0.5 * len(cat_for_v) + 3)))
        sns.heatmap(V, cmap="Blues", vmin=0, vmax=1, annot=len(cat_for_v) <= 12, fmt=".2f", annot_kws={"size": 7}, ax=ax)
        ax.set_title("Cramér's V (categorical associations)")
        save_fig("cramers_v_matrix")

    if HAS_SM_SURV and len(num_for_corr) >= 2:
        Xv = train[num_for_corr].copy()
        Xv = Xv.fillna(Xv.median())
        Xv = (Xv - Xv.mean()) / Xv.std().replace(0, 1)
        Xv = sm.add_constant(Xv, has_constant="add")
        vif = pd.DataFrame({"feature": num_for_corr,
                            "VIF": [variance_inflation_factor(Xv.values, i + 1) for i in range(len(num_for_corr))]})
        vif = vif.sort_values("VIF", ascending=False)
        save_table(vif, "vif")
        display(vif.head(15))
        if (vif["VIF"] > 10).any():
            print("⚠️ VIF > 10:", vif[vif["VIF"] > 10]["feature"].tolist(), "→ hati-hati interpretasi koefisien LogReg/Cox")

# 9. Segment & Interaction Analysis

Churn jarang disebabkan 1 faktor. Section ini mencari **kombinasi** yang berisiko:
1. **Heatmap 2-arah** churn rate untuk pasangan driver teratas (misal `payment_method × tenure_band`).
2. **Decision-tree segment rules** — pohon dangkal (depth 3–4) → aturan IF-THEN yang mudah dijelaskan ke manajemen:
   *"IF tenure < 12 months AND payment = Cash THEN churn rate 58% (n = 412, lift 2.8x)"*.
3. **Top risky segments** — tabel segmen dengan churn rate & lift tertinggi (min. ukuran segmen dijaga supaya tidak noise).

**Jika X → Y:**
- Interaksi terlihat (pola satu fitur berubah tergantung fitur lain) → sebut di laporan sebagai insight bernilai tinggi + jadikan target kampanye spesifik.
- Rule dengan n kecil (< 2% populasi) → jangan diprioritaskan.

**Untuk laporan:** 1 heatmap paling "bercerita" + tabel 3–5 segment rules = bahan rekomendasi yang sangat konkret.

In [ ]:
SEGMENT_RULES = pd.DataFrame()

with safe_section("Two-way interaction heatmaps"):
    cand = [f for f in DRIVER_TABLE["feature"] if f in LEVEL_TABLES and DRIVER_TABLE.set_index("feature").loc[f, "iv"] < 0.5
            and DRIVER_TABLE.set_index("feature").loc[f, "univariate_auc"] < CFG.LEAKAGE_AUC_THRESHOLD]
    cand = dedupe_features(cand)[:4]
    pairs = list(itertools.combinations(cand, 2))[:4]
    if pairs:
        fig, axes = grid_axes(len(pairs), ncols=2, w=7, h=5)
        for a, (f1, f2) in zip(axes, pairs):
            l1, o1 = bin_feature(train[f1], pd.api.types.is_numeric_dtype(train[f1]), n_bins=5, max_levels=8)
            l2, o2 = bin_feature(train[f2], pd.api.types.is_numeric_dtype(train[f2]), n_bins=5, max_levels=8)
            d = pd.DataFrame({"a": l1, "b": l2, "y": y})
            piv = d.pivot_table(index="a", columns="b", values="y", aggfunc="mean").reindex(index=[o for o in o1 if o in set(l1)],
                                                                                         columns=[o for o in o2 if o in set(l2)])
            cnt = d.pivot_table(index="a", columns="b", values="y", aggfunc="count").reindex_like(piv)
            annot = piv.copy().astype(object)
            for i in piv.index:
                for j in piv.columns:
                    annot.loc[i, j] = "" if pd.isna(piv.loc[i, j]) else f"{piv.loc[i, j]:.0%}\n(n={int(cnt.loc[i, j])})"
            sns.heatmap(piv.astype(float), annot=annot.values, fmt="", cmap="Reds", ax=a, cbar_kws={"format": matplotlib.ticker.PercentFormatter(1.0)},
                        annot_kws={"size": 7})
            a.set_title(f"Churn rate: {f1[:25]} × {f2[:25]}", fontsize=10)
            a.set_ylabel(f1[:30])
            a.set_xlabel(f2[:30])
        plt.tight_layout()
        save_fig("interaction_heatmaps")

In [ ]:
def encode_for_tree(df, feats):
    """One-hot (kategori, max 15 level) + numeric apa adanya (NaN → median) untuk tree rules yang mudah dibaca."""
    parts = []
    for f in feats:
        s = df[f]
        if pd.api.types.is_numeric_dtype(s):
            parts.append(s.fillna(s.median()).rename(f))
        else:
            lab = s.astype(str).where(s.notna(), "Missing")
            top = lab.value_counts().index[:15]
            lab = lab.where(lab.isin(top), "Other")
            parts.append(pd.get_dummies(lab, prefix=f, prefix_sep="=").astype(int))
    return pd.concat(parts, axis=1)


def tree_leaf_rules(tree, feature_names, X, y_arr):
    t = tree.tree_
    leaf_id = tree.apply(X)
    rules = {}

    def recurse(node, conds):
        if t.children_left[node] == -1:
            rules[node] = conds
            return
        f = feature_names[t.feature[node]]
        thr = t.threshold[node]
        if "=" in f:   # one-hot
            recurse(t.children_left[node], conds + [f"{f.split('=')[0]} ≠ {f.split('=', 1)[1]}"])
            recurse(t.children_right[node], conds + [f"{f.split('=')[0]} = {f.split('=', 1)[1]}"])
        elif f in BINARY_TREE_COLS:
            recurse(t.children_left[node], conds + [f"{f} = 0"])
            recurse(t.children_right[node], conds + [f"{f} = 1"])
        else:
            recurse(t.children_left[node], conds + [f"{f} ≤ {thr:,.2f}"])
            recurse(t.children_right[node], conds + [f"{f} > {thr:,.2f}"])

    recurse(0, [])
    out = []
    for leaf, conds in rules.items():
        m = leaf_id == leaf
        if m.sum() == 0:
            continue
        rate = y_arr[m].mean()
        out.append({"segment_rule": " AND ".join(conds), "n_customers": int(m.sum()), "share_%": 100 * m.mean(),
                    "churn_rate": rate, "lift": rate / y_arr.mean(), "churners_captured_%": 100 * y_arr[m].sum() / y_arr.sum()})
    return pd.DataFrame(out).sort_values("churn_rate", ascending=False).reset_index(drop=True)


with safe_section("Decision-tree segment rules"):
    seg_feats = [f for f in DRIVER_TABLE["feature"] if (DRIVER_TABLE.set_index("feature").loc[f, "univariate_auc"] < CFG.LEAKAGE_AUC_THRESHOLD)
                 and DRIVER_TABLE.set_index("feature").loc[f, "iv"] < 0.5]
    seg_feats = dedupe_features(seg_feats)[:15]
    Xt = encode_for_tree(train, seg_feats)
    BINARY_TREE_COLS = {c for c in Xt.columns if "=" not in c and set(pd.unique(Xt[c])) <= {0, 1}}
    min_leaf = max(30, int(0.02 * len(Xt)))
    dt = DecisionTreeClassifier(max_depth=3 if len(Xt) < 3000 else 4, min_samples_leaf=min_leaf, random_state=CFG.SEED)
    dt.fit(Xt.values, y)
    SEGMENT_RULES = tree_leaf_rules(dt, list(Xt.columns), Xt.values, y)
    save_table(SEGMENT_RULES, "segment_rules")
    display(SEGMENT_RULES)

    fig, ax = plt.subplots(figsize=(22, 9))
    plot_tree(dt, feature_names=[c[:28] for c in Xt.columns], class_names=["Retained", "Churned"], filled=True, rounded=True,
              proportion=True, impurity=False, fontsize=7, ax=ax)
    ax.set_title("Churn Segmentation Tree (each box: share of customers & class proportion)")
    save_fig("segment_tree")

    for _, r in SEGMENT_RULES.head(3).iterrows():
        if r["lift"] > 1.2:
            add_insight("Segments", f"High-risk segment: IF {r['segment_rule']} THEN churn rate = {pct(r['churn_rate'])} "
                                    f"({r['lift']:.1f}x average; {r['n_customers']:,} customers, {r['share_%']:.1f}% of base, "
                                    f"capturing {r['churners_captured_%']:.1f}% of all churners).")
    low = SEGMENT_RULES.iloc[-1]
    add_insight("Segments", f"Most loyal segment: IF {low['segment_rule']} THEN churn rate = {pct(low['churn_rate'])} "
                            f"({low['n_customers']:,} customers).")

# 10. Survival Analysis — *kapan* customer churn?

Churn analysis biasa menjawab **siapa** yang churn. Survival analysis menjawab **kapan** risiko churn paling tinggi — sangat relevan untuk asuransi (lapse banyak terjadi di tahun-tahun awal polis).

**Setup:** `duration` = tenure, `event` = churn (1) / masih aktif (0 = *censored*, belum churn sampai saat data diambil).
Asumsi: data snapshot — tenure churner = lama sampai churn; tenure customer aktif = lama berlangganan sampai sekarang.

| Analisis | Output | Cara baca |
|---|---|---|
| **Kaplan-Meier (KM)** | kurva S(t) = peluang masih bertahan setelah t | kurva turun tajam di awal → churn dini dominan |
| **Median survival** | waktu saat S(t) = 50% | "setengah customer churn sebelum bulan ke-X" (bisa *not reached*) |
| **Retention at t** | S(12), S(24), S(36) | "Retention rate tahun pertama = 78%" |
| **KM per grup + log-rank test** | apakah kurva antar grup beda signifikan | p < 0.05 → pola bertahan beda nyata |
| **Hazard rate per interval tenure** | jumlah churn ÷ person-time (misal per customer-bulan) di interval tsb | identifikasi "masa kritis"; adil walau lebar interval beda |
| **Cox Proportional Hazards** | **Hazard Ratio (HR)** per fitur | HR = 1.5 → risiko churn 50% lebih tinggi kapan pun; HR < 1 = protektif |

**Jika X → Y:**
- Tidak ada kolom tenure → section di-skip (set `CFG.COLUMN_ROLES = {"tenure": "nama_kolom"}` kalau sebenarnya ada; atau buat tenure dari `start_date` di `custom_feature_engineering`).
- Hazard paling tinggi di 0–12 bulan → rekomendasi **onboarding program / early-life engagement**.
- Hazard naik di bulan tertentu (misal bulan 12, 24) → **momen renewal** → rekomendasi renewal reminder / loyalty offer sebelum jatuh tempo.
- Kurva KM grup bersilangan → asumsi proportional hazards Cox mungkin dilanggar → interpretasi HR hati-hati (sebut sebagai limitation).

Fitur yang diturunkan dari tenure (band tenure, tanggal mulai, total paid) **dikeluarkan dari Cox** karena tenure adalah sumbu waktunya.

**Untuk laporan:** 1 figure KM (overall + per driver utama) + kalimat retention rate + tabel hazard ratio top 8 → bagian "When do customers churn?"

In [ ]:
SURV_AVAILABLE = False
COX_TABLE = pd.DataFrame()
KM_SUMMARY = pd.DataFrame()


def tenure_related_cols():
    rel = set()
    if "tenure" in ROLES:
        rel.add(ROLES["tenure"])
    rel |= {c for c in train.columns if c.startswith("fe_tenure") or c in ("fe_is_new_customer", "fe_avg_paid_per_period",
                                                                            "fe_paid_vs_expected", "fe_claims_per_year")}
    rel |= {c for c in train.columns if c.endswith("_days_since") or c.endswith("_year") or c.endswith("_month")}
    if "total_paid" in ROLES:
        rel.add(ROLES["total_paid"])
    return rel


def km_curve(time_arr, event_arr):
    sf = SurvfuncRight(time_arr, event_arr)
    t = np.asarray(sf.surv_times)
    s = np.asarray(sf.surv_prob)
    se = np.asarray(sf.surv_prob_se)
    return sf, t, s, np.clip(s - 1.96 * se, 0, 1), np.clip(s + 1.96 * se, 0, 1)


def surv_at(t, s, when):
    if len(t) == 0:
        return np.nan
    idx = np.searchsorted(t, when, side="right") - 1
    return 1.0 if idx < 0 else float(s[idx])


with safe_section("Survival: Kaplan-Meier"):
    if "tenure" not in ROLES:
        print("⏭️ Tidak ada kolom tenure terdeteksi → survival analysis di-skip.")
    elif not HAS_SM_SURV:
        print("⏭️ statsmodels tidak tersedia → survival analysis di-skip. (pip install statsmodels)")
    else:
        tcol = ROLES["tenure"]
        dur_all = train[tcol].astype(float)
        m_ok = dur_all.notna() & (dur_all >= 0)
        DUR, EVT = dur_all[m_ok].values, y[m_ok.values]
        if (DUR == 0).any():
            DUR = DUR + 0.5  # hindari durasi 0
        SURV_AVAILABLE = True
        unit = {"months": "months", "years": "years", "days": "days"}.get(TENURE_UNIT, "periods")
        SURV_UNIT = unit
        sf, t, s, lo, hi = km_curve(DUR, EVT)
        checkpoints = {"months": [6, 12, 24, 36, 60], "years": [1, 2, 3, 5, 10], "days": [90, 180, 365, 730, 1095]}.get(TENURE_UNIT, [6, 12, 24, 36])
        med = np.nan
        try:
            med = sf.quantile(0.5)
        except Exception:
            pass
        rows = [{"checkpoint": f"{cp} {unit}", "survival_prob": surv_at(t, s, cp)} for cp in checkpoints if cp <= DUR.max()]
        KM_SUMMARY = pd.DataFrame(rows)
        display(KM_SUMMARY)
        print(f"Median survival time: {med if pd.notna(med) else 'not reached (lebih dari 50% customer belum churn)'}")

        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.step(t, s, where="post", color=CFG.PALETTE[0], lw=2, label="All customers")
        ax.fill_between(t, lo, hi, step="post", alpha=0.2, color=CFG.PALETTE[0])
        for cp in checkpoints:
            if cp <= DUR.max():
                ax.axvline(cp, ls=":", color="gray", lw=0.7)
                ax.text(cp, surv_at(t, s, cp) + 0.02, f"{surv_at(t, s, cp):.0%}", fontsize=8, ha="center")
        ax.set_ylim(0, 1.05)
        ax.set_xlabel(f"Tenure ({unit})")
        ax.set_ylabel("Probability of remaining a customer")
        ax.set_title("Kaplan-Meier Customer Survival Curve")
        ax.legend()
        save_fig("km_overall")
        for r in rows[:3]:
            add_insight("Survival", f"Estimated probability of retaining a customer beyond {r['checkpoint']}: {pct(r['survival_prob'])} (Kaplan-Meier).", show=False)
        if rows:
            print("💡", INSIGHTS[-1][1])

        # discrete hazard per interval
        edges = {"months": [0, 3, 6, 12, 18, 24, 36, 48, 60, 120, np.inf], "years": [0, 1, 2, 3, 5, 10, np.inf],
                 "days": [0, 90, 180, 365, 545, 730, 1095, 1825, np.inf]}.get(TENURE_UNIT, [0, 6, 12, 24, 48, np.inf])
        edges = [e for e in edges if e < DUR.max()] + [np.inf]
        hz = []
        for a, b in zip(edges[:-1], edges[1:]):
            at_risk = (DUR >= a).sum()
            ev = ((DUR >= a) & (DUR < b) & (EVT == 1)).sum()
            exposure = np.clip(np.minimum(DUR, b) - a, 0, None).sum()   # person-time di interval
            if at_risk > 0 and exposure > 0:
                hz.append({"interval": f"[{a:g}, {b:g})", "at_risk_start": int(at_risk), "churn_events": int(ev),
                           "person_time": exposure, "hazard": ev / exposure})
        HAZARD_TABLE = pd.DataFrame(hz)
        HAZARD_TABLE = HAZARD_TABLE[HAZARD_TABLE["at_risk_start"] >= 30]   # interval terlalu sedikit data → tidak stabil
        save_table(HAZARD_TABLE, "hazard_by_tenure_interval")
        display(HAZARD_TABLE)
        fig, ax = plt.subplots(figsize=(8, 3.8))
        ax.bar(HAZARD_TABLE["interval"], HAZARD_TABLE["hazard"], color=CFG.PALETTE[1])
        ax.set_ylabel(f"Churn hazard per {unit[:-1] if unit.endswith('s') else unit}")
        ax.set_xlabel(f"Tenure interval ({unit})")
        ax.set_title("When Do Customers Churn? Hazard by Tenure Interval")
        ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
        plt.xticks(rotation=30)
        save_fig("hazard_by_tenure")
        peak = HAZARD_TABLE.sort_values("hazard", ascending=False).iloc[0]
        add_insight("Survival", f"Churn hazard is highest in tenure interval {peak['interval']} {unit}: about {pct(peak['hazard'], 2)} of active customers churn per {unit[:-1] if unit.endswith('s') else unit} in this period "
                                f"(vs {pct(HAZARD_TABLE['churn_events'].sum() / HAZARD_TABLE['person_time'].sum(), 2)} on average).")

In [ ]:
with safe_section("Survival: KM per group + log-rank"):
    if SURV_AVAILABLE:
        rel = tenure_related_cols()
        grp_feats = [f for f in DRIVER_TABLE["feature"] if f in CAT_COLS + [c for c in NUM_COLS if train[c].nunique() <= 6]
                     and f not in rel and f not in LEAKY_COLS and train[f].nunique() <= 8]
        grp_feats = dedupe_features(grp_feats)[:6]
        logrank_rows = []
        if grp_feats:
            fig, axes = grid_axes(len(grp_feats), ncols=3, w=5.2, h=3.8)
            for a, f in zip(axes, grp_feats):
                g = train.loc[m_ok, f].astype(str).where(train.loc[m_ok, f].notna(), "Missing").values
                levels = pd.Series(g).value_counts()
                levels = levels[levels >= 30].index[:6]
                for i, lv in enumerate(levels):
                    mm = g == lv
                    _, tt, ss, _, _ = km_curve(DUR[mm], EVT[mm])
                    a.step(tt, ss, where="post", lw=1.6, label=f"{str(lv)[:18]} (n={mm.sum()})", color=CFG.PALETTE[i % len(CFG.PALETTE)])
                try:
                    keep = np.isin(g, levels)
                    chi2, p = survdiff(DUR[keep], EVT[keep], g[keep])
                except Exception:
                    chi2, p = np.nan, np.nan
                logrank_rows.append({"feature": f, "logrank_chi2": chi2, "p_value": p})
                a.set_title(f"{f[:30]} (log-rank p {p_fmt(p)})", fontsize=9)
                a.set_ylim(0, 1.05)
                a.set_xlabel(f"Tenure ({unit})", fontsize=8)
                a.legend(fontsize=6)
            fig.suptitle("Kaplan-Meier Survival by Customer Group", y=1.02)
            plt.tight_layout()
            save_fig("km_by_group")
            LOGRANK = pd.DataFrame(logrank_rows)
            save_table(LOGRANK, "logrank_tests")
            display(LOGRANK)

In [ ]:
def forest_plot(df, est, lo, hi, label_col, title, xlabel, ref=1.0, log=True, max_rows=20):
    d = df.head(max_rows).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(d) + 1)))
    colors = [CFG.PALETTE[1] if v > ref else CFG.PALETTE[0] for v in d[est]]
    ax.errorbar(d[est], range(len(d)), xerr=[d[est] - d[lo], d[hi] - d[est]], fmt="none", ecolor="#777", capsize=2)
    ax.scatter(d[est], range(len(d)), c=colors, zorder=3)
    ax.axvline(ref, color="black", ls="--", lw=0.8)
    ax.set_yticks(range(len(d)))
    ax.set_yticklabels([str(s)[:45] for s in d[label_col]], fontsize=8)
    if log:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    return fig


def build_design_matrix(df, feats, max_levels=8):
    """Numeric → standardized (z), binary 0/1 tetap; kategori → dummies (reference = level paling sering)."""
    parts, meta = [], {}
    for f in feats:
        s = df[f]
        if pd.api.types.is_numeric_dtype(s):
            v = s.fillna(s.median())
            if v.nunique() <= 2:
                parts.append(v.rename(f))
                meta[f] = "binary (0→1)"
            else:
                sd = v.std() if v.std() > 0 else 1
                parts.append(((v - v.mean()) / sd).rename(f))
                meta[f] = f"per 1 SD (= {sd:,.3g})"
        else:
            lab = s.astype(str).where(s.notna(), "Missing")
            top = lab.value_counts().index[:max_levels]
            lab = lab.where(lab.isin(top), "Other")
            ref = lab.value_counts().index[0]
            d = pd.get_dummies(lab, prefix=f, prefix_sep=" = ").astype(float)
            d = d.drop(columns=[f"{f} = {ref}"])
            d = d.loc[:, d.sum() >= 10]
            for c in d.columns:
                meta[c] = f"vs '{ref}'"
            parts.append(d)
    X = pd.concat(parts, axis=1)
    X = X.loc[:, X.std() > 0]
    return X, meta


def select_interpretable_features(max_feats=15, exclude=()):
    """Ambil driver teratas (non-leak), hindari duplikasi info (|ρ|>0.8) & prefer band untuk age."""
    chosen, fams = [], set()
    tbl = DRIVER_TABLE[(DRIVER_TABLE["univariate_auc"] < CFG.LEAKAGE_AUC_THRESHOLD) & (DRIVER_TABLE["iv"] >= 0.02)]
    for f in tbl["feature"]:
        if f in exclude or f in LEAKY_COLS or f in HIGH_CARD_COLS:
            continue
        if f == ROLES.get("age") and "fe_age_band" in tbl["feature"].values and "fe_age_band" not in exclude:
            continue  # umur sering U-shape → pakai band
        fam = FEATURE_FAMILY.get(f)
        if fam is not None and fam in fams and not (fam == "age" and f == "fe_age_band"):
            continue
        if pd.api.types.is_numeric_dtype(train[f]):
            redundant = False
            for g in chosen:
                if pd.api.types.is_numeric_dtype(train[g]):
                    try:
                        if abs(train[[f, g]].corr(method="spearman").iloc[0, 1]) > 0.8:
                            redundant = True
                            break
                    except Exception:
                        pass
            if redundant:
                continue
        chosen.append(f)
        if fam is not None:
            fams.add(fam)
        if len(chosen) >= max_feats:
            break
    return chosen


with safe_section("Survival: Cox proportional hazards"):
    if SURV_AVAILABLE:
        cox_feats = select_interpretable_features(12, exclude=tenure_related_cols())
        Xc, meta_c = build_design_matrix(train.loc[m_ok].reset_index(drop=True), cox_feats)
        res = None
        for attempt in range(3):
            try:
                mod = PHReg(DUR, Xc.values, status=EVT, ties="efron")
                res = mod.fit(disp=False)
                if not np.all(np.isfinite(res.bse)):
                    raise ValueError("SE tidak finite")
                break
            except Exception as e:
                print(f"  Cox attempt {attempt + 1} gagal ({e}) → kurangi fitur")
                keep = Xc.columns[: max(3, int(len(Xc.columns) * 0.6))]
                Xc = Xc[keep]
        if res is not None:
            ci = np.asarray(res.conf_int())
            COX_TABLE = pd.DataFrame({"term": Xc.columns, "coef": res.params, "hazard_ratio": np.exp(res.params),
                                      "hr_ci_low": np.exp(ci[:, 0]), "hr_ci_high": np.exp(ci[:, 1]), "p_value": res.pvalues})
            COX_TABLE["scale"] = COX_TABLE["term"].map(meta_c)
            COX_TABLE["interpretation"] = COX_TABLE.apply(
                lambda r: f"{'+' if r.hazard_ratio > 1 else '-'}{abs(r.hazard_ratio - 1):.0%} churn hazard ({r.scale})", axis=1)
            COX_TABLE = COX_TABLE.reindex(COX_TABLE["coef"].abs().sort_values(ascending=False).index).reset_index(drop=True)
            save_table(COX_TABLE, "cox_hazard_ratios")
            display(COX_TABLE)
            fig = forest_plot(COX_TABLE[COX_TABLE["p_value"] < 0.1] if (COX_TABLE["p_value"] < 0.1).sum() >= 3 else COX_TABLE,
                              "hazard_ratio", "hr_ci_low", "hr_ci_high", "term",
                              "Cox Proportional Hazards: Hazard Ratios (95% CI)", "Hazard ratio (log scale; >1 = higher churn risk)")
            save_fig("cox_hazard_ratios", fig)
            for _, r in COX_TABLE[COX_TABLE["p_value"] < 0.05].head(5).iterrows():
                add_insight("Survival", f"Cox model: {r['term']} has a hazard ratio of {r['hazard_ratio']:.2f} "
                                        f"(95% CI {r['hr_ci_low']:.2f}–{r['hr_ci_high']:.2f}, p {p_fmt(r['p_value'])}), i.e. "
                                        f"{r['interpretation']}, holding other factors constant.")
            try:
                print(f"Concordance (C-index) proxy: hitung via lifelines bila tersedia.")
                if HAS_LIFELINES:
                    from lifelines import CoxPHFitter
                    dfl = Xc.copy()
                    dfl.columns = [sanitize_name(c) for c in dfl.columns]
                    dfl["duration"], dfl["event"] = DUR, EVT
                    cph = CoxPHFitter(penalizer=0.01).fit(dfl, "duration", "event")
                    print(f"lifelines C-index: {cph.concordance_index_:.3f} (0.5 = acak, >0.7 bagus)")
                    print("Uji asumsi proportional hazards (Schoenfeld) — p < 0.05 berarti asumsi dilanggar untuk fitur tsb:")
                    from lifelines.statistics import proportional_hazard_test
                    pht = proportional_hazard_test(cph, dfl, time_transform="rank")
                    display(pht.summary.sort_values("p").head(10))
            except Exception as e:
                print("  (lifelines check di-skip:", e, ")")

# 11. Interpretable Model — Logistic Regression (Odds Ratio)

GBDT biasanya paling akurat, tapi **Logistic Regression** paling mudah dijelaskan ke manajemen dan juri:
- **Odds Ratio (OR)** > 1 → meningkatkan peluang churn; < 1 → menurunkan (protektif).
- Numeric distandarisasi → OR dibaca **per kenaikan 1 standar deviasi**. Fitur biner → OR punya vs tidak punya. Kategori → dibanding **reference level** (level paling umum).
- **Average Marginal Effect (AME)** → perubahan *probabilitas* churn (dalam poin persen) — lebih intuitif daripada OR.
- Ini analisis **multivariat**: efek tiap fitur sudah *dikontrol* oleh fitur lain (beda dengan driver analysis univariat di Section 7).

**Jika X → Y:**
- Error `Singular matrix` / `PerfectSeparation` → fitur terlalu berkorelasi atau ada level yang memprediksi sempurna; notebook otomatis fallback ke versi regularized (tanpa CI) atau mengurangi fitur.
- Arah koefisien berlawanan dengan driver analysis → kemungkinan **confounding** (misal fitur A tampak berisiko hanya karena berkorelasi dengan B). Ini insight menarik untuk dibahas!
- OR dengan CI melewati 1 → tidak signifikan, jangan dijadikan klaim.

**Untuk laporan:** forest plot OR + 2–3 kalimat: *"Holding other factors constant, customers paying by auto-debit have 52% lower odds of churning (OR = 0.48, 95% CI 0.41–0.56)."*

In [ ]:
LOGIT_TABLE = pd.DataFrame()

with safe_section("Interpretable logistic regression"):
    if not HAS_SM:
        print("⏭️ statsmodels tidak ada → pakai sklearn LogisticRegression (tanpa CI/p-value).")
    logit_feats = select_interpretable_features(15)
    Xl, meta_l = build_design_matrix(train, logit_feats)
    print(f"Fitur yang dipakai ({len(logit_feats)}):", logit_feats)
    fitted = False
    if HAS_SM:
        for attempt in range(3):
            try:
                lres = sm.Logit(y, sm.add_constant(Xl, has_constant="add")).fit(disp=0, maxiter=300)
                if not np.all(np.isfinite(lres.bse)):
                    raise ValueError("SE tidak finite")
                ci = lres.conf_int()
                LOGIT_TABLE = pd.DataFrame({"term": lres.params.index, "coef": lres.params.values,
                                            "odds_ratio": np.exp(lres.params.values), "or_ci_low": np.exp(ci[0].values),
                                            "or_ci_high": np.exp(ci[1].values), "p_value": lres.pvalues.values})
                LOGIT_TABLE = LOGIT_TABLE[LOGIT_TABLE["term"] != "const"]
                LOGIT_TABLE["scale"] = LOGIT_TABLE["term"].map(meta_l)
                try:
                    ame = lres.get_margeff(at="overall").summary_frame()
                    LOGIT_TABLE["avg_marginal_effect_pp"] = LOGIT_TABLE["term"].map(100 * ame["dy/dx"])
                except Exception:
                    pass
                print(f"McFadden pseudo-R² = {lres.prsquared:.3f} | AIC = {lres.aic:,.1f} | n = {int(lres.nobs)}")
                fitted = True
                break
            except Exception as e:
                print(f"  Logit attempt {attempt + 1} gagal: {e} → kurangi fitur")
                Xl = Xl.iloc[:, : max(3, int(Xl.shape[1] * 0.7))]
    if not fitted:
        lr = LogisticRegression(C=1.0, max_iter=3000).fit(Xl, y)
        LOGIT_TABLE = pd.DataFrame({"term": Xl.columns, "coef": lr.coef_[0], "odds_ratio": np.exp(lr.coef_[0]),
                                    "or_ci_low": np.nan, "or_ci_high": np.nan, "p_value": np.nan})
        LOGIT_TABLE["scale"] = LOGIT_TABLE["term"].map(meta_l)
    LOGIT_TABLE["interpretation"] = LOGIT_TABLE.apply(
        lambda r: f"{'+' if r.odds_ratio > 1 else '-'}{abs(r.odds_ratio - 1):.0%} odds of churn ({r.scale})", axis=1)
    LOGIT_TABLE = LOGIT_TABLE.reindex(LOGIT_TABLE["coef"].abs().sort_values(ascending=False).index).reset_index(drop=True)
    save_table(LOGIT_TABLE, "logit_odds_ratios")
    display(LOGIT_TABLE)
    if LOGIT_TABLE["or_ci_low"].notna().any():
        sig = LOGIT_TABLE[LOGIT_TABLE["p_value"] < 0.05]
        fig = forest_plot(sig if len(sig) >= 3 else LOGIT_TABLE, "odds_ratio", "or_ci_low", "or_ci_high", "term",
                          "Logistic Regression: Odds Ratios for Churn (95% CI)", "Odds ratio (log scale; >1 = higher churn odds)")
        save_fig("logit_odds_ratios", fig)
        for _, r in sig.head(6).iterrows():
            ame_txt = f"; average marginal effect {r['avg_marginal_effect_pp']:+.1f} pp" if "avg_marginal_effect_pp" in r and pd.notna(r.get("avg_marginal_effect_pp")) else ""
            add_insight("Interpretable model", f"Holding other factors constant, {r['term']} ({r['scale']}) changes the odds of churn by "
                                               f"{(r['odds_ratio'] - 1):+.0%} (OR = {r['odds_ratio']:.2f}, 95% CI {r['or_ci_low']:.2f}–{r['or_ci_high']:.2f}, "
                                               f"p {p_fmt(r['p_value'])}{ame_txt}).")
    # performa model interpretable (CV)
    try:
        oof_lr = cross_val_predict(LogisticRegression(C=1.0, max_iter=3000), Xl, y,
                                   cv=StratifiedKFold(5, shuffle=True, random_state=CFG.SEED), method="predict_proba")[:, 1]
        INTERPRETABLE_AUC = roc_auc_score(y, oof_lr)
        print(f"CV ROC-AUC model interpretable (LogReg, {Xl.shape[1]} terms): {INTERPRETABLE_AUC:.4f}")
    except Exception as e:
        INTERPRETABLE_AUC = np.nan
        print("CV AUC LogReg gagal:", e)

# 12. Train–Test Drift & Adversarial Validation

Model hanya bagus di test kalau **distribusi test mirip train**. Cek:
- **PSI (Population Stability Index)** per fitur: < 0.1 stabil, 0.1–0.25 sedikit bergeser, **> 0.25 bergeser signifikan**.
- **KS test** untuk numeric (statistic besar & p kecil → distribusi beda).
- **Unseen categories**: level kategori yang muncul di test tapi tidak di train.
- **Adversarial validation**: latih classifier membedakan baris train vs test. AUC ≈ 0.5 → mirip (bagus). **AUC > 0.7** → ada drift; fitur paling penting di classifier ini = biang drift.

**Jika X → Y:**
- AUC adversarial tinggi karena kolom ID/tanggal/urutan → buang kolom itu (biasanya tidak bermakna bisnis).
- Drift di fitur penting (misal premi naik di periode test) → pertimbangkan: (1) pakai fitur relatif (rasio) bukan absolut, (2) CV yang meniru test (time-based), (3) sebut sebagai limitation.
- Tidak ada test file → section di-skip.

In [ ]:
def psi(expected, actual, bins=10):
    e, a = expected.dropna(), actual.dropna()
    if len(e) == 0 or len(a) == 0:
        return np.nan
    if pd.api.types.is_numeric_dtype(e) and e.nunique() > 10:
        qs = np.unique(np.quantile(e, np.linspace(0, 1, bins + 1)))
        if len(qs) < 3:
            return 0.0
        qs[0], qs[-1] = -np.inf, np.inf
        ec = pd.cut(e, qs).value_counts(normalize=True, sort=False)
        ac = pd.cut(a, qs).value_counts(normalize=True, sort=False)
    else:
        ec = e.astype(str).value_counts(normalize=True)
        ac = a.astype(str).value_counts(normalize=True).reindex(ec.index).fillna(0)
    ec, ac = ec.values + 1e-4, ac.values + 1e-4
    return float(np.sum((ac - ec) * np.log(ac / ec)))


DRIFT_TABLE = pd.DataFrame()
ADV_AUC = np.nan

with safe_section("Drift & adversarial validation"):
    if test is None:
        print("⏭️ Tidak ada test → skip.")
    else:
        rows = []
        for c in NUM_COLS + CAT_COLS + HIGH_CARD_COLS:
            if c not in test.columns:
                continue
            r = {"feature": c, "psi": psi(train[c], test[c])}
            if pd.api.types.is_numeric_dtype(train[c]) and train[c].nunique() > 2:
                try:
                    ks = stats.ks_2samp(train[c].dropna(), test[c].dropna())
                    r.update({"ks_stat": ks.statistic, "ks_p": ks.pvalue})
                except Exception:
                    pass
            else:
                unseen = set(test[c].dropna().astype(str)) - set(train[c].dropna().astype(str))
                r["n_unseen_levels"] = len(unseen)
                r["unseen_rows_%"] = 100 * test[c].astype(str).isin(unseen).mean()
            rows.append(r)
        DRIFT_TABLE = pd.DataFrame(rows).sort_values("psi", ascending=False)
        DRIFT_TABLE["psi_flag"] = pd.cut(DRIFT_TABLE["psi"], [-np.inf, 0.1, 0.25, np.inf], labels=["stable", "minor shift", "MAJOR shift"])
        save_table(DRIFT_TABLE, "train_test_drift")
        display(DRIFT_TABLE.head(15))

        # adversarial validation
        feats_adv = [c for c in NUM_COLS + CAT_COLS + HIGH_CARD_COLS if c in test.columns]
        A = pd.concat([train[feats_adv], test[feats_adv]], ignore_index=True)
        for c in feats_adv:
            if not pd.api.types.is_numeric_dtype(A[c]):
                A[c] = A[c].astype(str).astype("category").cat.codes
        A = A.fillna(-999)
        is_test = np.r_[np.zeros(len(train)), np.ones(len(test))]
        clf = (lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1, random_state=CFG.SEED)
               if HAS_LGB else HistGradientBoostingClassifier(max_iter=300, random_state=CFG.SEED))
        p_adv = cross_val_predict(clf, A, is_test, cv=StratifiedKFold(5, shuffle=True, random_state=CFG.SEED), method="predict_proba")[:, 1]
        ADV_AUC = roc_auc_score(is_test, p_adv)
        print(f"\nAdversarial validation AUC = {ADV_AUC:.4f}")
        if ADV_AUC < 0.6:
            print("✅ Train & test sangat mirip → CV score bisa dipercaya sebagai estimasi test score.")
        elif ADV_AUC < 0.75:
            print("⚠️ Ada sedikit drift. Cek fitur teratas di bawah.")
        else:
            print("🚨 Drift kuat! CV score mungkin terlalu optimis. Cek & pertimbangkan buang fitur penyebab drift yang tidak bermakna bisnis.")
        clf.fit(A, is_test)
        imp = pd.Series(getattr(clf, "feature_importances_", np.zeros(len(feats_adv))), index=feats_adv).sort_values(ascending=False)
        if imp.sum() > 0:
            print("Fitur paling membedakan train vs test:")
            display(imp.head(10).to_frame("adv_importance"))

# 13. Modeling Setup

## 13.1 Metric
| Metric lomba | Jenis | Submission | Catatan |
|---|---|---|---|
| `roc_auc` / `gini` | ranking | probability | paling umum untuk churn; tidak tergantung threshold |
| `pr_auc` | ranking | probability | lebih sensitif untuk kelas minoritas |
| `logloss` / `brier` | probabilitas | probability | butuh probabilitas **terkalibrasi** (Section 18) |
| `f1`, `f2`, `precision`, `recall`, `accuracy`, `balanced_accuracy`, `mcc`, `kappa` | label | label 0/1 | **threshold tuning wajib** (Section 18) |

Untuk metrik label, ranking model di leaderboard memakai skor pada **threshold optimal OOF** (dicetak juga AUC sebagai pembanding yang stabil).

## 13.2 Representasi fitur per keluarga model (otomatis)
| Model | Numeric | Kategori | High-cardinality | Missing |
|---|---|---|---|---|
| LightGBM / XGBoost | apa adanya | pandas `category` (native) | frequency + target encoding (in-fold) | native |
| CatBoost | apa adanya | string (native, ordered TS) | string native | native / "NA" |
| HistGradientBoosting | apa adanya | ordinal code | freq + TE | native |
| RandomForest / ExtraTrees | median impute | ordinal code | freq + TE | imputed |
| Logistic Regression | signed-log untuk skew tinggi + impute + indikator missing + standardize | one-hot (rare digabung) | TE | imputed |

**Target encoding** selalu di-fit **di dalam fold training** (dengan inner K-fold untuk baris training sendiri) → tidak bocor.

## 13.3 Imbalance
`CFG.IMBALANCE = "auto"` → metrik ranking/probabilitas: **tanpa reweighting** (menjaga probabilitas tetap jujur; AUC tidak terbantu oleh reweighting).
Metrik label & churn rate < 20% → `class_weight` (+ threshold tuning). `"smote"` tersedia (hanya di dalam fold, untuk model numeric) tapi **jarang lebih baik** dari class weight + threshold tuning untuk GBDT.

**Jika X → Y:**
- Metric lomba tidak ada di daftar → tambahkan di fungsi `compute_metric`.
- Error terkait nama kolom (LightGBM "special JSON characters") → sudah di-handle `sanitize_columns`.

In [ ]:
PROB_METRICS = {"roc_auc", "gini", "pr_auc", "logloss", "brier"}
LABEL_METRICS = {"f1", "f2", "accuracy", "balanced_accuracy", "precision", "recall", "mcc", "kappa"}
LOWER_IS_BETTER = {"logloss", "brier"}
if CFG.METRIC not in PROB_METRICS | LABEL_METRICS:
    print(f"⚠️ METRIC '{CFG.METRIC}' tidak dikenal → pakai roc_auc. Tambahkan di compute_metric kalau perlu.")
    CFG.METRIC = "roc_auc"
THR_METRIC = (CFG.METRIC if CFG.METRIC in LABEL_METRICS else "f1") if CFG.THRESHOLD_METRIC == "auto" else CFG.THRESHOLD_METRIC
if CFG.SUBMISSION_MODE == "probability" and CFG.METRIC in LABEL_METRICS:
    print("⚠️ METRIC berbasis label tapi SUBMISSION_MODE='probability'. Cek format sample_submission! Biasanya harus 'label'.")


def compute_metric(metric, y_true, p, thr=0.5):
    p = np.clip(np.asarray(p, dtype=float), 1e-7, 1 - 1e-7)
    yl = (p >= thr).astype(int)
    if metric == "roc_auc":
        return roc_auc_score(y_true, p)
    if metric == "gini":
        return 2 * roc_auc_score(y_true, p) - 1
    if metric == "pr_auc":
        return average_precision_score(y_true, p)
    if metric == "logloss":
        return log_loss(y_true, p)
    if metric == "brier":
        return brier_score_loss(y_true, p)
    if metric == "f1":
        return f1_score(y_true, yl, zero_division=0)
    if metric == "f2":
        return fbeta_score(y_true, yl, beta=2, zero_division=0)
    if metric == "accuracy":
        return accuracy_score(y_true, yl)
    if metric == "balanced_accuracy":
        return balanced_accuracy_score(y_true, yl)
    if metric == "precision":
        return precision_score(y_true, yl, zero_division=0)
    if metric == "recall":
        return recall_score(y_true, yl, zero_division=0)
    if metric == "mcc":
        return matthews_corrcoef(y_true, yl)
    if metric == "kappa":
        return cohen_kappa_score(y_true, yl)
    raise ValueError(metric)


def best_threshold(y_true, p, metric=None, grid=None):
    metric = metric or THR_METRIC
    grid = grid if grid is not None else np.unique(np.quantile(p, np.linspace(0.01, 0.99, 197)))
    best_t, best_s = 0.5, -np.inf
    for t in grid:
        s = compute_metric(metric, y_true, p, t)
        if s > best_s:
            best_s, best_t = s, t
    return float(best_t), float(best_s)


def rank_score(y_true, p):
    """Skor untuk ranking model: metric lomba (threshold optimal kalau metrik label). Selalu 'lebih besar = lebih baik'."""
    if CFG.METRIC in LABEL_METRICS:
        return best_threshold(y_true, p, CFG.METRIC)[1]
    s = compute_metric(CFG.METRIC, y_true, p)
    return -s if CFG.METRIC in LOWER_IS_BETTER else s


def show_score(s):
    return -s if CFG.METRIC in LOWER_IS_BETTER else s


IMB_MODE = CFG.IMBALANCE
if IMB_MODE == "auto":
    IMB_MODE = "class_weight" if (CFG.METRIC in LABEL_METRICS and CHURN_RATE < 0.2) else "none"
if IMB_MODE == "smote" and not HAS_IMBLEARN:
    print("imblearn tidak ada → SMOTE diganti class_weight")
    IMB_MODE = "class_weight"
SCALE_POS = (1 - CHURN_RATE) / max(CHURN_RATE, 1e-6)
print("Metric        :", CFG.METRIC, "| threshold metric:", THR_METRIC)
print("Imbalance mode:", IMB_MODE, f"(scale_pos_weight = {SCALE_POS:.2f})")

## 13.4 Bangun matriks fitur

In [ ]:
with safe_section("Build feature matrices", critical=True):
    MODEL_FEATURES = [c for c in NUM_COLS + CAT_COLS + HIGH_CARD_COLS if c in train.columns]
    if not CFG.AUTO_DROP_LEAKY and LEAKY_COLS:
        print(f"⚠️ Fitur leak tetap ikut model (AUTO_DROP_LEAKY=False): {[c for c in LEAKY_COLS if c in MODEL_FEATURES]}")
    tr_df = train[MODEL_FEATURES].copy()
    te_df = test[MODEL_FEATURES].copy() if test is not None else pd.DataFrame(columns=MODEL_FEATURES)
    for c in tr_df.columns:
        if c not in te_df.columns:
            te_df[c] = np.nan
    te_df = te_df[MODEL_FEATURES]
    full_df = pd.concat([tr_df, te_df], ignore_index=True)
    N_TR = len(tr_df)

    cat_like = CAT_COLS + HIGH_CARD_COLS
    if CFG.USE_FREQ_ENCODING:
        for c in cat_like:
            full_df[f"{c}__freq"] = full_df[c].map(full_df[c].value_counts(normalize=True)).astype(float)
    full_df, NAME_MAP = sanitize_columns(full_df)
    INV_NAME = {v: k for k, v in NAME_MAP.items()}
    S_NUM = [NAME_MAP[c] for c in NUM_COLS if c in NAME_MAP] + [NAME_MAP[c] for c in NAME_MAP if c.endswith("__freq")]
    S_CAT = [NAME_MAP[c] for c in CAT_COLS if c in NAME_MAP]
    S_HC = [NAME_MAP[c] for c in HIGH_CARD_COLS if c in NAME_MAP]

    # 1) tree (LightGBM/XGB): category dtype, high-card raw dibuang (pakai freq + TE)
    X_tree_full = full_df[S_NUM + S_CAT].copy()
    for c in S_CAT:
        X_tree_full[c] = X_tree_full[c].astype(str).where(full_df[c].notna(), np.nan).astype("category")
    for c in S_NUM:
        X_tree_full[c] = pd.to_numeric(X_tree_full[c], errors="coerce").astype(float).replace([np.inf, -np.inf], np.nan)
    # 2) catboost: string kategori (termasuk high-card)
    X_cb_full = full_df[S_NUM + S_CAT + S_HC].copy()
    for c in S_CAT + S_HC:
        X_cb_full[c] = X_cb_full[c].astype(str).where(full_df[c].notna(), "NA")
    X_cb_full[S_NUM] = X_tree_full[S_NUM]
    # 3) ordinal (HGB/RF/ET)
    X_ord_full = X_tree_full.copy()
    for c in S_CAT:
        X_ord_full[c] = X_ord_full[c].cat.codes.replace(-1, np.nan).astype(float)
    # 4) linear raw (signed log untuk skew tinggi)
    X_lin_full = X_tree_full[S_NUM].copy()
    for c in S_NUM:
        s = X_lin_full[c]
        if s.nunique() > 10 and abs(s.skew()) > 2:
            X_lin_full[c] = np.sign(s) * np.log1p(np.abs(s))
    for c in S_CAT:
        X_lin_full[c] = full_df[c].astype(str).where(full_df[c].notna(), "Missing")

    REPS = {}
    for key, M in [("tree", X_tree_full), ("cb", X_cb_full), ("ord", X_ord_full), ("lin", X_lin_full)]:
        REPS[key] = (M.iloc[:N_TR].reset_index(drop=True), M.iloc[N_TR:].reset_index(drop=True))
    HC_RAW = (full_df[S_HC].iloc[:N_TR].reset_index(drop=True), full_df[S_HC].iloc[N_TR:].reset_index(drop=True))
    print(f"Fitur model: {len(MODEL_FEATURES)} asli → tree={X_tree_full.shape[1]}, catboost={X_cb_full.shape[1]}, "
          f"linear={X_lin_full.shape[1]} (+one-hot) | high-card TE: {S_HC}")

## 13.5 Cross-validation folds
- Default: **StratifiedKFold** (proporsi churn sama di tiap fold).
- `CFG.GROUP_COL` diisi (misal `household_id`, `agent_id`) → **StratifiedGroupKFold**: satu grup tidak boleh ada di train & valid sekaligus (mencegah leakage antar anggota grup).
- `N_REPEATS > 1` → repeated CV dengan seed berbeda → estimasi lebih stabil.

**Cara baca:** churn rate per fold harus hampir sama. Kalau CV std antar fold besar (> 0.02 AUC) → data kecil / noisy → pakai repeats & jangan terlalu percaya beda kecil antar model.

In [ ]:
def make_folds(n_folds, seed):
    idx = np.arange(N_TR)
    if (CFG.CV_STRATEGY in ("group", "auto")) and CFG.GROUP_COL and CFG.GROUP_COL in raw_train.columns:
        groups = raw_train[CFG.GROUP_COL].astype(str).values
        if HAS_SGKF:
            spl = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed).split(idx, y, groups)
        else:
            spl = GroupKFold(n_splits=n_folds).split(idx, y, groups)
    else:
        spl = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed).split(idx, y)
    return [(tr, va) for tr, va in spl]


with safe_section("CV folds", critical=True):
    FOLD_SETS = [make_folds(CFG.N_FOLDS, CFG.SEED + r) for r in range(CFG.N_REPEATS)]
    OPT_FOLDS = make_folds(CFG.OPTUNA_FOLDS, CFG.SEED + 100)
    chk = pd.DataFrame([{"fold": i, "n_valid": len(va), "churn_rate_valid": y[va].mean()} for i, (tr, va) in enumerate(FOLD_SETS[0])])
    display(chk)
    print("Strategi:", "group" if CFG.GROUP_COL else "stratified", "| folds:", CFG.N_FOLDS, "| repeats:", CFG.N_REPEATS)

# 14. Model Training (CV OOF)

Setiap model dilatih K kali (K fold). Prediksi pada fold validasi dikumpulkan jadi **OOF prediction** untuk seluruh train → dipakai untuk evaluasi, ensemble, threshold, dan analisis bisnis. Prediksi test = rata-rata K model.

**Cara baca leaderboard:**
- `cv_mean ± cv_std`: performa & stabilitas.
- `train_score` vs `cv_mean`: gap besar (misal train AUC 0.99 vs CV 0.80) → **overfitting** → naikkan regularisasi (`min_child_samples`, `reg_lambda`), turunkan `num_leaves`/`depth`, atau tambah data.
- `oof_score`: skor dari seluruh OOF sekaligus (ini yang dipakai untuk ranking).
- Model `dummy` = baseline. Semua model harus jauh lebih baik.

**Jika X → Y:**
- Semua model ≈ dummy → bug (target salah mapping? fitur ke-drop semua?) → cek Section 3–5.
- Skor terlalu bagus (AUC > 0.98) → hampir pasti **leakage** → cek Section 5.3.
- Satu model gagal (❌) → lihat `FAILED_MODELS`; model lain tetap jalan.
- LogReg ≈ GBDT → hubungan mostly linear; model sederhana cukup & lebih mudah dijelaskan (bagus untuk laporan!).

In [ ]:
FAILED_MODELS = {}
RESULTS = OrderedDict()


def target_encode_fold(tr_s, y_tr, others, smoothing=20, inner=5, seed=0):
    prior = y_tr.mean()
    tr_s = tr_s.astype(str).values
    enc_tr = np.zeros(len(tr_s))
    for itr, iva in StratifiedKFold(inner, shuffle=True, random_state=seed).split(tr_s, y_tr):
        st = pd.DataFrame({"k": tr_s[itr], "y": y_tr[itr]}).groupby("k")["y"].agg(["sum", "count"])
        m = (st["sum"] + smoothing * prior) / (st["count"] + smoothing)
        enc_tr[iva] = pd.Series(tr_s[iva]).map(m).fillna(prior).values
    st = pd.DataFrame({"k": tr_s, "y": y_tr}).groupby("k")["y"].agg(["sum", "count"])
    m = (st["sum"] + smoothing * prior) / (st["count"] + smoothing)
    return enc_tr, [pd.Series(o.astype(str).values).map(m).fillna(prior).values for o in others]


ACTIVE_DROP = set()   # kolom (nama sanitized) yang dibuang sementara (dipakai feature selection)


def fold_data(rep, tr_idx, va_idx, seed, y_vec=None):
    """Ambil representasi `rep` untuk fold, tambah target-encoding high-card (fit di train fold)."""
    y_vec = y if y_vec is None else y_vec
    Xtr_all, Xte_all = REPS[rep]
    keep = [c for c in Xtr_all.columns if c not in ACTIVE_DROP]
    Xtr, Xva, Xte = Xtr_all.iloc[tr_idx][keep].copy(), Xtr_all.iloc[va_idx][keep].copy(), Xte_all[keep].copy()
    if CFG.USE_TARGET_ENCODING and S_HC:
        hc_tr, hc_te = HC_RAW
        for c in S_HC:
            if f"{c}__te" in ACTIVE_DROP:
                continue
            e_tr, (e_va, e_te) = target_encode_fold(hc_tr[c].iloc[tr_idx], y_vec[tr_idx], [hc_tr[c].iloc[va_idx], hc_te[c]], seed=seed)
            Xtr[f"{c}__te"], Xva[f"{c}__te"], Xte[f"{c}__te"] = e_tr, e_va, e_te
    return Xtr, Xva, Xte


def default_params(name):
    es_metric_auc = CFG.METRIC in ("roc_auc", "gini", "pr_auc") or CFG.METRIC in LABEL_METRICS
    if name == "lgbm":
        return dict(n_estimators=10000, learning_rate=0.03, num_leaves=31, min_child_samples=30, subsample=0.8, subsample_freq=1,
                    colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1.0, cat_smooth=10, verbose=-1, n_jobs=CFG.N_JOBS,
                    metric="auc" if es_metric_auc else "binary_logloss")
    if name == "xgb":
        return dict(n_estimators=10000, learning_rate=0.03, max_depth=5, min_child_weight=3, subsample=0.8, colsample_bytree=0.7,
                    reg_lambda=1.0, tree_method="hist", eval_metric="auc" if es_metric_auc else "logloss", n_jobs=CFG.N_JOBS)
    if name == "catboost":
        return dict(iterations=10000, learning_rate=0.05, depth=6, l2_leaf_reg=3, loss_function="Logloss",
                    eval_metric="AUC" if es_metric_auc else "Logloss", verbose=0, allow_writing_files=False,
                    thread_count=CFG.N_JOBS if CFG.N_JOBS > 0 else -1)
    if name == "hgb":
        return dict(max_iter=2000, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=1.0,
                    early_stopping=True, validation_fraction=0.1, n_iter_no_change=50)
    if name == "rf":
        return dict(n_estimators=500, min_samples_leaf=3, max_features="sqrt", n_jobs=CFG.N_JOBS)
    if name == "et":
        return dict(n_estimators=500, min_samples_leaf=3, max_features="sqrt", n_jobs=CFG.N_JOBS)
    if name == "logreg":
        return dict(C=0.5, max_iter=5000)
    if name == "dummy":
        return dict(strategy="prior")
    raise ValueError(name)


MODEL_REP = {"lgbm": "tree", "xgb": "tree", "catboost": "cb", "hgb": "ord", "rf": "ord", "et": "ord", "logreg": "lin", "dummy": "ord"}


def maybe_smote(Xtr, ytr, seed):
    if IMB_MODE != "smote":
        return Xtr, ytr
    from imblearn.over_sampling import SMOTE
    Xn = Xtr.fillna(Xtr.median(numeric_only=True)).fillna(0)
    Xs, ys = SMOTE(random_state=seed).fit_resample(Xn, ytr)
    return pd.DataFrame(Xs, columns=Xtr.columns), np.asarray(ys)


def fit_predict(name, params, Xtr, ytr, Xva, yva, Xte, seed):
    """Latih 1 model di 1 fold. Return p_va, p_te, model, best_iter, p_tr."""
    p = dict(default_params(name))
    p.update(params or {})
    cw = IMB_MODE in ("class_weight", "smote")
    best_iter = None
    if name == "lgbm":
        if cw:
            p["scale_pos_weight"] = SCALE_POS
        if CFG.USE_GPU:
            p["device"] = "gpu"
        model = lgb.LGBMClassifier(random_state=seed, **p)
        cbs = []
        try:
            cbs = [lgb.early_stopping(CFG.EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(0)]
        except Exception:
            pass
        try:
            model.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=cbs)
        except Exception as e:
            if CFG.USE_GPU:
                p.pop("device", None)
                model = lgb.LGBMClassifier(random_state=seed, **p)
                model.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=cbs)
            else:
                raise e
        best_iter = getattr(model, "best_iteration_", None)
    elif name == "xgb":
        if cw:
            p["scale_pos_weight"] = SCALE_POS
        if CFG.USE_GPU:
            p["device"] = "cuda"
        es = CFG.EARLY_STOPPING_ROUNDS
        try:
            model = xgb.XGBClassifier(random_state=seed, enable_categorical=True, max_cat_to_onehot=1, early_stopping_rounds=es, **p)
            model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        except TypeError:
            # xgboost lama: early_stopping_rounds di fit(), kategori → kode
            Xtr2, Xva2, Xte = [d.apply(lambda s: s.cat.codes if str(s.dtype) == "category" else s) for d in (Xtr, Xva, Xte)]
            p.pop("device", None)
            model = xgb.XGBClassifier(random_state=seed, **p)
            model.fit(Xtr2, ytr, eval_set=[(Xva2, yva)], early_stopping_rounds=es, verbose=False)
            Xtr, Xva = Xtr2, Xva2
        except Exception as e:
            if CFG.USE_GPU:
                p.pop("device", None)
                model = xgb.XGBClassifier(random_state=seed, enable_categorical=True, max_cat_to_onehot=1, early_stopping_rounds=es, **p)
                model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
            else:
                raise e
        best_iter = getattr(model, "best_iteration", None)
    elif name == "catboost":
        cat_idx = [i for i, c in enumerate(Xtr.columns) if Xtr[c].dtype == object]
        if cw:
            p["auto_class_weights"] = "Balanced"
        if CFG.USE_GPU:
            p["task_type"] = "GPU"
        p.setdefault("od_type", "Iter")
        p.setdefault("od_wait", CFG.EARLY_STOPPING_ROUNDS)
        try:
            model = cb.CatBoostClassifier(random_seed=seed, cat_features=cat_idx, **p)
            model.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True)
        except Exception as e:
            if CFG.USE_GPU:
                p.pop("task_type", None)
                model = cb.CatBoostClassifier(random_seed=seed, cat_features=cat_idx, **p)
                model.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True)
            else:
                raise e
        best_iter = model.get_best_iteration()
    elif name == "hgb":
        Xtr, ytr = maybe_smote(Xtr, ytr, seed)
        model = HistGradientBoostingClassifier(random_state=seed, class_weight="balanced" if IMB_MODE == "class_weight" else None, **p) \
            if "class_weight" in HistGradientBoostingClassifier().get_params() else HistGradientBoostingClassifier(random_state=seed, **p)
        model.fit(Xtr, ytr)
        best_iter = getattr(model, "n_iter_", None)
    elif name in ("rf", "et"):
        Est = RandomForestClassifier if name == "rf" else ExtraTreesClassifier
        cwv = "balanced_subsample" if IMB_MODE == "class_weight" else None
        Xtr, ytr = maybe_smote(Xtr, ytr, seed)
        model = Pipeline([("imp", SimpleImputer(strategy="median")), ("m", Est(random_state=seed, class_weight=cwv, **p))])
        model.fit(Xtr, ytr)
    elif name == "logreg":
        num = [c for c in Xtr.columns if Xtr[c].dtype != object]
        cat = [c for c in Xtr.columns if Xtr[c].dtype == object]
        pre = ColumnTransformer([
            ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]), num),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
                              ("ohe", make_ohe(min_frequency=CFG.RARE_MIN_FREQ))]), cat)], remainder="drop")
        model = Pipeline([("pre", pre), ("m", LogisticRegression(class_weight="balanced" if IMB_MODE != "none" else None, **p))])
        model.fit(Xtr, ytr)
    elif name == "dummy":
        model = DummyClassifier(**p)
        model.fit(Xtr.fillna(0), ytr)
        Xva, Xte = Xva.fillna(0), Xte.fillna(0)
        Xtr = Xtr.fillna(0)
    else:
        raise ValueError(f"Model tidak dikenal: {name}")

    p_va = model.predict_proba(Xva)[:, 1]
    p_te = model.predict_proba(Xte)[:, 1] if len(Xte) else np.array([])
    sub = np.random.RandomState(seed).choice(len(Xtr), min(len(Xtr), 5000), replace=False)
    p_tr = (sub, model.predict_proba(Xtr.iloc[sub])[:, 1])
    return p_va, p_te, model, best_iter, p_tr


def model_available(name):
    return {"lgbm": HAS_LGB, "xgb": HAS_XGB, "catboost": HAS_CB, "hgb": HAS_HGB}.get(name, True)


def run_cv(name, params=None, seeds=None, fold_sets=None, label=None, verbose=True, keep_models=True):
    """CV lengkap untuk 1 model (bisa multi-seed & repeated). Return dict hasil."""
    label = label or name
    seeds = seeds or [CFG.SEED]
    fold_sets = fold_sets or FOLD_SETS
    rep = MODEL_REP[name]
    t0 = time.time()
    oof_sum, oof_cnt = np.zeros(N_TR), np.zeros(N_TR)
    n_te = len(REPS[rep][1])
    te_sum, n_models = np.zeros(n_te), 0
    fold_scores, train_scores, models, iters = [], [], [], []
    for r, folds in enumerate(fold_sets):
        for k, (tr, va) in enumerate(folds):
            Xtr, Xva, Xte = fold_data(rep, tr, va, seed=CFG.SEED + k)
            p_va_seeds = np.zeros(len(va))
            for s in seeds:
                p_va, p_te, model, bi, (sub, p_tr) = fit_predict(name, params, Xtr, y[tr], Xva, y[va], Xte, s + 17 * k + 1000 * r)
                p_va_seeds += p_va / len(seeds)
                if n_te:
                    te_sum += p_te
                n_models += 1
                iters.append(bi)
                if keep_models and r == 0:
                    models.append({"fold": k, "model": model, "va_idx": va, "rep": rep, "X_va": Xva})
                try:
                    train_scores.append(compute_metric("roc_auc", y[tr][sub], p_tr))
                except Exception:
                    pass
            oof_sum[va] += p_va_seeds
            oof_cnt[va] += 1
            fs = rank_score(y[va], p_va_seeds)
            fold_scores.append(show_score(fs))
            if verbose:
                print(f"  [{label}] repeat {r} fold {k}: {CFG.METRIC} = {show_score(fs):.5f}" + (f" | best_iter = {bi}" if bi else ""))
    oof = oof_sum / np.maximum(oof_cnt, 1)
    test_pred = te_sum / max(n_models, 1) if n_te else np.array([])
    res = {"name": label, "model": name, "params": params or {}, "oof": oof, "test": test_pred,
           "fold_scores": fold_scores, "cv_mean": float(np.mean(fold_scores)), "cv_std": float(np.std(fold_scores)),
           "oof_score": show_score(rank_score(y, oof)), "oof_auc": roc_auc_score(y, oof),
           "train_auc": float(np.mean(train_scores)) if train_scores else np.nan,
           "best_iters": [i for i in iters if i is not None], "models": models, "time_s": time.time() - t0, "seeds": seeds}
    if verbose:
        print(f"  ⇒ [{label}] OOF {CFG.METRIC} = {res['oof_score']:.5f} | CV {res['cv_mean']:.5f} ± {res['cv_std']:.5f} | "
              f"OOF AUC {res['oof_auc']:.4f} | train AUC {res['train_auc']:.4f} | {res['time_s']:.0f}s")
    return res


def leaderboard():
    rows = []
    for k, r in RESULTS.items():
        rows.append({"model": k, f"oof_{CFG.METRIC}": r["oof_score"], "cv_mean": r["cv_mean"], "cv_std": r["cv_std"],
                     "oof_auc": r["oof_auc"], "train_auc": r["train_auc"], "overfit_gap_auc": r["train_auc"] - r["oof_auc"],
                     "pr_auc": average_precision_score(y, r["oof"]), "logloss": log_loss(y, np.clip(r["oof"], 1e-7, 1 - 1e-7)),
                     "time_s": r["time_s"]})
    lb = pd.DataFrame(rows)
    if len(lb):
        lb = lb.sort_values(f"oof_{CFG.METRIC}", ascending=CFG.METRIC in LOWER_IS_BETTER).reset_index(drop=True)
    return lb


with safe_section("Train baseline models"):
    for name in ["dummy"] + [m for m in CFG.MODELS if m != "dummy"]:
        if not model_available(name):
            print(f"⏭️ {name}: library tidak tersedia → skip")
            continue
        print(f"\n===== {name} =====")
        try:
            RESULTS[name] = run_cv(name)
        except Exception as e:
            FAILED_MODELS[name] = f"{type(e).__name__}: {e}"
            print(f"❌ {name} gagal: {type(e).__name__}: {e}")
            traceback.print_exc(limit=2)
    LB = leaderboard()
    save_table(LB, "leaderboard_baseline")
    display(LB)
    if FAILED_MODELS:
        print("Model gagal:", FAILED_MODELS)

In [ ]:
with safe_section("Leaderboard plot"):
    LB = leaderboard()
    lbp = LB[LB["model"] != "dummy"]
    if len(lbp):
        fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(lbp))))
        colname = f"oof_{CFG.METRIC}"
        ax.barh(lbp["model"][::-1], lbp[colname][::-1], xerr=lbp["cv_std"][::-1], color=CFG.PALETTE[0], capsize=3)
        lo = lbp[colname].min() - 3 * lbp["cv_std"].max()
        hi = lbp[colname].max() + 2 * lbp["cv_std"].max()
        ax.set_xlim(max(0, lo) if CFG.METRIC not in LOWER_IS_BETTER else lo, hi)
        for i, v in enumerate(lbp[colname][::-1]):
            ax.text(v, i, f" {v:.4f}", va="center", fontsize=8)
        ax.set_xlabel(f"OOF {CFG.METRIC} (error bar = CV std)")
        ax.set_title("Model Comparison (cross-validated)")
        save_fig("model_leaderboard")
        gap = lbp[lbp["overfit_gap_auc"] > 0.08]
        if len(gap):
            print("⚠️ Gap train-vs-OOF AUC > 0.08 (indikasi overfit):", gap["model"].tolist())

# 15. Hyperparameter Tuning (Optuna)

Optuna mencari kombinasi hyperparameter terbaik dengan **TPE sampler** (lebih efisien dari grid/random search) + **pruning** (trial jelek dihentikan setelah beberapa fold).
- Tuning pakai `CFG.OPTUNA_FOLDS` fold (default 3, lebih cepat), hasil final dilatih ulang dengan `N_FOLDS`.
- Budget: `OPTUNA_TRIALS` trial **atau** `OPTUNA_TIMEOUT` detik per model (mana yang duluan).
- Untuk metrik label, tuning mengoptimalkan **ROC-AUC** (lebih stabil); threshold dituning belakangan.

**Jika X → Y:**
- Waktu mepet → `CFG.USE_OPTUNA = False` (default params sudah cukup kuat) atau `OPTUNA_TIMEOUT = 300`.
- Hasil tuned **lebih jelek** dari default di CV final → wajar terjadi (noise / overfit ke fold tuning); notebook tetap menyimpan keduanya dan ensemble memilih yang terbaik.
- Improvement < 0.002 AUC → tuning tidak terlalu penting untuk data ini; fokus ke fitur & insight.

**Untuk laporan:** cukup tulis "Hyperparameters were optimised with Bayesian optimisation (Optuna TPE, N trials, 3-fold CV)".

In [ ]:
def suggest_params(trial, name):
    if name == "lgbm":
        return dict(learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                    num_leaves=trial.suggest_int("num_leaves", 8, 256, log=True),
                    min_child_samples=trial.suggest_int("min_child_samples", 5, 200, log=True),
                    subsample=trial.suggest_float("subsample", 0.5, 1.0),
                    colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),
                    reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
                    reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
                    min_split_gain=trial.suggest_float("min_split_gain", 0, 1),
                    cat_smooth=trial.suggest_float("cat_smooth", 1, 50))
    if name == "xgb":
        return dict(learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                    max_depth=trial.suggest_int("max_depth", 2, 10),
                    min_child_weight=trial.suggest_float("min_child_weight", 1, 50, log=True),
                    subsample=trial.suggest_float("subsample", 0.5, 1.0),
                    colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),
                    reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
                    reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
                    gamma=trial.suggest_float("gamma", 0, 5))
    if name == "catboost":
        return dict(learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
                    depth=trial.suggest_int("depth", 3, 8),
                    l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 30, log=True),
                    random_strength=trial.suggest_float("random_strength", 0, 3),
                    bagging_temperature=trial.suggest_float("bagging_temperature", 0, 1))
    raise ValueError(name)


def tune_model(name):
    direction = "maximize"
    tune_metric = CFG.METRIC if CFG.METRIC in PROB_METRICS else "roc_auc"
    rep = MODEL_REP[name]

    def objective(trial):
        params = suggest_params(trial, name)
        scores = []
        for k, (tr, va) in enumerate(OPT_FOLDS):
            Xtr, Xva, Xte = fold_data(rep, tr, va, seed=CFG.SEED + k)
            p_va, _, _, _, _ = fit_predict(name, params, Xtr, y[tr], Xva, y[va], Xte.iloc[:0], CFG.SEED)
            s = compute_metric(tune_metric, y[va], p_va)
            scores.append(-s if tune_metric in LOWER_IS_BETTER else s)
            trial.report(float(np.mean(scores)), k)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(scores))

    study = optuna.create_study(direction=direction, sampler=optuna.samplers.TPESampler(seed=CFG.SEED),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
    study.optimize(objective, n_trials=CFG.OPTUNA_TRIALS, timeout=CFG.OPTUNA_TIMEOUT, gc_after_trial=True, catch=(Exception,))
    return study


TUNED_PARAMS = {}
with safe_section("Optuna tuning"):
    if not (CFG.USE_OPTUNA and HAS_OPTUNA):
        print("⏭️ Optuna dimatikan / tidak tersedia → pakai default params.")
    else:
        for name in [m for m in CFG.OPTUNA_MODELS if m in RESULTS and model_available(m)]:
            print(f"\n===== Tuning {name} ({CFG.OPTUNA_TRIALS} trials / {CFG.OPTUNA_TIMEOUT}s) =====")
            try:
                study = tune_model(name)
                done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
                if not done:
                    print("  tidak ada trial yang selesai")
                    continue
                TUNED_PARAMS[name] = study.best_params
                print(f"  best tuning score = {study.best_value:.5f} | trials selesai = {len(done)}")
                print("  best params:", study.best_params)
                RESULTS[f"{name}_tuned"] = run_cv(name, params=study.best_params, label=f"{name}_tuned")
                try:
                    hist = pd.DataFrame([{"trial": t.number, "value": t.value} for t in done])
                    fig, ax = plt.subplots(figsize=(7, 3.2))
                    ax.plot(hist["trial"], hist["value"], "o", alpha=0.4, ms=3)
                    ax.plot(hist["trial"], hist["value"].cummax(), "-", color=CFG.PALETTE[1])
                    ax.set_xlabel("Trial")
                    ax.set_ylabel("CV score")
                    ax.set_title(f"Optuna optimisation history — {name}")
                    save_fig(f"optuna_history_{name}")
                except Exception:
                    pass
            except Exception as e:
                FAILED_MODELS[f"{name}_tuned"] = str(e)
                print(f"❌ tuning {name} gagal: {e}")
        with open(os.path.join(CFG.MODEL_DIR, "tuned_params.json"), "w") as f:
            json.dump(TUNED_PARAMS, f, indent=2, default=str)
    LB = leaderboard()
    display(LB)

# 16. Multi-seed Final Models

GBDT punya randomness (subsample, colsample). Melatih model yang sama dengan **beberapa seed lalu dirata-rata** hampir selalu sedikit menaikkan & menstabilkan skor (gratis, hanya butuh waktu).
Diterapkan ke 2 model GBDT terbaik (versi default atau tuned, mana yang lebih bagus).

In [ ]:
with safe_section("Multi-seed final models"):
    if len(CFG.FINAL_SEEDS) <= 1:
        print("⏭️ FINAL_SEEDS hanya 1 seed → skip multi-seed.")
    else:
        LB = leaderboard()
        best_per_family = OrderedDict()
        for _, r in LB.iterrows():
            fam = RESULTS[r["model"]]["model"]
            if fam in ("lgbm", "xgb", "catboost", "hgb") and fam not in best_per_family:
                best_per_family[fam] = r["model"]
        for fam, key in list(best_per_family.items())[:2]:
            print(f"\n===== {fam} multi-seed (base: {key}, seeds {CFG.FINAL_SEEDS}) =====")
            try:
                RESULTS[f"{fam}_multiseed"] = run_cv(fam, params=RESULTS[key]["params"], seeds=CFG.FINAL_SEEDS,
                                                     label=f"{fam}_multiseed", verbose=False)
                r = RESULTS[f"{fam}_multiseed"]
                print(f"  ⇒ OOF {CFG.METRIC} = {r['oof_score']:.5f} (single-seed: {RESULTS[key]['oof_score']:.5f})")
            except Exception as e:
                FAILED_MODELS[f"{fam}_multiseed"] = str(e)
                print(f"❌ {e}")
    LB = leaderboard()
    save_table(LB, "leaderboard_all")
    display(LB)

## 16.2 Feature Selection (permutation importance)

Fitur yang **tidak membantu** (permutation importance ≤ 0: mengacak fitur itu tidak menurunkan skor) bisa jadi noise.
Notebook menghitung permutation importance pada fold validasi model GBDT terbaik, membuang fitur ≤ 0, lalu melatih ulang.
Hasil **hanya dipakai kalau CV membaik** (> 0.0005) — model baru masuk leaderboard sebagai `<model>_fs` dan ikut kandidat ensemble.

**Jika X → Y:** banyak fitur ber-importance negatif → fitur noise / redundant; untuk laporan, fokus pada fitur ber-importance tinggi saja.

In [ ]:
def permutation_importance_oof(res, n_repeats=None, max_folds=3, metric="auto"):
    """Permutation importance model-agnostic di fold validasi (pakai model fold yang tersimpan)."""
    n_repeats = n_repeats or CFG.PERM_IMPORTANCE_REPEATS
    metric = ("roc_auc" if CFG.METRIC in LABEL_METRICS else CFG.METRIC) if metric == "auto" else metric
    sign = -1 if metric in LOWER_IS_BETTER else 1
    rng = np.random.RandomState(CFG.SEED)
    rows = {}
    for md in res["models"][:max_folds]:
        model, Xva, va = md["model"], md["X_va"], md["va_idx"]
        base = sign * compute_metric(metric, y[va], model.predict_proba(Xva)[:, 1])
        for c in Xva.columns:
            drops = []
            for _ in range(n_repeats):
                Xp = Xva.copy()
                Xp[c] = Xp[c].values[rng.permutation(len(Xp))]
                drops.append(base - sign * compute_metric(metric, y[va], model.predict_proba(Xp)[:, 1]))
            rows.setdefault(c, []).extend(drops)
    out = pd.DataFrame([{"feature_model": c, "importance_mean": np.mean(v), "importance_std": np.std(v)} for c, v in rows.items()])
    out["feature"] = out["feature_model"].map(lambda c: INV_NAME.get(c, INV_NAME.get(c.replace("__te", ""), c)))
    return out.sort_values("importance_mean", ascending=False).reset_index(drop=True)


PERM_IMP = pd.DataFrame()
with safe_section("Feature selection"):
    LB = leaderboard()
    gb = [m for m in LB["model"] if RESULTS[m]["model"] in ("lgbm", "catboost", "xgb", "hgb") and RESULTS[m]["models"]]
    if not gb:
        print("⏭️ Tidak ada model GBDT untuk feature selection.")
    else:
        base_key = gb[0]
        base_res = RESULTS[base_key]
        PERM_IMP = permutation_importance_oof(base_res)
        print(f"Permutation importance ({base_key}):")
        display(PERM_IMP.head(25))
        useless = PERM_IMP[PERM_IMP["importance_mean"] <= 0]["feature_model"].tolist()
        print(f"Fitur dengan importance ≤ 0: {len(useless)} → {useless[:30]}")
        if CFG.USE_FEATURE_SELECTION and 0 < len(useless) < len(PERM_IMP) - 3:
            ACTIVE_DROP = set(useless)
            try:
                fam = base_res["model"]
                r_fs = run_cv(fam, params=base_res["params"], seeds=base_res["seeds"], label=f"{fam}_fs", verbose=False)
                r_fs["dropped_features"] = useless
                print(f"  {base_key}: {base_res['oof_score']:.5f} → {fam}_fs (tanpa {len(useless)} fitur): {r_fs['oof_score']:.5f}")
                if rank_score(y, r_fs["oof"]) > rank_score(y, base_res["oof"]) + 0.0005:
                    RESULTS[f"{fam}_fs"] = r_fs
                    print("  ✅ Feature selection membantu → model _fs masuk kandidat ensemble.")
                else:
                    print("  ➖ Tidak membantu signifikan → semua fitur dipertahankan.")
            finally:
                ACTIVE_DROP = set()
    save_table(PERM_IMP, "permutation_importance_selection")

# 17. Ensemble

Menggabungkan model yang **berbeda karakter** (misal CatBoost + LightGBM + LogReg) biasanya lebih baik dari model terbaik tunggal.

| Metode | Cara | Kapan bagus |
|---|---|---|
| Best single | model terbaik saja | baseline |
| Mean top-k | rata-rata probabilitas k model teratas | simpel & robust |
| Rank mean | rata-rata ranking (untuk AUC) | model dengan skala probabilitas beda |
| **Hill climbing** (Caruana) | tambah model satu per satu (boleh berulang) selama skor OOF naik | biasanya terbaik |
| Optimized weights | bobot non-negatif dioptimasi (scipy) | mirip hill climbing |
| **Stacking** | Logistic Regression di atas logit(OOF) dengan CV | kalau model saling melengkapi |

**Peringatan:** bobot dioptimasi di OOF yang sama → sedikit optimis. Kalau selisih metode < 0.0005, notebook memilih metode **paling sederhana** (lebih aman di test).

**Cek korelasi OOF antar model:** korelasi sangat tinggi (> 0.98) → ensemble tidak banyak membantu (modelnya "berpikir" sama).

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def rank01(a):
    return (stats.rankdata(a) - 1) / max(len(a) - 1, 1)


ENSEMBLES = OrderedDict()

with safe_section("Ensemble"):
    LB = leaderboard()
    cand = [m for m in LB["model"] if m != "dummy"]
    if not cand:
        raise RuntimeError("Tidak ada model yang berhasil dilatih.")
    oofs = {m: RESULTS[m]["oof"] for m in cand}
    tests = {m: RESULTS[m]["test"] for m in cand}
    has_test = all(len(tests[m]) for m in cand)

    if len(cand) > 1:
        corr = pd.DataFrame(oofs).corr(method="spearman")
        fig, ax = plt.subplots(figsize=(min(10, 1 + 0.7 * len(cand)), min(8, 1 + 0.6 * len(cand))))
        sns.heatmap(corr, annot=True, fmt=".3f", cmap="viridis", ax=ax, annot_kws={"size": 7})
        ax.set_title("Correlation of OOF Predictions Between Models")
        save_fig("oof_prediction_correlation")

    best = cand[0]
    ENSEMBLES["best_single"] = (oofs[best], tests[best], {best: 1.0})
    if CFG.USE_ENSEMBLE and len(cand) > 1:
        k = min(3, len(cand))
        top = cand[:k]
        ENSEMBLES[f"mean_top{k}"] = (np.mean([oofs[m] for m in top], 0), np.mean([tests[m] for m in top], 0) if has_test else np.array([]),
                                     {m: 1 / k for m in top})
        if CFG.METRIC not in LOWER_IS_BETTER:
            ENSEMBLES[f"rank_mean_top{k}"] = (np.mean([rank01(oofs[m]) for m in top], 0),
                                              np.mean([rank01(tests[m]) for m in top], 0) if has_test else np.array([]), {m: 1 / k for m in top})
        # hill climbing
        pool = cand[:min(10, len(cand))]
        chosen = [best]
        cur = oofs[best].copy()
        cur_s = rank_score(y, cur)
        for _ in range(30):
            best_m, best_s = None, cur_s
            for m in pool:
                trial = (cur * len(chosen) + oofs[m]) / (len(chosen) + 1)
                s = rank_score(y, trial)
                if s > best_s + 1e-6:
                    best_m, best_s = m, s
            if best_m is None:
                break
            chosen.append(best_m)
            cur = (cur * (len(chosen) - 1) + oofs[best_m]) / len(chosen)
            cur_s = best_s
        w = pd.Series(chosen).value_counts(normalize=True).to_dict()
        ENSEMBLES["hill_climb"] = (sum(w[m] * oofs[m] for m in w), sum(w[m] * tests[m] for m in w) if has_test else np.array([]), w)
        # optimized weights (minimize logloss)
        try:
            from scipy.optimize import minimize
            P = np.column_stack([oofs[m] for m in pool])

            def obj(wv):
                wv = np.abs(wv) / np.abs(wv).sum()
                return log_loss(y, np.clip(P @ wv, 1e-7, 1 - 1e-7))
            r = minimize(obj, np.ones(len(pool)) / len(pool), method="Nelder-Mead", options={"maxiter": 2000})
            wv = np.abs(r.x) / np.abs(r.x).sum()
            wd = {m: float(v) for m, v in zip(pool, wv) if v > 0.01}
            tot = sum(wd.values())
            wd = {m: v / tot for m, v in wd.items()}
            ENSEMBLES["optimized_weights"] = (sum(wd[m] * oofs[m] for m in wd), sum(wd[m] * tests[m] for m in wd) if has_test else np.array([]), wd)
        except Exception as e:
            print("optimized weights gagal:", e)
        # stacking
        try:
            Z = np.column_stack([logit(oofs[m]) for m in pool])
            Zt = np.column_stack([logit(tests[m]) for m in pool]) if has_test else None
            meta = LogisticRegression(C=1.0, max_iter=2000)
            meta_oof = cross_val_predict(meta, Z, y, cv=StratifiedKFold(5, shuffle=True, random_state=CFG.SEED + 7), method="predict_proba")[:, 1]
            meta.fit(Z, y)
            meta_test = meta.predict_proba(Zt)[:, 1] if has_test else np.array([])
            ENSEMBLES["stacking_logreg"] = (meta_oof, meta_test, dict(zip(pool, np.round(meta.coef_[0], 3))))
        except Exception as e:
            print("stacking gagal:", e)

    ens_rows = [{"method": k, f"oof_{CFG.METRIC}": show_score(rank_score(y, v[0])), "oof_auc": roc_auc_score(y, v[0]),
                 "weights": {m: round(float(w_), 3) for m, w_ in v[2].items()}} for k, v in ENSEMBLES.items()]
    ENS_TABLE = pd.DataFrame(ens_rows)
    save_table(ENS_TABLE.astype({"weights": str}), "ensemble_comparison")
    display(ENS_TABLE)

    simplicity = list(ENSEMBLES.keys())
    scores = {k: rank_score(y, v[0]) for k, v in ENSEMBLES.items()}
    top_s = max(scores.values())
    FINAL_NAME = next(k for k in simplicity if scores[k] >= top_s - 0.0005)
    FINAL_OOF, FINAL_TEST, FINAL_WEIGHTS = ENSEMBLES[FINAL_NAME]
    if roc_auc_score(y, FINAL_OOF) > 0.98:
        print("🚨🚨 OOF ROC-AUC > 0.98 — hampir pasti ada LEAKAGE (fitur yang tercatat setelah churn). Cek Section 5.3 & permutation importance teratas!")
    print(f"\n🏁 Final pilihan: {FINAL_NAME} | OOF {CFG.METRIC} = {show_score(scores[FINAL_NAME]):.5f} | weights = {FINAL_WEIGHTS}")
    add_insight("Model", f"The final model ({FINAL_NAME}) achieves a cross-validated {CFG.METRIC} of {show_score(scores[FINAL_NAME]):.4f} "
                         f"(ROC-AUC {roc_auc_score(y, FINAL_OOF):.4f}) versus 0.50 for a random baseline.")

# 18. Threshold & Calibration

## Threshold
Model mengeluarkan **probabilitas**. Untuk keputusan (churn / tidak) butuh **threshold**. 0.5 jarang optimal kalau data imbalanced.
- Threshold dioptimasi di OOF untuk `THR_METRIC` (F1 default; atau metric lomba kalau berbasis label).
- Plot precision/recall/F1 vs threshold membantu memilih trade-off.
- **Threshold bisnis** (biaya salah tebak) dihitung di Section 22.

## Calibration
Probabilitas **terkalibrasi** = kalau model bilang 30%, kira-kira 30% dari mereka benar churn. Penting untuk: logloss/brier, expected loss, CLV, ROI campaign.
- Reliability diagram: titik di garis diagonal = terkalibrasi.
- `CFG.CALIBRATION="auto"` → coba isotonic & sigmoid (Platt) via CV di atas OOF; dipakai kalau memperbaiki logloss.

**Jika X → Y:**
- Kurva di bawah diagonal → model over-confident (sering terjadi setelah class_weight/SMOTE) → kalibrasi wajib untuk analisis bisnis.
- Metric lomba AUC → kalibrasi tidak mengubah ranking; submission tetap pakai probabilitas mentah (kalibrasi hanya untuk analisis bisnis).

In [ ]:
with safe_section("Threshold tuning"):
    BEST_THR, BEST_THR_SCORE = best_threshold(y, FINAL_OOF, THR_METRIC)
    grid = np.linspace(0.02, 0.98, 97)
    curves = pd.DataFrame({"threshold": grid,
                           "precision": [precision_score(y, FINAL_OOF >= t, zero_division=0) for t in grid],
                           "recall": [recall_score(y, FINAL_OOF >= t, zero_division=0) for t in grid],
                           "f1": [f1_score(y, FINAL_OOF >= t, zero_division=0) for t in grid],
                           "accuracy": [accuracy_score(y, FINAL_OOF >= t) for t in grid]})
    fig, ax = plt.subplots(figsize=(8, 4))
    for c, col in zip(["precision", "recall", "f1", "accuracy"], CFG.PALETTE):
        ax.plot(curves["threshold"], curves[c], label=c, color=col)
    ax.axvline(BEST_THR, color="black", ls="--", lw=1, label=f"best {THR_METRIC} thr = {BEST_THR:.3f}")
    ax.axvline(0.5, color="gray", ls=":", lw=1)
    ax.set_xlabel("Decision threshold")
    ax.set_title("Precision / Recall / F1 vs Threshold (OOF)")
    ax.legend(fontsize=8)
    save_fig("threshold_curves")
    print(f"Threshold optimal untuk {THR_METRIC}: {BEST_THR:.4f} → {THR_METRIC} = {BEST_THR_SCORE:.4f} "
          f"(threshold 0.5 → {compute_metric(THR_METRIC, y, FINAL_OOF, 0.5):.4f})")

In [ ]:
FINAL_OOF_CAL, FINAL_TEST_CAL, CAL_METHOD = FINAL_OOF, FINAL_TEST, "none"

with safe_section("Calibration"):
    def cv_calibrate(p, method):
        out = np.zeros_like(p)
        for tr, va in StratifiedKFold(5, shuffle=True, random_state=CFG.SEED + 3).split(p, y):
            if method == "isotonic":
                cal = IsotonicRegression(out_of_bounds="clip", y_min=1e-4, y_max=1 - 1e-4).fit(p[tr], y[tr])
                out[va] = cal.predict(p[va])
            else:
                cal = LogisticRegression(C=1e6, max_iter=2000).fit(logit(p[tr]).reshape(-1, 1), y[tr])
                out[va] = cal.predict_proba(logit(p[va]).reshape(-1, 1))[:, 1]
        return out

    base_ll = log_loss(y, np.clip(FINAL_OOF, 1e-7, 1 - 1e-7))
    cal_res = {"none": (base_ll, FINAL_OOF)}
    methods = ["isotonic", "sigmoid"] if CFG.CALIBRATION == "auto" else ([] if CFG.CALIBRATION == "none" else [CFG.CALIBRATION])
    for m in methods:
        pc = cv_calibrate(FINAL_OOF, m)
        cal_res[m] = (log_loss(y, np.clip(pc, 1e-7, 1 - 1e-7)), pc)
    CAL_METHOD = min(cal_res, key=lambda k: cal_res[k][0]) if CFG.CALIBRATION == "auto" else (CFG.CALIBRATION if CFG.CALIBRATION in cal_res else "none")
    if CAL_METHOD != "none" and cal_res[CAL_METHOD][0] > base_ll - 0.001:
        CAL_METHOD = "none"   # perbaikan tidak berarti
    for k, (ll, _) in cal_res.items():
        print(f"  {k:<9}: OOF logloss = {ll:.5f}, Brier = {brier_score_loss(y, cal_res[k][1]):.5f}")
    print("Kalibrasi dipakai:", CAL_METHOD)
    if CAL_METHOD != "none":
        FINAL_OOF_CAL = cal_res[CAL_METHOD][1]
        if len(FINAL_TEST):
            if CAL_METHOD == "isotonic":
                cal = IsotonicRegression(out_of_bounds="clip", y_min=1e-4, y_max=1 - 1e-4).fit(FINAL_OOF, y)
                FINAL_TEST_CAL = cal.predict(FINAL_TEST)
            else:
                cal = LogisticRegression(C=1e6, max_iter=2000).fit(logit(FINAL_OOF).reshape(-1, 1), y)
                FINAL_TEST_CAL = cal.predict_proba(logit(FINAL_TEST).reshape(-1, 1))[:, 1]

    from sklearn.calibration import calibration_curve
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
    for k, (ll, pc) in cal_res.items():
        fy, fx = calibration_curve(y, pc, n_bins=10, strategy="quantile")
        ax.plot(fx, fy, "o-", ms=4, label=f"{k} (logloss {ll:.4f})")
    ax.set_xlabel("Mean predicted churn probability")
    ax.set_ylabel("Observed churn rate")
    ax.set_title("Reliability Diagram (OOF)")
    ax.legend(fontsize=8)
    save_fig("calibration_reliability")

# 19. Evaluation & Debugging

| Cek | Untuk apa | Tanda bahaya → tindakan |
|---|---|---|
| ROC & PR curve | performa di semua threshold | kurva PR rendah walau AUC bagus → kelas minoritas sulit; fokus recall/precision trade-off |
| Confusion matrix | jenis kesalahan (FN vs FP) | FN banyak = churner lolos → turunkan threshold kalau recall penting |
| Fold stability | konsistensi | std tinggi → data kecil/noisy, pakai repeats |
| **Learning curve** | butuh data lebih / model lebih kompleks? | train ≫ valid & valid masih naik → **variance** (tambah regularisasi/data); keduanya rendah & datar → **bias** (tambah fitur) |
| **Shuffled-target test** | deteksi bug/leakage di pipeline | AUC dengan target diacak harus ≈ 0.5. Kalau > 0.55 → ada kebocoran (misal target encoding salah) |
| OOF vs test distribution | apakah test "mirip" | rata-rata prediksi test jauh dari churn rate train → drift / perbedaan populasi |
| Error analysis | siapa yang salah ditebak | pola di False Negative = ide fitur baru |

**Untuk laporan:** ROC curve + confusion matrix (di threshold terpilih) + 1 kalimat recall/precision cukup. Learning curve & shuffled test → lampiran / 1 kalimat validasi metodologi.

In [ ]:
with safe_section("ROC / PR curves & confusion matrix"):
    LB = leaderboard()
    show_models = [m for m in LB["model"] if m != "dummy"][:4]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
    curves_src = [("FINAL (" + FINAL_NAME + ")", FINAL_OOF)] + [(m, RESULTS[m]["oof"]) for m in show_models]
    for i, (nm, p) in enumerate(curves_src):
        fpr, tpr, _ = roc_curve(y, p)
        prc, rcl, _ = precision_recall_curve(y, p)
        lw = 2.5 if i == 0 else 1.2
        ax[0].plot(fpr, tpr, lw=lw, label=f"{nm} (AUC {roc_auc_score(y, p):.3f})", color=CFG.PALETTE[i % len(CFG.PALETTE)])
        ax[1].plot(rcl, prc, lw=lw, label=f"{nm} (AP {average_precision_score(y, p):.3f})", color=CFG.PALETTE[i % len(CFG.PALETTE)])
    ax[0].plot([0, 1], [0, 1], "k--", lw=0.8)
    ax[0].set_xlabel("False positive rate")
    ax[0].set_ylabel("True positive rate (recall)")
    ax[0].set_title("ROC Curve (OOF)")
    ax[1].axhline(CHURN_RATE, color="k", ls="--", lw=0.8, label=f"baseline {CHURN_RATE:.2f}")
    ax[1].set_xlabel("Recall")
    ax[1].set_ylabel("Precision")
    ax[1].set_title("Precision-Recall Curve (OOF)")
    for a in ax:
        a.legend(fontsize=7)
    save_fig("roc_pr_curves")

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    for a, thr, ttl in [(ax[0], 0.5, "threshold 0.5"), (ax[1], BEST_THR, f"optimal threshold {BEST_THR:.3f} ({THR_METRIC})")]:
        cm = confusion_matrix(y, (FINAL_OOF >= thr).astype(int))
        annot = np.array([[f"{v:,}\n({v / cm.sum():.1%})" for v in row] for row in cm])
        sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", cbar=False, ax=a,
                    xticklabels=["Pred: Retain", "Pred: Churn"], yticklabels=["Actual: Retain", "Actual: Churn"])
        a.set_title(f"Confusion Matrix — {ttl}", fontsize=10)
    save_fig("confusion_matrices")
    yl = (FINAL_OOF >= BEST_THR).astype(int)
    print(classification_report(y, yl, target_names=["Retained", "Churned"], digits=4))
    rec, prec = recall_score(y, yl), precision_score(y, yl)
    add_insight("Model", f"At the selected threshold ({BEST_THR:.2f}) the model identifies {pct(rec)} of churners (recall) with a precision of {pct(prec)}, "
                         f"i.e. {prec / CHURN_RATE:.1f}x better targeting than random selection.")

### 19.1 Bootstrap confidence interval & uji signifikansi antar model
Skor tunggal (misal AUC 0.749) tidak menunjukkan **ketidakpastian**. Bootstrap (resample customer dengan pengembalian, N kali) memberi **95% CI**.
**Paired bootstrap** membandingkan model final vs pembanding pada sampel yang sama → apakah selisihnya nyata atau kebetulan?

**Cara baca:** CI sempit → estimasi stabil. Selisih model dengan CI yang melewati 0 / p > 0.05 → **tidak berbeda signifikan** → pilih yang lebih sederhana/interpretable (argumen kuat di laporan).
**Untuk laporan:** *"The final model achieves ROC-AUC = 0.749 (95% CI 0.737–0.761); its improvement over the interpretable logistic regression is +0.003 (95% CI −0.001 to 0.007, p = 0.12)."*

In [ ]:
BOOT_TABLE = pd.DataFrame()
with safe_section("Bootstrap CI & model comparison"):
    rng = np.random.RandomState(CFG.SEED)
    B = CFG.N_BOOTSTRAP
    idx_b = [rng.randint(0, N_TR, N_TR) for _ in range(B)]

    def boot_scores(p, metric, thr=0.5):
        out = []
        for ib in idx_b:
            if y[ib].min() == y[ib].max():
                continue
            out.append(compute_metric(metric, y[ib], p[ib], thr))
        return np.array(out)

    metrics_to_report = ["roc_auc", "pr_auc", "logloss"] + ([CFG.METRIC] if CFG.METRIC not in ("roc_auc", "pr_auc", "logloss") else [])
    rows = []
    for m in metrics_to_report:
        thr = BEST_THR if m in LABEL_METRICS else 0.5
        bs = boot_scores(FINAL_OOF, m, thr)
        rows.append({"metric": m, "estimate": compute_metric(m, y, FINAL_OOF, thr), "ci_low": np.percentile(bs, 2.5), "ci_high": np.percentile(bs, 97.5)})
    BOOT_TABLE = pd.DataFrame(rows)
    display(BOOT_TABLE)
    save_table(BOOT_TABLE, "bootstrap_ci_final_model")
    r0 = BOOT_TABLE.iloc[0]
    FINAL_AUC_CI = (r0["ci_low"], r0["ci_high"])
    add_insight("Model", f"Final model ROC-AUC = {r0['estimate']:.3f} (95% bootstrap CI {r0['ci_low']:.3f}–{r0['ci_high']:.3f}).")

    LB = leaderboard()
    comps = [m for m in LB["model"] if m not in ("dummy",)][:4]
    if "logreg" in RESULTS and "logreg" not in comps:
        comps.append("logreg")
    cmp_metric = "roc_auc" if CFG.METRIC in LABEL_METRICS else CFG.METRIC
    sign = -1 if cmp_metric in LOWER_IS_BETTER else 1
    crow = []
    for m in comps:
        pm = RESULTS[m]["oof"]
        diffs = np.array([sign * (compute_metric(cmp_metric, y[ib], FINAL_OOF[ib]) - compute_metric(cmp_metric, y[ib], pm[ib]))
                          for ib in idx_b[: min(B, 500)] if y[ib].min() != y[ib].max()])
        crow.append({"final_vs": m, f"diff_{cmp_metric}": sign * (compute_metric(cmp_metric, y, FINAL_OOF) - compute_metric(cmp_metric, y, pm)),
                     "ci_low": np.percentile(diffs, 2.5), "ci_high": np.percentile(diffs, 97.5),
                     "p_value_one_sided": float((diffs <= 0).mean()), "significant": bool(np.percentile(diffs, 2.5) > 0)})
    MODEL_CMP = pd.DataFrame(crow)
    print(f"Paired bootstrap: final ({FINAL_NAME}) vs model lain ({cmp_metric}; positif = final lebih baik):")
    display(MODEL_CMP)
    save_table(MODEL_CMP, "bootstrap_model_comparison")
    if "logreg" in RESULTS:
        lr_row = MODEL_CMP[MODEL_CMP["final_vs"] == "logreg"]
        if len(lr_row):
            lr_row = lr_row.iloc[0]
            verdict = "significantly better than" if lr_row["significant"] else "not significantly different from"
            add_insight("Model", f"The final model is {verdict} the interpretable logistic regression "
                                 f"(Δ{cmp_metric} = {lr_row[f'diff_{cmp_metric}']:+.4f}, 95% CI {lr_row['ci_low']:+.4f} to {lr_row['ci_high']:+.4f}).")

In [ ]:
with safe_section("Fold stability"):
    LB = leaderboard()
    ms = [m for m in LB["model"] if m != "dummy"][:8]
    fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(ms))))
    ax.boxplot([RESULTS[m]["fold_scores"] for m in ms][::-1], vert=False, labels=ms[::-1])
    ax.set_xlabel(f"{CFG.METRIC} per fold")
    ax.set_title("Cross-Validation Score Distribution per Model")
    save_fig("fold_stability")
    for m in ms:
        if RESULTS[m]["cv_std"] > 0.02 and CFG.METRIC in ("roc_auc", "gini", "pr_auc"):
            print(f"⚠️ {m}: std antar fold {RESULTS[m]['cv_std']:.4f} > 0.02 → tidak stabil")

In [ ]:
with safe_section("Learning curve"):
    fam = "lgbm" if HAS_LGB else ("catboost" if HAS_CB else "hgb")
    base_params = dict(RESULTS.get(f"{fam}_tuned", RESULTS.get(fam, {"params": {}}))["params"])
    if fam in ("lgbm", "xgb", "catboost", "hgb"):
        base_params["learning_rate"] = max(base_params.get("learning_rate", 0.05), 0.08)  # lebih cepat; kurva relatif tetap valid
    sizes = [0.2, 0.5, 1.0] if CFG.RUN_MODE == "fast" else [0.1, 0.25, 0.5, 0.75, 1.0]
    folds3 = make_folds(3, CFG.SEED + 55)
    rows = []
    for frac in sizes:
        for k, (tr, va) in enumerate(folds3):
            rng = np.random.RandomState(k)
            tr_s = np.sort(rng.choice(tr, max(50, int(len(tr) * frac)), replace=False))
            Xtr, Xva, Xte = fold_data(MODEL_REP[fam], tr_s, va, seed=k)
            p_va, _, mdl, _, (sub, p_tr) = fit_predict(fam, base_params, Xtr, y[tr_s], Xva, y[va], Xte.iloc[:0], CFG.SEED)
            rows.append({"train_size": len(tr_s), "train_auc": roc_auc_score(y[tr_s][sub], p_tr), "valid_auc": roc_auc_score(y[va], p_va)})
    LC = pd.DataFrame(rows).groupby("train_size").agg(["mean", "std"]).reset_index()
    fig, ax = plt.subplots(figsize=(7, 4))
    for col, c in [("train_auc", CFG.PALETTE[1]), ("valid_auc", CFG.PALETTE[0])]:
        ax.plot(LC["train_size"], LC[(col, "mean")], "o-", color=c, label=col.replace("_", " "))
        ax.fill_between(LC["train_size"], LC[(col, "mean")] - LC[(col, "std")], LC[(col, "mean")] + LC[(col, "std")], color=c, alpha=0.15)
    ax.set_xlabel("Training set size")
    ax.set_ylabel("ROC-AUC")
    ax.set_title(f"Learning Curve ({fam})")
    ax.legend()
    save_fig("learning_curve")
    gap = LC[("train_auc", "mean")].iloc[-1] - LC[("valid_auc", "mean")].iloc[-1]
    slope = LC[("valid_auc", "mean")].iloc[-1] - LC[("valid_auc", "mean")].iloc[-2]
    print(f"Gap train-valid di ukuran penuh: {gap:.4f} | kenaikan valid dari 75%→100% data: {slope:+.4f}")
    if gap > 0.08:
        print("→ Variance tinggi (overfit): naikkan regularisasi / min_child_samples, kurangi num_leaves/depth.")
    if slope > 0.005:
        print("→ Valid masih naik: tambahan data akan membantu (sebut di laporan sebagai peluang).")
    if gap < 0.03 and slope <= 0.005:
        print("→ Model sudah jenuh terhadap data; peningkatan berikutnya datang dari fitur baru, bukan tuning.")

In [ ]:
with safe_section("Shuffled-target sanity test"):
    fam = "lgbm" if HAS_LGB else "hgb"
    y_perm = np.random.RandomState(CFG.SEED).permutation(y)
    p_perm = np.zeros(N_TR)
    for k, (tr, va) in enumerate(make_folds(3, CFG.SEED + 77)):
        Xtr, Xva, Xte = fold_data(MODEL_REP[fam], tr, va, seed=k, y_vec=y_perm)
        p_va, _, _, _, _ = fit_predict(fam, {"n_estimators": 300} if fam == "lgbm" else {}, Xtr, y_perm[tr], Xva, y_perm[va], Xte.iloc[:0], CFG.SEED)
        p_perm[va] = p_va
    auc_perm = roc_auc_score(y_perm, p_perm)
    print(f"AUC dengan target DIACAK = {auc_perm:.4f} (harus ≈ 0.50)")
    print("✅ Pipeline bersih dari kebocoran target." if auc_perm < 0.55 else "🚨 AUC > 0.55 dengan target acak → ada leakage di pipeline! Cek target encoding / fitur.")

In [ ]:
with safe_section("OOF vs test prediction distribution"):
    if len(FINAL_TEST):
        fig, ax = plt.subplots(figsize=(7, 3.8))
        ax.hist(FINAL_OOF, bins=50, alpha=0.55, density=True, label=f"Train OOF (mean {FINAL_OOF.mean():.3f})", color=CFG.PALETTE[0])
        ax.hist(FINAL_TEST, bins=50, alpha=0.55, density=True, label=f"Test (mean {FINAL_TEST.mean():.3f})", color=CFG.PALETTE[1])
        ax.set_xlabel("Predicted churn probability")
        ax.set_title("Prediction Distribution: Train OOF vs Test")
        ax.legend()
        save_fig("oof_vs_test_predictions")
        ks = stats.ks_2samp(FINAL_OOF, FINAL_TEST)
        print(f"KS = {ks.statistic:.4f} (p = {ks.pvalue:.4f}) | mean OOF = {FINAL_OOF.mean():.4f}, mean test = {FINAL_TEST.mean():.4f}, churn rate train = {CHURN_RATE:.4f}")
        if ks.statistic > 0.1:
            print("⚠️ Distribusi prediksi test berbeda cukup jauh → cek drift (Section 12).")
    else:
        print("⏭️ Tidak ada test.")

In [ ]:
with safe_section("Error analysis"):
    yl = (FINAL_OOF >= BEST_THR).astype(int)
    err = train.copy()
    err["_y"] = y
    err["_p"] = FINAL_OOF
    err["_type"] = np.select([(y == 1) & (yl == 1), (y == 1) & (yl == 0), (y == 0) & (yl == 1)], ["TP", "FN", "FP"], "TN")
    print(err["_type"].value_counts().to_string())
    prof_feats = [f for f in dedupe_features(DRIVER_TABLE["feature"].tolist()) if f in train.columns][:12]
    prof = pd.DataFrame({"feature": prof_feats})
    for t in ["TP", "FN", "FP", "TN"]:
        m = (err["_type"] == t).values
        vals = []
        for f in prof_feats:
            s = train.loc[m, f]
            vals.append(s.mean() if pd.api.types.is_numeric_dtype(s) else (s.mode().iloc[0] if s.notna().any() else None))
        prof[t] = vals
    print("\nProfil per jenis prediksi (mean numeric / modus kategori). FN = churner yang lolos:")
    display(prof)
    save_table(prof, "error_profile")
    print("\nFalse Negative paling 'meyakinkan' (churn tapi diprediksi aman):")
    cols_show = ([ID_COL] if ID_COL else []) + prof_feats[:8] + ["_p"]
    display(err[err["_type"] == "FN"].sort_values("_p")[cols_show].head(10))
    print("💡 Cari pola di FN: kalau mereka mirip customer setia di semua fitur → churn mereka dipicu faktor yang tidak ada di data (sebut sebagai limitation / usul data tambahan).")

    seg_rows = []
    for f in [f for f in DRIVER_TABLE["feature"] if f in CAT_COLS][:4]:
        lab = train[f].astype(str).where(train[f].notna(), "Missing")
        for lv in lab.value_counts().index[:8]:
            m = (lab == lv).values
            if m.sum() >= 100 and 0 < y[m].sum() < m.sum():
                seg_rows.append({"feature": f, "level": lv, "n": int(m.sum()), "churn_rate": y[m].mean(), "auc": roc_auc_score(y[m], FINAL_OOF[m])})
    if seg_rows:
        SEG_PERF = pd.DataFrame(seg_rows)
        print("\nPerforma model per segmen (AUC jauh lebih rendah di suatu segmen → model kurang andal di sana):")
        display(SEG_PERF)
        save_table(SEG_PERF, "performance_by_segment")

# 20. Feature Importance & Explainability

Beberapa sudut pandang (pakai lebih dari satu supaya kesimpulan kuat):

| Metode | Arti | Kelebihan / kekurangan |
|---|---|---|
| **Gain importance** | total kontribusi split fitur di pohon | cepat; bias ke fitur numerik/kardinalitas tinggi |
| **Permutation importance** | turunnya skor kalau fitur diacak (di data validasi) | model-agnostic, jujur; fitur berkorelasi bisa "berbagi" importance |
| **SHAP** | kontribusi tiap fitur ke prediksi **tiap customer** (log-odds) | konsisten, punya **arah** (+ naikkan risiko / − turunkan), bisa per individu |
| **PDP** | rata-rata prediksi saat fitur diubah | menunjukkan bentuk hubungan (linear, threshold, U-shape) |

**Cara baca SHAP beeswarm:** tiap titik = 1 customer. Sumbu x = dampak ke risiko churn (kanan = naik). Warna = nilai fitur (merah tinggi, biru rendah).
Contoh: titik merah di kanan untuk `n_complaints` → keluhan banyak menaikkan risiko churn.

**Jika X → Y:**
- Fitur penting tapi tidak bisa dikontrol (umur, region) → dipakai untuk **targeting/segmentasi**, bukan intervensi.
- Fitur penting dan bisa dikontrol (metode bayar, auto-renew, keluhan, kenaikan premi) → **basis rekomendasi** (Section 22).
- Ranking SHAP sangat beda dengan permutation → fitur berkorelasi; sebut keduanya / gabungkan family.

**Untuk laporan:** SHAP bar/beeswarm (1 figure) + 1–2 dependence/PDP untuk driver utama + 1 waterfall contoh customer → sangat meyakinkan juri.

In [ ]:
def pick_explain_model():
    LB = leaderboard()
    for m in LB["model"]:
        r = RESULTS[m]
        if r["model"] in ("lgbm", "catboost", "xgb") and r["models"]:
            return m
    for m in LB["model"]:
        if RESULTS[m]["models"] and m != "dummy":
            return m
    return None


def display_name(c):
    if c.endswith("__te"):
        return f"{INV_NAME.get(c[:-4], c[:-4])} (target-enc)"
    return INV_NAME.get(c, c)


def get_contribs(md, fam):
    model, X = md["model"], md["X_va"]
    if fam == "lgbm":
        c = model.predict(X, pred_contrib=True)
    elif fam == "xgb":
        dm = xgb.DMatrix(X, enable_categorical=True)
        bi = getattr(model, "best_iteration", None)
        try:
            c = model.get_booster().predict(dm, pred_contribs=True, iteration_range=(0, bi + 1) if bi is not None else (0, 0))
        except TypeError:
            c = model.get_booster().predict(dm, pred_contribs=True)
    elif fam == "catboost":
        idx = [i for i, col in enumerate(X.columns) if X[col].dtype == object]
        c = model.get_feature_importance(cb.Pool(X, cat_features=idx), type="ShapValues")
    else:
        return None, None
    c = np.asarray(c)
    return c[:, :-1], c[:, -1]


EXPLAIN_KEY = None
SHAP_VALUES, SHAP_X, SHAP_IDX = None, None, None
SHAP_IMP = pd.DataFrame()
GAIN_IMP = pd.DataFrame()

with safe_section("Gain importance"):
    EXPLAIN_KEY = pick_explain_model()
    print("Model untuk explainability:", EXPLAIN_KEY)
    fam = RESULTS[EXPLAIN_KEY]["model"]
    imps = []
    for md in RESULTS[EXPLAIN_KEY]["models"]:
        m, cols = md["model"], md["X_va"].columns
        if fam == "lgbm":
            v = m.booster_.feature_importance("gain")
        elif fam == "xgb":
            sc = m.get_booster().get_score(importance_type="gain")
            v = [sc.get(c, 0) for c in cols]
        elif fam == "catboost":
            v = m.get_feature_importance()
        elif fam in ("rf", "et"):
            v = m.named_steps["m"].feature_importances_
            if len(v) != len(cols):
                continue
        else:
            continue
        imps.append(pd.Series(np.asarray(v, dtype=float), index=cols))
    if imps:
        g = pd.concat(imps, axis=1).mean(axis=1)
        GAIN_IMP = pd.DataFrame({"feature": [display_name(c) for c in g.index], "gain": g.values})
        GAIN_IMP["gain_share_%"] = 100 * GAIN_IMP["gain"] / GAIN_IMP["gain"].sum()
        GAIN_IMP = GAIN_IMP.sort_values("gain", ascending=False).reset_index(drop=True)
        save_table(GAIN_IMP, "gain_importance")
        top = GAIN_IMP.head(CFG.TOP_N_FEATURES_PLOT)
        fig, ax = plt.subplots(figsize=(8, max(3, 0.32 * len(top))))
        ax.barh(top["feature"][::-1], top["gain_share_%"][::-1], color=CFG.PALETTE[0])
        ax.set_xlabel("Share of total gain (%)")
        ax.set_title(f"Feature Importance (gain) — {EXPLAIN_KEY}")
        save_fig("gain_importance")
    else:
        print("Gain importance tidak tersedia untuk model ini.")

In [ ]:
with safe_section("Permutation importance"):
    if EXPLAIN_KEY:
        if len(PERM_IMP) == 0 or RESULTS[EXPLAIN_KEY]["models"][0]["X_va"].shape[1] != len(PERM_IMP):
            PERM_IMP = permutation_importance_oof(RESULTS[EXPLAIN_KEY])
        top = PERM_IMP.head(CFG.TOP_N_FEATURES_PLOT)
        fig, ax = plt.subplots(figsize=(8, max(3, 0.32 * len(top))))
        ax.barh([display_name(c) for c in top["feature_model"]][::-1], top["importance_mean"][::-1], xerr=top["importance_std"][::-1],
                color=CFG.PALETTE[2], capsize=2)
        ax.axvline(0, color="k", lw=0.8)
        ax.set_xlabel("Drop in validation score when feature is shuffled")
        ax.set_title("Permutation Importance (out-of-fold)")
        save_fig("permutation_importance")
        save_table(PERM_IMP, "permutation_importance")

In [ ]:
with safe_section("SHAP values"):
    fam = RESULTS[EXPLAIN_KEY]["model"]
    vals, Xs, idxs, base_vals = [], [], [], []
    for md in RESULTS[EXPLAIN_KEY]["models"]:
        v, b = get_contribs(md, fam)
        if v is None:
            break
        vals.append(v)
        Xs.append(md["X_va"])
        idxs.append(md["va_idx"])
        base_vals.append(np.mean(b))
    if not vals and HAS_SHAP:
        md = RESULTS[EXPLAIN_KEY]["models"][0]
        Xb = md["X_va"].iloc[:300]
        expl = shap.Explainer(lambda d: md["model"].predict_proba(pd.DataFrame(d, columns=Xb.columns))[:, 1], Xb)
        sv = expl(Xb)
        vals, Xs, idxs, base_vals = [sv.values], [Xb], [md["va_idx"][:300]], [float(np.mean(sv.base_values))]
        print("(SHAP model-agnostic, sampel 300 baris — skala probabilitas)")
    if not vals:
        print("⏭️ SHAP tidak tersedia untuk model ini. Pakai permutation importance.")
    else:
        SHAP_VALUES = np.vstack(vals)
        SHAP_X = pd.concat(Xs, ignore_index=True)
        SHAP_IDX = np.concatenate(idxs)
        SHAP_BASE = float(np.mean(base_vals))
        if len(SHAP_X) > CFG.SHAP_SAMPLE:
            sel = np.random.RandomState(CFG.SEED).choice(len(SHAP_X), CFG.SHAP_SAMPLE, replace=False)
            SHAP_VALUES, SHAP_X, SHAP_IDX = SHAP_VALUES[sel], SHAP_X.iloc[sel].reset_index(drop=True), SHAP_IDX[sel]
        SHAP_NAMES = [display_name(c) for c in SHAP_X.columns]
        SHAP_XNUM = SHAP_X.copy()
        for c in SHAP_XNUM.columns:
            if not pd.api.types.is_numeric_dtype(SHAP_XNUM[c]):
                SHAP_XNUM[c] = SHAP_XNUM[c].astype("category").cat.codes.replace(-1, np.nan)
        mean_abs = np.abs(SHAP_VALUES).mean(0)
        rows = []
        for j, c in enumerate(SHAP_X.columns):
            r = {"feature": SHAP_NAMES[j], "feature_model": c, "mean_abs_shap": mean_abs[j]}
            col = SHAP_X[c]
            if pd.api.types.is_numeric_dtype(col) and col.nunique() > 2:
                rho = pd.Series(col.values).corr(pd.Series(SHAP_VALUES[:, j]), method="spearman")
                r["direction"] = "higher value → higher churn risk" if rho > 0.1 else "higher value → lower churn risk" if rho < -0.1 else "non-monotonic / mixed"
                r["spearman_value_vs_shap"] = rho
            else:
                lv = pd.DataFrame({"lv": col.astype(str).values, "s": SHAP_VALUES[:, j]}).groupby("lv")["s"].agg(["mean", "count"])
                lv = lv[lv["count"] >= 20]
                if len(lv):
                    r["direction"] = f"riskiest: '{lv['mean'].idxmax()}' (+{lv['mean'].max():.2f}); safest: '{lv['mean'].idxmin()}' ({lv['mean'].min():.2f})"
            rows.append(r)
        SHAP_IMP = pd.DataFrame(rows).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
        SHAP_IMP["share_%"] = 100 * SHAP_IMP["mean_abs_shap"] / SHAP_IMP["mean_abs_shap"].sum()
        save_table(SHAP_IMP, "shap_importance")
        display(SHAP_IMP.head(20))

        top = SHAP_IMP.head(CFG.TOP_N_FEATURES_PLOT)
        fig, ax = plt.subplots(figsize=(8, max(3, 0.32 * len(top))))
        ax.barh(top["feature"][::-1], top["mean_abs_shap"][::-1], color=CFG.PALETTE[4])
        ax.set_xlabel("Mean |SHAP value| (impact on churn log-odds)")
        ax.set_title("Global Feature Importance (SHAP)")
        save_fig("shap_importance_bar")

        order = [SHAP_X.columns.get_loc(c) for c in top["feature_model"]]
        if HAS_SHAP:
            try:
                plt.figure()
                shap.summary_plot(SHAP_VALUES[:, order], SHAP_XNUM.iloc[:, order].astype(float), feature_names=[SHAP_NAMES[j] for j in order],
                                  show=False, max_display=len(order), plot_size=(9, max(4, 0.35 * len(order))))
                plt.title("SHAP Summary (beeswarm): impact of each feature on churn risk")
                save_fig("shap_beeswarm", plt.gcf())
            except Exception as e:
                print("shap.summary_plot gagal → manual:", e)
                HAS_SHAP_PLOT = False
        if not HAS_SHAP:
            fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(order))))
            for rank_i, j in enumerate(order[::-1]):
                v = SHAP_XNUM.iloc[:, j].astype(float)
                col = (v.rank(pct=True)).fillna(0.5).values
                ax.scatter(SHAP_VALUES[:, j], rank_i + np.random.uniform(-0.3, 0.3, len(v)), c=col, cmap="coolwarm", s=4, alpha=0.6)
            ax.set_yticks(range(len(order)))
            ax.set_yticklabels([SHAP_NAMES[j] for j in order[::-1]], fontsize=8)
            ax.axvline(0, color="k", lw=0.8)
            ax.set_xlabel("SHAP value (impact on churn log-odds); color = feature value (red high / blue low)")
            ax.set_title("SHAP Summary (beeswarm)")
            save_fig("shap_beeswarm")

In [ ]:
with safe_section("SHAP dependence & categorical effects"):
    if SHAP_VALUES is not None:
        num_top = [c for c in SHAP_IMP["feature_model"] if pd.api.types.is_numeric_dtype(SHAP_X[c]) and SHAP_X[c].nunique() > 5][:6]
        if num_top:
            fig, axes = grid_axes(len(num_top), ncols=3, w=5, h=3.6)
            for a, c in zip(axes, num_top):
                j = SHAP_X.columns.get_loc(c)
                xv = SHAP_X[c].astype(float)
                lo, hi = xv.quantile([0.01, 0.99])
                m = xv.between(lo, hi)
                a.scatter(xv[m], SHAP_VALUES[m.values, j], s=5, alpha=0.35, color=CFG.PALETTE[0])
                try:
                    bins = pd.qcut(xv[m], 15, duplicates="drop")
                    mm = pd.DataFrame({"b": bins, "x": xv[m], "s": SHAP_VALUES[m.values, j]}).groupby("b", observed=True).mean()
                    a.plot(mm["x"], mm["s"], color=CFG.PALETTE[1], lw=2)
                except Exception:
                    pass
                a.axhline(0, color="k", lw=0.7)
                a.set_title(display_name(c)[:40], fontsize=9)
                a.set_ylabel("SHAP (churn log-odds)", fontsize=8)
            fig.suptitle("SHAP Dependence: how each driver changes churn risk", y=1.02)
            plt.tight_layout()
            save_fig("shap_dependence")
        cat_top = [c for c in SHAP_IMP["feature_model"] if not pd.api.types.is_numeric_dtype(SHAP_X[c]) or SHAP_X[c].nunique() <= 5][:4]
        if cat_top:
            fig, axes = grid_axes(len(cat_top), ncols=2, w=6, h=3.4)
            for a, c in zip(axes, cat_top):
                j = SHAP_X.columns.get_loc(c)
                d = pd.DataFrame({"lv": SHAP_X[c].astype(str).values, "s": SHAP_VALUES[:, j]}).groupby("lv")["s"].agg(["mean", "count"])
                d = d[d["count"] >= 10].sort_values("mean")
                a.barh(d.index.astype(str).str.slice(0, 25), d["mean"], color=[CFG.PALETTE[1] if v > 0 else CFG.PALETTE[0] for v in d["mean"]])
                a.axvline(0, color="k", lw=0.7)
                a.set_title(display_name(c)[:40], fontsize=9)
                a.set_xlabel("Mean SHAP (churn log-odds)", fontsize=8)
            fig.suptitle("Effect of Each Category Level on Churn Risk (SHAP)", y=1.02)
            plt.tight_layout()
            save_fig("shap_categorical_levels")

In [ ]:
def plot_waterfall(i, title):
    j_sorted = np.argsort(-np.abs(SHAP_VALUES[i]))[:10]
    contrib = SHAP_VALUES[i, j_sorted]
    labels = [f"{SHAP_NAMES[j][:28]} = {str(SHAP_X.iloc[i, j])[:14]}" for j in j_sorted]
    rest = SHAP_VALUES[i].sum() - contrib.sum()
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.barh(labels[::-1] + ["(all other features)"][:0], contrib[::-1], color=[CFG.PALETTE[1] if v > 0 else CFG.PALETTE[0] for v in contrib[::-1]])
    ax.axvline(0, color="k", lw=0.8)
    logodds = SHAP_BASE + SHAP_VALUES[i].sum()
    ax.set_xlabel(f"SHAP contribution (log-odds). Base {SHAP_BASE:.2f} + others {rest:+.2f} → p(churn) = {1 / (1 + np.exp(-logodds)):.1%}")
    ax.set_title(title)
    return fig


with safe_section("Individual explanations (waterfall)"):
    if SHAP_VALUES is not None:
        p_s = FINAL_OOF[SHAP_IDX]
        i_high = int(np.argmax(p_s))
        churners = np.where(y[SHAP_IDX] == 1)[0]
        i_typ = int(churners[np.argsort(p_s[churners])[len(churners) // 2]]) if len(churners) else i_high
        save_fig("shap_waterfall_highest_risk", plot_waterfall(i_high, f"Why is this customer high-risk? (predicted {p_s[i_high]:.0%})"))
        save_fig("shap_waterfall_typical_churner", plot_waterfall(i_typ, f"Typical churner explanation (predicted {p_s[i_typ]:.0%})"))

In [ ]:
def partial_dependence(md, col, grid, n=800):
    X = md["X_va"].iloc[:n].copy()
    out = []
    for g in grid:
        X[col] = g
        out.append(md["model"].predict_proba(X)[:, 1].mean())
    return np.array(out)


with safe_section("Partial dependence (PDP)"):
    md0 = RESULTS[EXPLAIN_KEY]["models"][0]
    rank_src = SHAP_IMP if len(SHAP_IMP) else PERM_IMP
    pd_feats = [c for c in rank_src["feature_model"] if c in md0["X_va"].columns and pd.api.types.is_numeric_dtype(md0["X_va"][c])
                and md0["X_va"][c].nunique() > 5][:6]
    if pd_feats:
        fig, axes = grid_axes(len(pd_feats), ncols=3, w=5, h=3.5)
        PDP_TABLES = {}
        for a, c in zip(axes, pd_feats):
            xv = md0["X_va"][c].dropna()
            grid = np.unique(np.quantile(xv, np.linspace(0.05, 0.95, 15)))
            pdv = partial_dependence(md0, c, grid)
            PDP_TABLES[c] = pd.DataFrame({"value": grid, "avg_predicted_churn": pdv})
            a.plot(grid, pdv, "o-", color=CFG.PALETTE[1])
            a.set_title(display_name(c)[:40], fontsize=9)
            a.set_ylabel("Avg predicted churn", fontsize=8)
            a.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
            a2 = a.twinx()
            a2.hist(xv.clip(grid.min(), grid.max()), bins=30, alpha=0.15, color="gray")
            a2.set_yticks([])
        fig.suptitle("Partial Dependence: predicted churn as each driver changes (others held at observed values)", y=1.02)
        plt.tight_layout()
        save_fig("partial_dependence")
        for c, t in PDP_TABLES.items():
            lo_, hi_ = t.iloc[0], t.iloc[-1]
            add_insight("Explainability", f"Partial dependence: moving {display_name(c)} from {fmt_num(lo_['value'])} to {fmt_num(hi_['value'])} changes average "
                                          f"predicted churn from {pct(lo_['avg_predicted_churn'])} to {pct(hi_['avg_predicted_churn'])}.", show=False)

### 20.4 Driver heterogeneity — apakah driver berbeda antar segmen?
Driver churn bisa beda per **jenis produk / channel / region**. Contoh: di Health insurance driver utama = klaim ditolak; di Auto = kenaikan premi.
Dihitung dari rata-rata |SHAP| per fitur di tiap level segmen (dinormalisasi per segmen → share %).

**Untuk laporan:** insight ini membuat rekomendasi lebih tajam ("untuk produk X fokus ke ..., untuk channel Y fokus ke ..."). Set segmen manual: `HETERO_SEGMENT = "policy_type"`.

In [ ]:
HETERO_SEGMENT = None   # None = auto (policy_type → channel → kategori pertama dengan 2–8 level)
with safe_section("Driver heterogeneity by segment"):
    if SHAP_VALUES is None:
        print("⏭️ SHAP tidak tersedia → skip.")
    else:
        seg = HETERO_SEGMENT
        if seg is None:
            for cand_seg in [ROLES.get("policy_type"), ROLES.get("channel"), ROLES.get("region")] + CAT_COLS:
                if cand_seg and cand_seg in train.columns and 2 <= train[cand_seg].nunique() <= 8 and cand_seg not in LEAKY_COLS:
                    seg = cand_seg
                    break
        if seg is None:
            print("Tidak ada kolom segmen yang cocok.")
        else:
            base_names = [n.replace("__freq", "").replace(" (target-enc)", "") for n in SHAP_NAMES]
            absS = pd.DataFrame(np.abs(SHAP_VALUES), columns=SHAP_NAMES)
            absS = absS.T.groupby(base_names).sum().T
            lev = train[seg].astype(str).iloc[SHAP_IDX].values
            topf = absS.mean().sort_values(ascending=False).head(10).index
            H = absS.assign(_seg=lev).groupby("_seg")[list(topf)].mean()
            cnt = pd.Series(lev).value_counts()
            H = H.loc[[l for l in H.index if cnt.get(l, 0) >= 50]]
            Hs = 100 * H.div(absS.assign(_seg=lev).groupby("_seg").mean().loc[H.index].sum(axis=1), axis=0)
            HETERO_TABLE = Hs.T
            save_table(HETERO_TABLE.reset_index().rename(columns={"index": "feature"}), f"driver_heterogeneity_by_{sanitize_name(seg)}")
            fig, ax = plt.subplots(figsize=(1.4 * Hs.shape[0] + 4, 0.45 * len(topf) + 1.5))
            sns.heatmap(HETERO_TABLE, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax, cbar_kws={"label": "share of SHAP importance (%)"})
            ax.set_title(f"Churn drivers by {seg} (share of total SHAP importance within each segment, %)")
            save_fig(f"driver_heterogeneity_{seg}")
            for lv in HETERO_TABLE.columns:
                t3 = HETERO_TABLE[lv].sort_values(ascending=False).head(3)
                add_insight("Explainability", f"For {seg} = '{lv}', the top churn drivers are " + ", ".join(f"{k} ({v:.0f}%)" for k, v in t3.items()) + ".", show=False)
            print(f"Segmen: {seg}")
            display(HETERO_TABLE.round(1))

## 20.5 Consolidated Driver Evidence Matrix
Satu tabel yang menggabungkan **semua bukti** per driver: univariate (IV, churn rate level berisiko), multivariate (odds ratio, hazard ratio), dan ML (rank SHAP/permutation), plus apakah fitur **actionable** (bisa diintervensi perusahaan).
**Ini tabel paling kuat untuk laporan** — driver yang konsisten muncul di semua metode = temuan yang robust.

In [ ]:
ACTIONABLE_KEYWORDS = ["payment", "bayar", "renew", "debit", "complain", "keluhan", "claim", "klaim", "settle", "premium", "premi", "price",
                       "discount", "diskon", "channel", "agent", "agen", "contract", "frequency", "paperless", "engagement", "digital",
                       "login", "app", "service", "support", "late", "telat", "product", "rider", "coverage", "plan", "tech", "online", "call"]


def is_actionable(f):
    if CFG.ACTIONABLE_FEATURES is not None:
        return f in CFG.ACTIONABLE_FEATURES or any(f.startswith(a) for a in CFG.ACTIONABLE_FEATURES)
    fl = f.lower()
    if any(k in fl for k in ["age", "usia", "umur", "gender", "sex", "region", "province", "city", "marital", "dependent", "income", "tenure", "days_since", "_year"]):
        return False
    return any(k in fl for k in ACTIONABLE_KEYWORDS)


EVIDENCE = pd.DataFrame()
with safe_section("Consolidated driver evidence"):
    ev = DRIVER_TABLE[["feature", "iv", "iv_strength", "effect_size", "effect_label", "q_value_BH", "highest_risk_level",
                       "highest_risk_rate", "highest_risk_lift", "direction"]].copy()
    ev = ev.rename(columns={"direction": "univariate_direction"})
    if len(SHAP_IMP):
        sh = SHAP_IMP.copy()
        sh["base_feature"] = sh["feature"].str.replace(" (target-enc)", "", regex=False).str.replace("__freq", "", regex=False)
        sh = sh.groupby("base_feature").agg(shap_share_pct=("share_%", "sum"), shap_direction=("direction", "first")).reset_index()
        sh["shap_rank"] = sh["shap_share_pct"].rank(ascending=False).astype(int)
        ev = ev.merge(sh.rename(columns={"base_feature": "feature"}), on="feature", how="left")
    if len(PERM_IMP):
        pi = PERM_IMP.copy()
        pi["base_feature"] = pi["feature"].astype(str).str.replace("__freq", "", regex=False)
        pi = pi.groupby("base_feature")["importance_mean"].sum().rename("perm_importance").reset_index()
        ev = ev.merge(pi.rename(columns={"base_feature": "feature"}), on="feature", how="left")
    if len(LOGIT_TABLE):
        lt = LOGIT_TABLE.copy()
        lt["feature"] = lt["term"].str.split(" = ").str[0]
        lt = lt.sort_values("p_value").groupby("feature").first().reset_index()[["feature", "odds_ratio", "p_value"]]
        ev = ev.merge(lt.rename(columns={"p_value": "or_p_value"}), on="feature", how="left")
    if len(COX_TABLE):
        ct = COX_TABLE.copy()
        ct["feature"] = ct["term"].str.split(" = ").str[0]
        ct = ct.sort_values("p_value").groupby("feature").first().reset_index()[["feature", "hazard_ratio", "p_value"]]
        ev = ev.merge(ct.rename(columns={"p_value": "hr_p_value"}), on="feature", how="left")
    ev["actionable"] = ev["feature"].apply(is_actionable)
    ev = ev[~ev["feature"].isin(LEAKY_COLS)]
    n_methods = (ev["q_value_BH"] < 0.05).astype(int)
    if "shap_rank" in ev:
        n_methods += (ev["shap_rank"] <= 10).astype(int)
    if "or_p_value" in ev:
        n_methods += (ev["or_p_value"] < 0.05).astype(int)
    if "hr_p_value" in ev:
        n_methods += (ev["hr_p_value"] < 0.05).astype(int)
    ev["evidence_score"] = n_methods
    sort_cols = ["evidence_score"] + (["shap_share_pct"] if "shap_share_pct" in ev else ["iv"])
    EVIDENCE = ev.sort_values(sort_cols, ascending=False).reset_index(drop=True)
    save_table(EVIDENCE, "driver_evidence_matrix")
    display(EVIDENCE.head(20))
    top_act = EVIDENCE[EVIDENCE["actionable"]].head(5)["feature"].tolist()
    top_non = EVIDENCE[~EVIDENCE["actionable"]].head(3)["feature"].tolist()
    add_insight("Drivers", f"Most robust actionable churn drivers (consistent across statistical tests, regression and SHAP): {', '.join(top_act)}.")
    if top_non:
        add_insight("Drivers", f"Key non-actionable risk indicators useful for targeting: {', '.join(top_non)}.")

# 21. Business Analytics — dari prediksi ke keputusan

Juri lomba bisnis menilai: **"so what?"** Section ini menerjemahkan model ke angka bisnis.

| Analisis | Pertanyaan bisnis | Output |
|---|---|---|
| **Lift / Gains / KS** | Seberapa efisien model menemukan churner? | "Top 20% skor menangkap 55% churner (lift 2.7x)" |
| **Risk tiers** | Berapa customer berisiko tinggi/sedang/rendah? | tabel tier + churn rate aktual |
| **CLV & value at risk** | Berapa uang yang terancam hilang? | total premi/margin berisiko |
| **Risk × Value matrix** | Siapa diprioritaskan? | 4 kuadran: Priority Save, Low-cost Save, Nurture & Upsell, Maintain |
| **Campaign ROI simulation** | Berapa customer yang optimal di-target? | kurva profit vs % target, ROI, titik optimal |
| **What-if scenarios** | Kalau perusahaan mengubah X, berapa churn turun? | estimasi churn reduction per intervensi |
| **Churn personas (SHAP clustering)** | Apa *alasan* tiap kelompok berisiko? | 3–5 persona dengan driver utama |

Parameter bisnis di `CFG`: `PROFIT_MARGIN`, `RETENTION_COST`, `RETENTION_SUCCESS_RATE`, `DISCOUNT_RATE`, `CLV_HORIZON_YEARS`. **Kalau soal tidak memberi angka, pakai asumsi wajar & tulis jelas sebagai asumsi di laporan** (+ sensitivity analysis di 21.5).

**Asumsi penting (tulis di laporan):** probabilitas churn model diperlakukan sebagai peluang churn dalam satu periode (≈ 1 tahun polis); what-if bersifat **asosiatif, bukan kausal** — rekomendasikan validasi dengan **A/B test / pilot**.

In [ ]:
PROB_BIZ = FINAL_OOF_CAL
PROB_BIZ_TEST = FINAL_TEST_CAL if len(FINAL_TEST) else np.array([])

with safe_section("Lift, gains & KS"):
    d = pd.DataFrame({"p": PROB_BIZ, "y": y})
    d["decile"] = pd.qcut(d["p"].rank(method="first", ascending=False), 10, labels=[f"D{i}" for i in range(1, 11)])
    dec = d.groupby("decile", observed=True).agg(n=("y", "size"), churners=("y", "sum"), avg_pred=("p", "mean"), min_pred=("p", "min")).reset_index()
    dec["churn_rate"] = dec["churners"] / dec["n"]
    dec["lift"] = dec["churn_rate"] / CHURN_RATE
    dec["cum_customers_%"] = 100 * dec["n"].cumsum() / dec["n"].sum()
    dec["cum_churners_captured_%"] = 100 * dec["churners"].cumsum() / dec["churners"].sum()
    dec["cum_lift"] = dec["cum_churners_captured_%"] / dec["cum_customers_%"]
    DECILE_TABLE = dec
    save_table(dec, "decile_lift_table")
    display(dec)
    fpr, tpr, _ = roc_curve(y, PROB_BIZ)
    KS = float(np.max(tpr - fpr))
    print(f"KS statistic = {KS:.3f} (>0.4 sangat baik, 0.3–0.4 baik, 0.2–0.3 cukup)")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
    xs = np.r_[0, dec["cum_customers_%"]]
    ax[0].plot(xs, np.r_[0, dec["cum_churners_captured_%"]], "o-", color=CFG.PALETTE[1], label="Model")
    ax[0].plot([0, 100], [0, 100], "k--", label="Random")
    ax[0].plot([0, 100 * CHURN_RATE, 100], [0, 100, 100], ":", color="gray", label="Perfect")
    ax[0].set_xlabel("% of customers targeted (highest risk first)")
    ax[0].set_ylabel("% of churners captured")
    ax[0].set_title("Cumulative Gains Chart")
    ax[0].legend()
    ax[1].bar(dec["decile"].astype(str), dec["lift"], color=CFG.PALETTE[0])
    ax[1].axhline(1, color="red", ls="--")
    ax[1].set_xlabel("Risk decile (D1 = highest predicted risk)")
    ax[1].set_ylabel("Lift vs average churn rate")
    ax[1].set_title("Lift by Decile")
    save_fig("gains_and_lift")
    top2 = dec.iloc[:2]
    add_insight("Business", f"Targeting the top 20% highest-risk customers captures {top2['cum_churners_captured_%'].iloc[-1]:.1f}% of all churners "
                            f"({top2['cum_lift'].iloc[-1]:.1f}x lift over random targeting); the top decile churns at {pct(dec['churn_rate'].iloc[0])} "
                            f"vs {pct(dec['churn_rate'].iloc[-1])} in the bottom decile. KS = {KS:.2f}.")

In [ ]:
def tier_of(p, cuts):
    return np.where(p >= cuts[1], "High", np.where(p >= cuts[0], "Medium", "Low"))


with safe_section("Risk tiers"):
    TIER_CUTS = tuple(np.quantile(PROB_BIZ, q) for q in CFG.RISK_TIER_CUTS)
    tr_tier = tier_of(PROB_BIZ, TIER_CUTS)
    tt = pd.DataFrame({"tier": tr_tier, "y": y, "p": PROB_BIZ}).groupby("tier").agg(
        n_train=("y", "size"), actual_churn_rate=("y", "mean"), avg_predicted=("p", "mean")).reindex(["High", "Medium", "Low"])
    tt["share_churners_%"] = 100 * pd.DataFrame({"tier": tr_tier, "y": y}).groupby("tier")["y"].sum().reindex(tt.index) / y.sum()
    if len(PROB_BIZ_TEST):
        TEST_TIER = tier_of(PROB_BIZ_TEST, TIER_CUTS)
        tt["n_test"] = pd.Series(TEST_TIER).value_counts().reindex(tt.index).fillna(0).astype(int).values
    TIER_TABLE = tt.reset_index()
    print(f"Batas tier: Medium ≥ {TIER_CUTS[0]:.3f}, High ≥ {TIER_CUTS[1]:.3f}")
    display(TIER_TABLE)
    save_table(TIER_TABLE, "risk_tiers")
    h = TIER_TABLE.set_index("tier").loc["High"]
    add_insight("Business", f"The High-risk tier ({pct(1 - CFG.RISK_TIER_CUTS[1], 0)} of customers, p ≥ {TIER_CUTS[1]:.2f}) has an actual churn rate of "
                            f"{pct(h['actual_churn_rate'])} and contains {h['share_churners_%']:.1f}% of all churners.")

In [ ]:
VALUE_SOURCE = None
with safe_section("Customer value & CLV"):
    vcol = CFG.VALUE_COL or ROLES.get("premium")
    if vcol and vcol in train.columns and pd.api.types.is_numeric_dtype(train[vcol]):
        VALUE_SOURCE = vcol
        mult = 12 if CFG.VALUE_PERIOD == "monthly" else 1
        val_tr = train[vcol].fillna(train[vcol].median()).clip(lower=0).values * mult
        val_te = test[vcol].fillna(train[vcol].median()).clip(lower=0).values * mult if test is not None and vcol in test else np.array([])
        unit = CFG.CURRENCY
    else:
        print("⚠️ Tidak ada kolom nilai (premi) → nilai customer = 1 (analisis berbasis jumlah customer). Set CFG.VALUE_COL kalau ada.")
        val_tr, val_te, unit = np.ones(N_TR), np.ones(len(PROB_BIZ_TEST)), "customers"
    H, dr = CFG.CLV_HORIZON_YEARS, CFG.DISCOUNT_RATE

    def clv(margin, p):
        r = (1 - np.clip(p, 0, 1)) / (1 + dr)
        return margin * (1 - r ** H) / np.maximum(1 - r, 1e-9)

    MARGIN_TR = val_tr * (CFG.PROFIT_MARGIN if unit != "customers" else 1)
    CLV_TR = clv(MARGIN_TR, PROB_BIZ)
    VALUE_SAVED_TR = clv(MARGIN_TR, np.full(N_TR, CHURN_RATE))   # nilai customer kalau berhasil dipertahankan (risiko rata-rata)
    if len(PROB_BIZ_TEST):
        MARGIN_TE = val_te * (CFG.PROFIT_MARGIN if unit != "customers" else 1)
        CLV_TE = clv(MARGIN_TE, PROB_BIZ_TEST)
        VALUE_SAVED_TE = clv(MARGIN_TE, np.full(len(MARGIN_TE), CHURN_RATE))
    at_risk_tr = (PROB_BIZ * val_tr).sum()
    print(f"Nilai tahunan total (train): {fmt_num(val_tr.sum())} {unit} | expected annual value at risk: {fmt_num(at_risk_tr)} {unit} ({pct(at_risk_tr / val_tr.sum())})")
    if len(PROB_BIZ_TEST):
        at_risk_te = (PROB_BIZ_TEST * val_te).sum()
        print(f"Test/portfolio aktif: expected annual value at risk = {fmt_num(at_risk_te)} {unit} dari {fmt_num(val_te.sum())}")
    print(f"CLV rata-rata (horizon {H} th, discount {dr:.0%}): {fmt_num(CLV_TR.mean())} {unit}")
    if unit != "customers":
        add_insight("Business", f"Expected annual premium at risk from churn is {fmt_num(at_risk_tr)} {unit} ({pct(at_risk_tr / val_tr.sum())} of the annual premium base); "
                                f"the High-risk tier alone accounts for {pct((PROB_BIZ * val_tr)[tr_tier == 'High'].sum() / at_risk_tr)} of it.")

In [ ]:
with safe_section("Risk × value matrix"):
    v_cut = np.median(val_tr)
    r_cut = TIER_CUTS[1]

    def quadrant(p, v):
        return np.where(p >= r_cut, np.where(v >= v_cut, "Priority Save (high risk, high value)", "Low-cost Save (high risk, low value)"),
                        np.where(v >= v_cut, "Nurture & Upsell (low risk, high value)", "Maintain (low risk, low value)"))

    q_tr = quadrant(PROB_BIZ, val_tr)
    qt = pd.DataFrame({"quadrant": q_tr, "y": y, "p": PROB_BIZ, "v": val_tr, "risk_value": PROB_BIZ * val_tr}).groupby("quadrant").agg(
        n=("y", "size"), actual_churn_rate=("y", "mean"), avg_pred=("p", "mean"), total_value=("v", "sum"), value_at_risk=("risk_value", "sum"))
    qt["share_value_at_risk_%"] = 100 * qt["value_at_risk"] / qt["value_at_risk"].sum()
    QUAD_TABLE = qt.reset_index()
    display(QUAD_TABLE)
    save_table(QUAD_TABLE, "risk_value_quadrants")
    fig, ax = plt.subplots(figsize=(8, 5.5))
    sel = np.random.RandomState(0).choice(N_TR, min(N_TR, 4000), replace=False)
    colors = {"Priority Save (high risk, high value)": CFG.PALETTE[1], "Low-cost Save (high risk, low value)": CFG.PALETTE[3],
              "Nurture & Upsell (low risk, high value)": CFG.PALETTE[2], "Maintain (low risk, low value)": "#9E9E9E"}
    for qn, c in colors.items():
        m = q_tr[sel] == qn
        ax.scatter(PROB_BIZ[sel][m], val_tr[sel][m], s=7, alpha=0.5, color=c, label=qn)
    ax.axvline(r_cut, color="k", ls="--", lw=0.8)
    ax.axhline(v_cut, color="k", ls="--", lw=0.8)
    if unit != "customers" and val_tr.min() > 0:
        ax.set_yscale("log")
    ax.set_xlabel("Predicted churn probability")
    ax.set_ylabel(f"Customer annual value ({unit})")
    if not (unit != "customers" and val_tr.min() > 0):
        money_axis(ax)
    ax.set_title("Risk × Value Prioritisation Matrix")
    ax.legend(fontsize=7, loc="upper right")
    save_fig("risk_value_matrix")
    ps = QUAD_TABLE.set_index("quadrant").loc["Priority Save (high risk, high value)"] if "Priority Save (high risk, high value)" in set(q_tr) else None
    if ps is not None:
        add_insight("Business", f"'Priority Save' customers (high risk & above-median value) are {pct(ps['n'] / N_TR)} of the base but hold "
                                f"{ps['share_value_at_risk_%']:.1f}% of the value at risk — the first target for personalised retention.")

## 21.5 Campaign ROI Simulation
Simulasi: perusahaan men-target top-k% customer (urut risiko). Untuk tiap k:
- **Benefit** = churner aktual yang ditarget × `RETENTION_SUCCESS_RATE` × nilai customer yang diselamatkan (CLV)
- **Cost** = jumlah ditarget × `RETENTION_COST`
- **Net profit** = benefit − cost; ROI = net / cost

Dibandingkan 3 strategi: **model (urut probabilitas)**, **model × value (urut expected loss)**, dan **random**. Dihitung di OOF (ada label aktual) → hasil jujur.
**Sensitivity analysis**: profit optimal dihitung ulang untuk beberapa success rate & cost → tunjukkan kesimpulan robust terhadap asumsi.

**Jika X → Y:** net profit negatif di semua k → biaya retensi terlalu mahal relatif nilai customer → rekomendasikan intervensi murah (digital/otomatis) untuk tier low-value, intervensi mahal hanya untuk Priority Save.

In [ ]:
def campaign_curve(order, success, cost, steps=50):
    rows = []
    n = len(order)
    for k in np.linspace(0.02, 1.0, steps):
        idx = order[: max(1, int(k * n))]
        benefit = (y[idx] * success * VALUE_SAVED_TR[idx]).sum()
        c = cost * len(idx)
        rows.append({"target_%": 100 * k, "n_targeted": len(idx), "churners_reached": int(y[idx].sum()),
                     "benefit": benefit, "cost": c, "net_profit": benefit - c, "roi": (benefit - c) / c if c else np.nan})
    return pd.DataFrame(rows)


with safe_section("Campaign ROI simulation"):
    cost = CFG.RETENTION_COST if unit != "customers" else 0.0
    succ = CFG.RETENTION_SUCCESS_RATE
    if unit == "customers":
        print("ℹ️ Tanpa kolom nilai → profit dihitung dalam 'customers saved' (cost diabaikan).")
    strategies = {"Model: churn probability": np.argsort(-PROB_BIZ),
                  "Model: expected loss (p × value)": np.argsort(-(PROB_BIZ * VALUE_SAVED_TR)),
                  "Random targeting": np.random.RandomState(CFG.SEED).permutation(N_TR)}
    curves = {k: campaign_curve(o, succ, cost) for k, o in strategies.items()}
    fig, ax = plt.subplots(figsize=(8, 4.5))
    best_rows = []
    for (k, cdf), col in zip(curves.items(), [CFG.PALETTE[1], CFG.PALETTE[4], "gray"]):
        ax.plot(cdf["target_%"], cdf["net_profit"], color=col, label=k, lw=2 if "Model" in k else 1)
        b = cdf.loc[cdf["net_profit"].idxmax()]
        best_rows.append({"strategy": k, **b.to_dict()})
        if "Model" in k:
            ax.scatter([b["target_%"]], [b["net_profit"]], color=col, zorder=3)
    ax.axhline(0, color="k", lw=0.7)
    ax.set_xlabel("% of customers targeted")
    ax.set_ylabel(f"Net profit ({unit})")
    money_axis(ax)
    ax.set_title(f"Retention Campaign Profit Curve (success rate {succ:.0%}, cost {fmt_num(cost)}/customer)")
    ax.legend(fontsize=8)
    save_fig("campaign_profit_curve")
    CAMPAIGN_BEST = pd.DataFrame(best_rows)
    display(CAMPAIGN_BEST)
    save_table(CAMPAIGN_BEST, "campaign_optimum")
    save_table(curves["Model: expected loss (p × value)"], "campaign_curve_expected_loss")
    bm = CAMPAIGN_BEST.iloc[1] if CAMPAIGN_BEST.iloc[1]["net_profit"] >= CAMPAIGN_BEST.iloc[0]["net_profit"] else CAMPAIGN_BEST.iloc[0]
    rnd = curves["Random targeting"]
    rnd_same = rnd.iloc[(rnd["target_%"] - bm["target_%"]).abs().idxmin()]
    add_insight("Business", f"Simulated retention campaign (assumed success rate {succ:.0%}, cost {fmt_num(cost)} {unit if unit != 'customers' else ''} per contact): "
                            f"targeting the top {bm['target_%']:.0f}% ranked by '{bm['strategy']}' maximises net profit at {fmt_num(bm['net_profit'])} "
                            f"(ROI {bm['roi']:.1f}x), versus {fmt_num(rnd_same['net_profit'])} for random targeting of the same size.")

    # sensitivity
    sens = []
    for s_ in [succ * 0.5, succ, min(1, succ * 1.5)]:
        for c_ in [cost * 0.5, cost, cost * 2]:
            cdf = campaign_curve(strategies["Model: expected loss (p × value)"], s_, c_, steps=25)
            b = cdf.loc[cdf["net_profit"].idxmax()]
            sens.append({"success_rate": s_, "cost_per_contact": c_, "optimal_target_%": b["target_%"], "max_net_profit": b["net_profit"], "roi": b["roi"]})
    SENS = pd.DataFrame(sens)
    print("Sensitivity analysis (asumsi success rate × cost):")
    display(SENS)
    save_table(SENS, "campaign_sensitivity")
    # business threshold per customer
    if unit != "customers" and len(PROB_BIZ_TEST):
        be = cost / (succ * np.maximum(VALUE_SAVED_TE, 1e-9))   # target kalau p > break-even prob
        TARGET_FLAG_TEST = PROB_BIZ_TEST > be
        print(f"Rule bisnis per customer: target jika p(churn) > cost / (success × value_saved). → {TARGET_FLAG_TEST.sum():,} dari {len(be):,} customer test layak ditarget.")

## 21.6 What-if Scenarios (simulasi intervensi)
Mengubah nilai fitur **actionable** untuk customer yang terdampak lalu memprediksi ulang:
- Kategori → pindahkan customer dari level berisiko ke level paling aman (misal *payment = Cash → Auto-debit*).
- Count (keluhan, telat bayar) → jadikan 0 (masalah diselesaikan).
- Kontinu yang menaikkan risiko (misal kenaikan premi) → dibatasi di median.

Hasil ditampilkan pada **adopsi 100%** (batas atas) dan **adopsi realistis** `CFG.WHAT_IF_ADOPTION` (default 30%).
Skenario custom: `CFG.WHAT_IF_SCENARIOS = [{"name": "...", "feature": "payment_method", "set_to": "Auto-debit"}, {"name": "...", "feature": "premium_change_pct", "cap": 5}]`.

⚠️ **Ini simulasi model, bukan bukti kausal.** Di laporan tulis: *"model-based simulation suggests up to X pp reduction; we recommend validating via a controlled pilot (A/B test)."*

In [ ]:
WHATIF = pd.DataFrame()


def auto_scenarios(max_n=6):
    sc = []
    feats = EVIDENCE[EVIDENCE["actionable"]]["feature"].tolist() if len(EVIDENCE) else []
    for f in feats:
        if f not in NAME_MAP or f in LEAKY_COLS:
            continue
        s = train[f]
        tbl = LEVEL_TABLES.get(f)
        if not pd.api.types.is_numeric_dtype(s):
            if tbl is None:
                continue
            ok = tbl[(tbl["n"] >= max(30, 0.02 * N_TR)) & (tbl["level"] != "Missing") & (tbl["level"] != "Other")]
            if len(ok) < 2:
                continue
            safest = ok.sort_values("churn_rate").iloc[0]["level"]
            sc.append({"name": f"Move all '{f}' customers to '{safest}'", "feature": f, "set_to": safest})
        elif s.nunique() <= 2:
            r = pd.Series(y).groupby(s.values).mean()
            if len(r) == 2:
                sc.append({"name": f"Set {f} = {r.idxmin():g} for everyone", "feature": f, "set_to": r.idxmin()})
        else:
            higher_worse = (DRIVER_TABLE.set_index("feature").loc[f, "effect_size"] > 0) if f in set(DRIVER_TABLE["feature"]) else True
            is_count = (s.dropna() % 1 == 0).all() and (s == 0).mean() > 0.3 and s.min() >= 0
            if is_count and higher_worse:
                sc.append({"name": f"Resolve issues: {f} → 0", "feature": f, "set_to": 0})
            elif higher_worse:
                sc.append({"name": f"Cap {f} at median ({s.median():g})", "feature": f, "cap": float(s.median())})
            else:
                sc.append({"name": f"Raise {f} to at least median ({s.median():g})", "feature": f, "floor": float(s.median())})
        if len(sc) >= max_n:
            break
    return sc


with safe_section("What-if scenarios"):
    md0 = RESULTS[EXPLAIN_KEY]["models"][0]
    X0 = md0["X_va"].copy()
    base_p = md0["model"].predict_proba(X0)[:, 1]
    va0 = md0["va_idx"]
    scen = CFG.WHAT_IF_SCENARIOS or auto_scenarios()
    rows = []
    for sc in scen:
        f = sc["feature"]
        col = NAME_MAP.get(f, f)
        if col not in X0.columns:
            print(f"  skip '{sc['name']}': kolom tidak ada di model")
            continue
        Xs = X0.copy()
        orig = train[f].iloc[va0].reset_index(drop=True)
        if "set_to" in sc:
            target_val = sc["set_to"]
            if pd.api.types.is_numeric_dtype(train[f]):
                affected = (orig != target_val) & orig.notna()
                Xs.loc[affected.values, col] = target_val
            else:
                affected = (orig.astype(str) != str(target_val))
                if str(Xs[col].dtype) == "category":
                    if target_val not in list(Xs[col].cat.categories):
                        print(f"  skip '{sc['name']}': level tidak dikenal model")
                        continue
                    Xs.loc[affected.values, col] = target_val
                elif Xs[col].dtype == object:
                    Xs.loc[affected.values, col] = str(target_val)
                else:  # ordinal code
                    cats = list(X_tree_full[col].cat.categories)
                    Xs.loc[affected.values, col] = float(cats.index(target_val)) if target_val in cats else np.nan
                fcol = f"{col}__freq"
                if fcol in Xs.columns:
                    Xs.loc[affected.values, fcol] = float(full_df[col].value_counts(normalize=True).get(target_val, np.nan))
        elif "cap" in sc:
            affected = orig > sc["cap"]
            Xs.loc[affected.values, col] = sc["cap"]
        elif "floor" in sc:
            affected = orig < sc["floor"]
            Xs.loc[affected.values, col] = sc["floor"]
        else:
            continue
        new_p = md0["model"].predict_proba(Xs)[:, 1]
        a = affected.values
        if a.sum() == 0:
            continue
        delta_aff = (base_p[a] - new_p[a]).mean()
        overall_full = (base_p - new_p).mean()
        rows.append({"scenario": sc["name"], "feature": f, "affected_%": 100 * a.mean(),
                     "churn_affected_before": base_p[a].mean(), "churn_affected_after": new_p[a].mean(),
                     "overall_reduction_pp_full": 100 * overall_full,
                     f"overall_reduction_pp_{int(CFG.WHAT_IF_ADOPTION * 100)}pct_adoption": 100 * overall_full * CFG.WHAT_IF_ADOPTION,
                     "churners_avoided_per_1000": 1000 * overall_full * CFG.WHAT_IF_ADOPTION})
    WHATIF = pd.DataFrame(rows).sort_values("overall_reduction_pp_full", ascending=False).reset_index(drop=True)
    display(WHATIF)
    save_table(WHATIF, "what_if_scenarios")
    if len(WHATIF):
        fig, ax = plt.subplots(figsize=(8, max(3, 0.5 * len(WHATIF))))
        ax.barh(WHATIF["scenario"].str.slice(0, 55)[::-1], WHATIF["overall_reduction_pp_full"][::-1], color=CFG.PALETTE[2], label="100% adoption")
        ax.barh(WHATIF["scenario"].str.slice(0, 55)[::-1], WHATIF["overall_reduction_pp_full"][::-1] * CFG.WHAT_IF_ADOPTION, color=CFG.PALETTE[0],
                label=f"{CFG.WHAT_IF_ADOPTION:.0%} adoption")
        ax.set_xlabel("Reduction in overall churn rate (percentage points)")
        ax.set_title("What-if Simulation: Impact of Interventions on Churn")
        ax.legend(fontsize=8)
        save_fig("what_if_scenarios")
        for _, r in WHATIF.head(3).iterrows():
            add_insight("Business", f"What-if: '{r['scenario']}' affects {r['affected_%']:.0f}% of customers and lowers their predicted churn from "
                                    f"{pct(r['churn_affected_before'])} to {pct(r['churn_affected_after'])}; at {CFG.WHAT_IF_ADOPTION:.0%} adoption this avoids "
                                    f"≈{r['churners_avoided_per_1000']:.0f} churners per 1,000 customers (model-based, associational).")

## 21.7 Churn Personas (clustering SHAP values)
Customer berisiko tinggi tidak churn karena alasan yang sama. Dengan meng-cluster **SHAP values** (bukan fitur mentah) customer high-risk, kita dapat kelompok berdasarkan **alasan risikonya** → tiap persona dapat intervensi berbeda.
Contoh hasil: *Persona A "Price-shocked": didorong kenaikan premi; Persona B "Service-frustrated": keluhan & klaim ditolak; Persona C "New & disengaged": tenure pendek, engagement rendah.*

**Untuk laporan:** tabel persona (ukuran, churn rate, 3 driver utama) + nama persona buatan sendiri → sangat disukai juri karena langsung bisa jadi strategi.

In [ ]:
PERSONAS = pd.DataFrame()
with safe_section("Churn personas (SHAP clustering)"):
    if SHAP_VALUES is None:
        print("⏭️ SHAP tidak tersedia → persona di-skip.")
    else:
        p_s = PROB_BIZ[SHAP_IDX]
        hi = p_s >= TIER_CUTS[0]
        S = SHAP_VALUES[hi]
        if hi.sum() < 60:
            print("Terlalu sedikit customer berisiko di sampel SHAP.")
        else:
            best_k, best_sil = 3, -1
            for k in range(2, 6):
                lab = KMeans(n_clusters=k, n_init=10, random_state=CFG.SEED).fit_predict(S)
                sil = silhouette_score(S, lab, sample_size=min(2000, len(S)), random_state=CFG.SEED)
                if sil > best_sil:
                    best_k, best_sil = k, sil
            km = KMeans(n_clusters=best_k, n_init=20, random_state=CFG.SEED).fit(S)
            lab = km.labels_
            print(f"Jumlah persona optimal (silhouette): {best_k} (silhouette = {best_sil:.3f})")
            rows = []
            prof_cols = [f for f in EVIDENCE["feature"].head(8)] if len(EVIDENCE) else []
            Xh = SHAP_X[hi].reset_index(drop=True)
            for c in range(best_k):
                m = lab == c
                mean_shap = pd.Series(S[m].mean(0), index=[n.replace("__freq", "").replace(" (target-enc)", "") for n in SHAP_NAMES]).groupby(level=0).sum()
                drivers = mean_shap.sort_values(ascending=False).head(3)
                r = {"persona": f"P{c + 1}", "n_in_sample": int(m.sum()), "share_of_at_risk_%": 100 * m.mean(),
                     "avg_pred_churn": p_s[hi][m].mean(), "actual_churn_rate": y[SHAP_IDX][hi][m].mean(),
                     "top_drivers": "; ".join(f"{k} (+{v:.2f})" for k, v in drivers.items() if v > 0)}
                for f in prof_cols:
                    if f in train.columns:
                        vals = train[f].iloc[SHAP_IDX[hi][m]]
                        r[f] = round(vals.mean(), 2) if pd.api.types.is_numeric_dtype(vals) else (vals.mode().iloc[0] if vals.notna().any() else None)
                rows.append(r)
            PERSONAS = pd.DataFrame(rows).sort_values("share_of_at_risk_%", ascending=False)
            display(PERSONAS)
            save_table(PERSONAS, "churn_personas")
            topf = pd.Series(np.abs(S).mean(0), index=SHAP_NAMES).sort_values(ascending=False).head(10).index
            heat = pd.DataFrame({f"P{c + 1}": pd.Series(S[lab == c].mean(0), index=SHAP_NAMES)[topf] for c in range(best_k)})
            fig, ax = plt.subplots(figsize=(1.6 * best_k + 4, 0.45 * len(topf) + 1.5))
            sns.heatmap(heat, cmap="RdBu_r", center=0, annot=True, fmt=".2f", ax=ax, annot_kws={"size": 8})
            ax.set_title("Churn Personas: average SHAP contribution of top drivers\n(red = pushes churn risk up)")
            save_fig("churn_personas_heatmap")
            for _, r in PERSONAS.iterrows():
                add_insight("Personas", f"Persona {r['persona']} ({r['share_of_at_risk_%']:.0f}% of at-risk customers, actual churn {pct(r['actual_churn_rate'])}): "
                                        f"risk driven mainly by {r['top_drivers']}.")
            print("👉 Beri nama persona sendiri berdasarkan top_drivers (misal 'Price-shocked renewers', 'Service-frustrated claimants').")

# 22. Recommendations (otomatis, berbasis bukti)

Rekomendasi dibangkitkan dari **driver actionable** (evidence matrix) + library strategi retensi asuransi, lengkap dengan **bukti angka**, **target segmen**, dan **KPI**.
Ini draft — **edit & personalisasi** sesuai konteks soal sebelum masuk laporan.

### Struktur rekomendasi yang disukai juri
1. **Insight** (angka) → 2. **Aksi spesifik** (siapa, apa, kapan) → 3. **Target segmen & ukuran** → 4. **Dampak estimasi** (what-if / ROI) → 5. **KPI & cara ukur** (A/B test).

### Kerangka prioritas
- **Quick wins (0–3 bulan):** trigger otomatis (reminder renewal, auto-debit enrollment, follow-up keluhan).
- **Medium (3–6 bulan):** program onboarding, revisi kebijakan pricing renewal, SLA klaim.
- **Long term (6–12 bulan):** churn early-warning system (model ini di-deploy, skor bulanan), loyalty program, bundling produk.

In [ ]:
REC_LIBRARY = [
    (["payment_method", "paymentmethod", "metode_bayar", "payment_type", "payment_mode", "cara_bayar"],
     "Shift manual payers to automatic payment",
     "Offer a small incentive (e.g. premium discount/cashback) and one-click enrolment for auto-debit/recurring card payment; send pre-due reminders to remaining manual payers.",
     "% of policies on auto-pay; lapse rate of converted vs control"),
    (["auto_renew", "autorenew", "auto-renew", "paperless"],
     "Make renewal effortless (default auto-renewal)",
     "Default new and renewing policies to auto-renewal with clear opt-out; 30/14/7-day renewal reminders with a one-click renew link.",
     "Auto-renew adoption; renewal rate"),
    (["complain", "keluhan", "komplain", "ticket", "service_call", "support"],
     "Complaint recovery programme",
     "Trigger a proactive call within 48h of any complaint, enforce resolution SLAs, offer service-recovery gestures to high-value complainants, and fix top root causes.",
     "Complaint resolution time; churn rate of complainants"),
    (["reject", "tolak", "denied", "settle", "claim", "klaim"],
     "Improve the claims experience",
     "Fast-track settlement, explain rejections transparently with an appeal path, and assign a claims concierge for high-value / high-risk customers after a claim event.",
     "Claim settlement days; post-claim churn rate; claim NPS"),
    (["premium_change", "increase", "kenaikan", "price", "rate_change", "premium", "premi", "charges"],
     "Smarter renewal pricing",
     "Cap or phase in premium increases for at-risk loyal customers, explain the value behind increases, and offer coverage/deductible adjustments instead of lapse.",
     "Churn rate among customers with increases; price elasticity"),
    (["late", "telat", "overdue", "arrear", "missed", "tunggakan", "grace"],
     "Early-warning on payment difficulty",
     "Treat the first late payment as a churn signal: friendly reminders, flexible instalments or payment-date change, and grace-period outreach before lapse.",
     "Cure rate of late payers; lapse after first missed payment"),
    (["tenure", "new_customer", "lama", "masa"],
     "First-year onboarding programme",
     "Welcome call, policy walkthrough, app activation and 90-day check-in for new customers; first renewal treated as a critical moment.",
     "13th-month persistency ratio; first-year churn rate"),
    (["product", "produk", "rider", "multi_product", "bundle", "policies"],
     "Bundling & cross-sell to raise switching costs",
     "Offer multi-policy discounts and relevant riders to single-product customers with good risk profiles.",
     "Products per customer; churn of bundled vs single"),
    (["engagement", "login", "app", "digital", "usage", "activity", "isactivemember"],
     "Re-engage dormant customers",
     "Use app/wellness features, personalised content and periodic check-ins; trigger re-engagement when activity drops.",
     "Monthly active users; churn of re-engaged customers"),
    (["channel", "agent", "agen", "broker", "online", "bancassurance", "source"],
     "Channel-specific retention",
     "Assign human follow-up to channels with weak relationships (e.g. online-acquired customers) and tie agent incentives to persistency, not only new sales.",
     "Persistency by channel; churn by agent"),
    (["frequency", "frekuensi", "contract", "billing_cycle", "term"],
     "Longer payment / contract terms",
     "Encourage annual/quarterly payment or longer contracts with a discount; monthly payers get extra reminders.",
     "Share of annual payers; churn by payment frequency"),
]


def find_recommendation(feature):
    fl = feature.lower()
    for kws, title, action, kpi in REC_LIBRARY:
        if any(k in fl for k in kws):
            return title, action, kpi
    return (f"Investigate and act on '{feature}'",
            f"Customers with high-risk values of '{feature}' churn more; design a targeted intervention and test it in a pilot.",
            f"Churn rate of the '{feature}' high-risk group")


RECS = pd.DataFrame()
with safe_section("Recommendations"):
    rows, used_titles = [], set()
    cand = EVIDENCE[EVIDENCE["actionable"]] if len(EVIDENCE) else pd.DataFrame()
    for _, r in cand.iterrows():
        title, action, kpi = find_recommendation(r["feature"])
        if title in used_titles:
            continue
        used_titles.add(title)
        ev_txt = (f"'{r['highest_risk_level']}' churns at {pct(r['highest_risk_rate'])} ({r['highest_risk_lift']:.1f}x avg); "
                  f"IV {r['iv']:.2f}")
        if "odds_ratio" in r and pd.notna(r.get("odds_ratio")):
            ev_txt += f"; OR {r['odds_ratio']:.2f}"
        if "hazard_ratio" in r and pd.notna(r.get("hazard_ratio")):
            ev_txt += f"; HR {r['hazard_ratio']:.2f}"
        if "shap_rank" in r and pd.notna(r.get("shap_rank")):
            ev_txt += f"; SHAP rank #{int(r['shap_rank'])}"
        impact = ""
        if len(WHATIF):
            w = WHATIF[WHATIF["feature"] == r["feature"]]
            if len(w):
                impact = f"-{w['overall_reduction_pp_full'].iloc[0]:.2f} pp churn at full adoption (≈{w['churners_avoided_per_1000'].iloc[0]:.0f} churners avoided /1,000 at {CFG.WHAT_IF_ADOPTION:.0%} adoption)"
        rows.append({"priority": len(rows) + 1, "driver": r["feature"], "recommendation": title, "action": action, "evidence": ev_txt,
                     "estimated_impact": impact, "target_segment": f"customers in '{r['highest_risk_level']}' + Priority Save quadrant",
                     "kpi": kpi, "evidence_score": r["evidence_score"]})
        if len(rows) >= 7:
            break
    rows.append({"priority": len(rows) + 1, "driver": "(model)", "recommendation": "Deploy a churn early-warning system",
                 "action": "Score all active policies monthly with this model; route High-risk/Priority-Save customers to retention teams, "
                           "lower tiers to automated digital nudges; refresh the model quarterly.",
                 "evidence": f"Model ROC-AUC {roc_auc_score(y, FINAL_OOF):.3f}; top-20% captures "
                             f"{DECILE_TABLE['cum_churners_captured_%'].iloc[1]:.0f}% of churners" if 'DECILE_TABLE' in globals() else "",
                 "estimated_impact": f"optimal campaign: target top {CAMPAIGN_BEST.iloc[1]['target_%']:.0f}% → net {fmt_num(CAMPAIGN_BEST.iloc[1]['net_profit'])}" if 'CAMPAIGN_BEST' in globals() else "",
                 "target_segment": "all active customers", "kpi": "Precision/recall of alerts; retained revenue vs control group", "evidence_score": np.nan})
    rows.append({"priority": len(rows) + 1, "driver": "(measurement)", "recommendation": "Validate with controlled pilots (A/B tests)",
                 "action": "For each intervention keep a random control group (e.g. 20% of targeted customers) to measure true uplift before scaling.",
                 "evidence": "What-if results are associational", "estimated_impact": "", "target_segment": "each campaign",
                 "kpi": "Uplift in retention rate (treated − control)", "evidence_score": np.nan})
    RECS = pd.DataFrame(rows)
    save_table(RECS, "recommendations")
    with pd.option_context("display.max_colwidth", 120):
        display(RECS[["priority", "driver", "recommendation", "evidence", "estimated_impact", "kpi"]])
    for _, r in RECS.iterrows():
        add_insight("Recommendations", f"{r['priority']}. {r['recommendation']} — {r['action']} Evidence: {r['evidence']}. "
                                       f"{('Impact: ' + r['estimated_impact'] + '. ') if r['estimated_impact'] else ''}KPI: {r['kpi']}.", show=False)
    print(f"\n{len(RECS)} rekomendasi disimpan ke tables/recommendations.csv & insights.md")

# 23. Submission + Validator

- Format mengikuti `sample_submission` (nama kolom, urutan ID). Kalau tidak ada sample → `[ID_COL, TARGET]`.
- `SUBMISSION_MODE = "probability"` → probabilitas churn (kalibrasi hanya dipakai kalau metrik logloss/brier & kalibrasi membantu).
- `SUBMISSION_MODE = "label"` → 0/1 pakai threshold optimal OOF; `SUBMISSION_LABEL_FORMAT="original"` → label asli (misal Yes/No).
- Validator mengecek: jumlah baris, ID cocok & unik, tidak ada NaN, range/nilai valid, nama kolom sama dengan sample.
- Disimpan beberapa varian di `submissions/` (final, best single, label & probability) → kalau boleh submit beberapa kali, coba varian lain.

**Jika X → Y:**
- `ID mismatch` → cek `ID_COL` & tipe data ID (string vs int: "001" vs 1). Notebook menyamakan sebagai string.
- Leaderboard jauh di bawah CV → cek format (probability vs label?), urutan baris, kolom target benar, dan drift (Section 12).

In [ ]:
def build_submission(pred, mode, thr):
    tcol = CFG.SUBMISSION_TARGET_COL
    if sample_sub is not None:
        sub = sample_sub.copy()
        if tcol is None:
            others = [c for c in sub.columns if c != ID_COL]
            tcol = TARGET if TARGET in sub.columns else (others[-1] if others else TARGET)
    else:
        sub = pd.DataFrame({ID_COL: test[ID_COL].values}) if ID_COL and ID_COL in test.columns else pd.DataFrame(index=range(len(pred)))
        tcol = tcol or TARGET
    if mode == "label":
        lab = (pred >= thr).astype(int)
        fmt = CFG.SUBMISSION_LABEL_FORMAT
        if fmt == "auto":
            sample_vals = set(sample_sub[tcol].astype(str)) if sample_sub is not None and tcol in sample_sub else set()
            orig_str = {str(v) for v in POS_ORIGINAL + NEG_ORIGINAL}
            fmt = "original" if (sample_vals & orig_str and not sample_vals <= {"0", "1", "0.0", "1.0", "0.5"}) else "01"
            if sample_sub is None and not all(str(v) in {"0", "1", "0.0", "1.0", "True", "False"} for v in orig_str):
                fmt = "original"
        vals = np.where(lab == 1, POS_ORIGINAL[0], NEG_ORIGINAL[0]) if fmt == "original" else lab
    else:
        vals = pred
    if sample_sub is not None and ID_COL and ID_COL in sub.columns and ID_COL in test.columns:
        m = pd.Series(vals, index=test[ID_COL].astype(str).values)
        if m.index.duplicated().any():
            raise ValueError("ID test tidak unik")
        sub[tcol] = sub[ID_COL].astype(str).map(m).values
    else:
        if sample_sub is not None and len(sub) != len(vals):
            raise ValueError(f"Jumlah baris sample ({len(sub)}) ≠ test ({len(vals)}) dan tidak ada ID untuk align")
        sub[tcol] = vals
    return sub, tcol


def validate_submission(sub, tcol, mode):
    ok = True
    checks = []
    def chk(cond, msg):
        nonlocal ok
        checks.append(("✅" if cond else "❌") + " " + msg)
        ok &= bool(cond)
    chk(len(sub) == len(test), f"jumlah baris = test ({len(sub)} vs {len(test)})")
    chk(sub[tcol].notna().all(), f"tidak ada NaN di '{tcol}' ({sub[tcol].isna().sum()} NaN)")
    if sample_sub is not None:
        chk(list(sub.columns) == list(sample_sub.columns), f"kolom sama dengan sample: {list(sub.columns)}")
        if ID_COL and ID_COL in sub.columns:
            chk((sub[ID_COL].astype(str).values == sample_sub[ID_COL].astype(str).values).all(), "urutan ID sama dengan sample")
    if ID_COL and ID_COL in sub.columns:
        chk(sub[ID_COL].is_unique, "ID unik")
    if mode == "probability":
        chk(sub[tcol].between(0, 1).all(), "probabilitas dalam [0, 1]")
    else:
        chk(sub[tcol].nunique() <= 2, f"label biner: {sorted(sub[tcol].astype(str).unique())[:5]}")
    for c in checks:
        print("  " + c)
    return ok


with safe_section("Submission"):
    if not len(FINAL_TEST):
        print("⏭️ Tidak ada test → submission di-skip.")
    else:
        use_cal = CFG.METRIC in ("logloss", "brier") and CAL_METHOD != "none"
        final_pred = FINAL_TEST_CAL if use_cal else FINAL_TEST
        variants = {"final": (final_pred, CFG.SUBMISSION_MODE)}
        LB = leaderboard()
        best_single = [m for m in LB["model"] if m != "dummy"][0]
        variants["best_single_" + best_single] = (RESULTS[best_single]["test"], CFG.SUBMISSION_MODE)
        variants["final_" + ("label" if CFG.SUBMISSION_MODE == "probability" else "probability")] = (
            final_pred, "label" if CFG.SUBMISSION_MODE == "probability" else "probability")
        for vname, (pred, mode) in variants.items():
            sub, tcol = build_submission(pred, mode, BEST_THR)
            path = os.path.join(CFG.SUB_DIR, f"submission_{vname}.csv")
            sub.to_csv(path, index=False)
            print(f"\n[{vname}] mode={mode} → {path}")
            okv = validate_submission(sub, tcol, mode)
            if vname == "final":
                SUBMISSION = sub
                sub.to_csv(os.path.join(CFG.OUTPUT_DIR, "submission.csv"), index=False)
                print("  ⭐ FINAL submission juga disimpan ke", os.path.join(CFG.OUTPUT_DIR, "submission.csv"), "| valid:", okv)
        display(SUBMISSION.head())
        print(f"Distribusi prediksi final: mean={final_pred.mean():.4f}, min={final_pred.min():.4f}, max={final_pred.max():.4f}")

        # daftar aksi untuk bisnis (customer test + tier + nilai + rekomendasi persona)
        act = pd.DataFrame({ID_COL if ID_COL else "row": test[ID_COL].values if ID_COL and ID_COL in test else np.arange(len(test)),
                            "churn_probability": PROB_BIZ_TEST, "risk_tier": TEST_TIER if "TEST_TIER" in globals() else None})
        if "MARGIN_TE" in globals():
            act["annual_value"] = val_te
            act["expected_value_at_risk"] = PROB_BIZ_TEST * val_te
            act["clv"] = CLV_TE
            act["quadrant"] = quadrant(PROB_BIZ_TEST, val_te)
            if "TARGET_FLAG_TEST" in globals():
                act["recommend_contact"] = TARGET_FLAG_TEST
        act = act.sort_values("churn_probability", ascending=False)
        act.to_csv(os.path.join(CFG.TABLE_DIR, "test_customer_action_list.csv"), index=False)
        print("Daftar customer test + tier + value disimpan ke tables/test_customer_action_list.csv")

In [ ]:
with safe_section("Save OOF & test predictions"):
    oof_df = pd.DataFrame({"y": y, "final_oof": FINAL_OOF, "final_oof_calibrated": FINAL_OOF_CAL})
    if ID_COL:
        oof_df.insert(0, ID_COL, train[ID_COL].values)
    for m in RESULTS:
        oof_df[f"oof_{m}"] = RESULTS[m]["oof"]
    oof_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "oof_predictions.csv"), index=False)
    if len(FINAL_TEST):
        tp = pd.DataFrame({"final_test": FINAL_TEST, "final_test_calibrated": FINAL_TEST_CAL})
        if ID_COL and ID_COL in test:
            tp.insert(0, ID_COL, test[ID_COL].values)
        for m in RESULTS:
            if len(RESULTS[m]["test"]):
                tp[f"test_{m}"] = RESULTS[m]["test"]
        tp.to_csv(os.path.join(CFG.OUTPUT_DIR, "test_predictions.csv"), index=False)
    print("Tersimpan: oof_predictions.csv, test_predictions.csv")

# 24. Export untuk Laporan & Ringkasan Error

Output di `CFG.OUTPUT_DIR`:
| File | Isi | Dipakai untuk |
|---|---|---|
| `figures/figNN_*.png` | semua grafik bernomor | pilih 6–10 terbaik untuk laporan (sisanya lampiran) |
| `figure_index.md` | daftar figure + judul | cepat cari figure |
| `tables/*.csv` + `report_tables.xlsx` | semua tabel | tabel laporan & lampiran |
| `insights.md` | semua kalimat insight (English) dikelompokkan per section | bahan menulis Findings & Recommendations |
| `run_summary.json` | config, skor, model final, threshold | dokumentasi metodologi |
| `experiments.csv` | log tiap run (append) | bandingkan eksperimen |
| `submission.csv` | submission final | upload |

Lihat juga `01_Churn_Insurance/PANDUAN_LAPORAN.md` untuk struktur artikel 10 halaman & mapping figure → bagian laporan.

In [ ]:
with safe_section("Export report assets"):
    if HAS_OPENPYXL and TABLES:
        xlsx = os.path.join(CFG.OUTPUT_DIR, "report_tables.xlsx")
        used = set()
        with pd.ExcelWriter(xlsx) as w:
            for name, df in TABLES.items():
                sheet = re.sub(r"[\[\]\*\?/\\:]", "_", name)[:31]
                while sheet in used:
                    sheet = sheet[:28] + f"_{len(used)}"
                used.add(sheet)
                df2 = df.copy()
                for c in df2.columns:
                    if df2[c].dtype == object:
                        df2[c] = df2[c].astype(str)
                df2.to_excel(w, sheet_name=sheet, index=False)
        print("Excel:", xlsx)
    with open(os.path.join(CFG.OUTPUT_DIR, "insights.md"), "w", encoding="utf-8") as f:
        f.write("# Auto-generated insights\n\n")
        for sec in OrderedDict.fromkeys(s for s, _ in INSIGHTS):
            f.write(f"## {sec}\n\n")
            for s, t in INSIGHTS:
                if s == sec:
                    f.write(f"- {t}\n")
            f.write("\n")
    with open(os.path.join(CFG.OUTPUT_DIR, "figure_index.md"), "w", encoding="utf-8") as f:
        f.write("# Figure index\n\n| File | Title |\n|---|---|\n")
        for fn, t in FIGURE_INDEX:
            f.write(f"| {fn} | {t} |\n")
    LB = leaderboard()
    summary = {"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "run_mode": CFG.RUN_MODE, "n_train": int(N_TR),
               "n_test": int(len(FINAL_TEST)), "churn_rate": CHURN_RATE, "target": TARGET, "target_mapping": {str(k): v for k, v in TARGET_MAPPING.items()},
               "metric": CFG.METRIC, "final_model": FINAL_NAME, "final_weights": {k: float(v) for k, v in FINAL_WEIGHTS.items()},
               "final_oof_score": float(show_score(rank_score(y, FINAL_OOF))), "final_oof_auc": float(roc_auc_score(y, FINAL_OOF)),
               "threshold": BEST_THR, "threshold_metric": THR_METRIC, "calibration": CAL_METHOD,
               "leaderboard": LB.drop(columns=[c for c in LB.columns if c == "time_s"]).to_dict(orient="records"),
               "leaky_cols": LEAKY_COLS, "failed_models": FAILED_MODELS, "section_errors": dict(SECTION_ERRORS),
               "n_features": len(MODEL_FEATURES), "roles": ROLES}
    with open(os.path.join(CFG.OUTPUT_DIR, "run_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, default=str)
    exp_path = os.path.join(CFG.OUTPUT_DIR, "experiments.csv")
    exp_row = pd.DataFrame([{"timestamp": summary["timestamp"], "run_mode": CFG.RUN_MODE, "metric": CFG.METRIC,
                             "final_model": FINAL_NAME, "oof_score": summary["final_oof_score"], "oof_auc": summary["final_oof_auc"],
                             "models": ",".join(RESULTS.keys()), "n_features": len(MODEL_FEATURES), "optuna": CFG.USE_OPTUNA}])
    exp_row.to_csv(exp_path, mode="a", header=not os.path.exists(exp_path), index=False)
    print(f"Figures: {len(FIGURE_INDEX)} | Tables: {len(TABLES)} | Insights: {len(INSIGHTS)}")
    print("Folder output:", os.path.abspath(CFG.OUTPUT_DIR))

In [ ]:
print("=" * 70)
print("RINGKASAN RUN")
print("=" * 70)
try:
    print(f"Final model : {FINAL_NAME} | OOF {CFG.METRIC} = {show_score(rank_score(y, FINAL_OOF)):.5f} | AUC = {roc_auc_score(y, FINAL_OOF):.5f}")
except Exception:
    print("Final model belum terbentuk (cek error modeling).")
print("Model gagal :", FAILED_MODELS if FAILED_MODELS else "tidak ada ✅")
if SECTION_ERRORS:
    print("Section error (notebook tetap jalan, cek & perbaiki kalau section ini penting):")
    for k, v in SECTION_ERRORS.items():
        print(f"  ❌ {k}: {v}")
else:
    print("Section error: tidak ada ✅")
print("\nInsight utama:")
for s, t in INSIGHTS[:12]:
    print(f"  [{s}] {t[:220]}")

## 24.2 Executive Summary Figure (1 gambar ringkasan untuk halaman pertama laporan / slide)
KPI utama + driver actionable teratas + gains curve dalam satu figure.

In [ ]:
with safe_section("Executive summary figure"):
    fig = plt.figure(figsize=(13, 7.2))
    gs = fig.add_gridspec(2, 4, height_ratios=[1, 2.6], hspace=0.45, wspace=0.35)
    auc_txt = f"{roc_auc_score(y, FINAL_OOF):.3f}"
    if "FINAL_AUC_CI" in globals():
        auc_txt += f"\n(95% CI {FINAL_AUC_CI[0]:.3f}–{FINAL_AUC_CI[1]:.3f})"
    kpis = [("Churn rate", pct(CHURN_RATE), f"{int(y.sum()):,} of {N_TR:,} customers"),
            ("Model ROC-AUC", auc_txt, f"final: {FINAL_NAME}")]
    if "DECILE_TABLE" in globals():
        kpis.append(("Top-20% capture", f"{DECILE_TABLE['cum_churners_captured_%'].iloc[1]:.0f}%",
                     f"of churners ({DECILE_TABLE['cum_lift'].iloc[1]:.1f}x lift)"))
    if "at_risk_tr" in globals() and VALUE_SOURCE:
        kpis.append(("Annual value at risk", f"{fmt_num(at_risk_tr)} {CFG.CURRENCY}", f"{pct(at_risk_tr / val_tr.sum())} of premium base"))
    elif "KS" in globals():
        kpis.append(("KS statistic", f"{KS:.2f}", "risk separation"))
    for i, (t, v, s) in enumerate(kpis[:4]):
        ax = fig.add_subplot(gs[0, i])
        ax.axis("off")
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, color="#F2F4F8", zorder=0))
        ax.text(0.5, 0.78, t, ha="center", va="center", fontsize=10, color="#555", transform=ax.transAxes)
        ax.text(0.5, 0.45, v, ha="center", va="center", fontsize=15 if "\n" not in v else 12, weight="bold", color=CFG.PALETTE[1] if i == 0 else "#222", transform=ax.transAxes)
        ax.text(0.5, 0.12, s, ha="center", va="center", fontsize=8, color="#777", transform=ax.transAxes)
    ax1 = fig.add_subplot(gs[1, :2])
    if len(EVIDENCE) and "shap_share_pct" in EVIDENCE:
        ev = EVIDENCE.dropna(subset=["shap_share_pct"]).sort_values("shap_share_pct", ascending=False).head(8)
        ax1.barh(ev["feature"][::-1].str.slice(0, 30), ev["shap_share_pct"][::-1],
                 color=[CFG.PALETTE[1] if a else "#9E9E9E" for a in ev["actionable"][::-1]])
        ax1.set_xlabel("Share of model importance (SHAP, %)")
        ax1.set_title("Top churn drivers (red = actionable)")
    elif len(DRIVER_TABLE):
        ev = DRIVER_TABLE.head(8)
        ax1.barh(ev["feature"][::-1].str.slice(0, 30), ev["iv"][::-1], color=CFG.PALETTE[1])
        ax1.set_xlabel("Information Value")
        ax1.set_title("Top churn drivers")
    ax2 = fig.add_subplot(gs[1, 2:])
    if "DECILE_TABLE" in globals():
        ax2.plot(np.r_[0, DECILE_TABLE["cum_customers_%"]], np.r_[0, DECILE_TABLE["cum_churners_captured_%"]], "o-", color=CFG.PALETTE[1], label="Model")
        ax2.plot([0, 100], [0, 100], "k--", label="Random")
        ax2.set_xlabel("% customers targeted (highest risk first)")
        ax2.set_ylabel("% churners captured")
        ax2.set_title("Targeting efficiency (cumulative gains)")
        ax2.legend(fontsize=8)
    fig.suptitle("Customer Churn — Executive Summary", fontsize=14, weight="bold", y=1.0)
    save_fig("executive_summary", fig)

## 24.3 Auto Report Draft (REPORT_DRAFT.docx + .md) — ⭐ alur deliverable

Notebook menyusun **draft artikel lengkap** mengikuti struktur `PANDUAN_LAPORAN.md`, sudah:
- berformat sesuai soal (**A4, Times New Roman 12, spasi 1.15, margin atas/bawah 4 cm, kiri/kanan 3 cm**),
- berisi **angka hasil run ini** (churn rate, driver, survival, AUC + CI, lift, value at risk, ROI, what-if, persona, rekomendasi),
- menyisipkan **figure & tabel utama** dengan caption bernomor; figure lain di **Appendix**,
- menandai bagian yang **WAJIB kamu edit** dengan `[EDIT: ...]` (di-highlight kuning di Word).

File:
| File | Isi |
|---|---|
| `REPORT_DRAFT.docx` | draft artikel siap edit di Word → Save As PDF |
| `REPORT_DRAFT.md` | versi teks (mudah dibaca/copy) |
| `REPORT_TODO.md` | daftar semua `[EDIT]` + checklist verifikasi + mapping section → file output |

### Alur deliverable (ikuti urut)
1. Isi `CFG.TEAM_NAME`, `CFG.COMPANY_NAME`, `CFG.REPORT_TITLE`, parameter bisnis (margin, cost, success rate) dari soal.
2. `RUN_MODE="full"`, `FIG_DPI=300` → Restart & Run All.
3. Buka `REPORT_TODO.md` → kerjakan semua item. Buka `REPORT_DRAFT.docx` → isi `[EDIT]`, parafrase kalimat otomatis dengan gaya tim, beri nama persona, sesuaikan rekomendasi dengan konteks soal.
4. Cek **≤ 10 halaman** (cover & appendix tidak dihitung). Kalau lebih: pindahkan figure ke Appendix, ringkas Findings.
5. Word → Save As **PDF** dengan nama `TeamName_Final Stage 1.pdf`.
6. Set `CFG.ARTICLE_PDF_PATH`, `CFG.MAKE_ZIP=True` → jalankan cell 24.4 → upload ZIP.

> Draft = kerangka + angka. **Nilai tinggi datang dari interpretasi & narasi tim** — jangan submit draft mentah.

In [ ]:
def _g(name, default=None):
    return globals().get(name, default)


FE_LABELS = {"fe_tenure_years": "tenure (years)", "fe_tenure_band": "tenure band", "fe_is_new_customer": "new customer (<12 months)",
             "fe_age_band": "age band", "fe_premium_to_income": "premium-to-income ratio", "fe_sum_insured_to_premium": "sum insured to premium ratio",
             "fe_has_claim": "has made a claim", "fe_claims_per_year": "claims per year of tenure", "fe_claim_reject_rate": "claim rejection rate",
             "fe_has_complaint": "has complained", "fe_has_late_payment": "has late payments", "fe_premium_increase": "premium increased at renewal",
             "fe_big_premium_increase": "premium increase above 10%", "fe_multi_product": "holds multiple products",
             "fe_avg_paid_per_period": "average amount paid per period", "fe_paid_vs_expected": "paid vs expected premium",
             "fe_n_missing": "number of missing fields"}


def pretty(f):
    """Nama fitur → label bisnis untuk laporan (CFG.FEATURE_LABELS > label FE bawaan > underscore jadi spasi)."""
    f = str(f)
    if f in CFG.FEATURE_LABELS:
        return CFG.FEATURE_LABELS[f]
    if f in FE_LABELS:
        return FE_LABELS[f]
    s = re.sub(r"^n_", "number of ", f)
    s = s.replace("__freq", " (frequency)").replace("_", " ").strip()
    return s


_PRETTY_KEYS = None


def prettify_text(text):
    global _PRETTY_KEYS
    if _PRETTY_KEYS is None:
        keys = set(train.columns) | set(FE_LABELS) | set(CFG.FEATURE_LABELS)
        _PRETTY_KEYS = sorted([k for k in keys if isinstance(k, str) and ("_" in k or k in CFG.FEATURE_LABELS)], key=len, reverse=True)
    if "[EDIT" in text:
        parts = re.split(r"(\[EDIT[^\]]*\])", text)
        return "".join(p if p.startswith("[EDIT") else prettify_text(p) for p in parts)
    for k in _PRETTY_KEYS:
        if k in text:
            text = re.sub(r"(?<![A-Za-z0-9_])" + re.escape(k) + r"(?![A-Za-z0-9_])", pretty(k), text)
    return text


def find_fig(key):
    for fn, title in FIGURE_INDEX:
        if key in fn:
            return os.path.join(CFG.FIG_DIR, fn)
    return None


def build_report_content():
    """Kumpulkan seluruh isi draft laporan sebagai list blok: (type, payload)."""
    B = []
    H = lambda t, lvl=1: B.append(("h", (t, lvl)))
    P = lambda t: B.append(("p", prettify_text(t)))
    FIG = lambda key, cap, w=14: B.append(("fig", (key, cap, w)))
    def TAB(df, cap, max_cols=8):
        df = df.copy().iloc[:, :max_cols]
        for c in df.columns:
            if df[c].dtype == object:
                df[c] = df[c].astype(str).map(prettify_text)
        df.columns = [prettify_text(str(c)) for c in df.columns]
        B.append(("tab", (df, cap)))
    ins = lambda sec: [t for s, t in INSIGHTS if s == sec]

    auc = roc_auc_score(y, FINAL_OOF)
    ci = _g("FINAL_AUC_CI")
    auc_txt = f"ROC-AUC of {auc:.3f}" + (f" (95% CI {ci[0]:.3f}–{ci[1]:.3f})" if ci else "")
    ev = EVIDENCE if len(EVIDENCE) else pd.DataFrame(columns=["feature", "actionable"])
    act = ev[ev["actionable"] == True]["feature"].head(3).tolist()
    dec = _g("DECILE_TABLE")
    top20 = f"{dec['cum_churners_captured_%'].iloc[1]:.0f}%" if dec is not None else "[EDIT]"
    camp = _g("CAMPAIGN_BEST")
    recs = _g("RECS", pd.DataFrame())
    wi = _g("WHATIF", pd.DataFrame())
    company = CFG.COMPANY_NAME

    # ---------------- Executive summary ----------------
    H("Executive Summary")
    s = (f"Customer churn is a direct threat to the premium income and profitability of {company}. Using data on {N_TR:,} customers "
         f"(overall churn rate {pct(CHURN_RATE)}), we combined statistical driver analysis, survival analysis, interpretable regression and "
         f"gradient-boosted machine learning to explain why and when customers leave and whom to prioritise. ")
    if act:
        s += f"Churn is most strongly and consistently associated with {', '.join(act[:-1]) + ' and ' + act[-1] if len(act) > 1 else act[0]}. "
    hz = _g("HAZARD_TABLE")
    if hz is not None and len(hz):
        pk = hz.sort_values("hazard", ascending=False).iloc[0]
        s += f"Churn risk is highest in tenure interval {pk['interval']} {_g('SURV_UNIT', 'periods')}. "
    s += f"Our final model reaches an out-of-fold {auc_txt}; targeting the 20% highest-risk customers captures {top20} of all churners. "
    if VALUE_SOURCE and _g("at_risk_tr") is not None:
        s += f"Expected annual premium at risk is {fmt_num(at_risk_tr)} {CFG.CURRENCY} ({pct(at_risk_tr / val_tr.sum())} of the premium base). "
    if len(recs):
        s += "We recommend: " + "; ".join(f"({i + 1}) {r}" for i, r in enumerate(recs["recommendation"].head(3))) + ". "
    if camp is not None and len(camp) > 1:
        b = camp.iloc[1]
        s += (f"A campaign simulation suggests that targeting the top {b['target_%']:.0f}% of customers by expected loss yields a net benefit of "
              f"{fmt_num(b['net_profit'])} {CFG.CURRENCY} (ROI {b['roi']:.1f}x) under our stated assumptions.")
    P(s)
    P("[EDIT: tambahkan 1 kalimat konteks spesifik dari soal (produk, pasar, tujuan manajemen) dan rapikan ringkasan ini maks. ½ halaman.]")

    # ---------------- 1. Introduction ----------------
    H("1. Introduction and Business Problem")
    P(f"In insurance, retaining an existing policyholder is considerably cheaper than acquiring a new one: acquisition costs (commissions, "
      f"marketing, underwriting) are recovered over several years of premiums, so early lapses and non-renewals directly erode profitability "
      f"and persistency. [EDIT: konteks {company} dari soal — lini produk, situasi churn saat ini, kenapa manajemen peduli.]")
    P("This study answers three questions: (i) Which customer, policy and service factors drive churn? (ii) When during the customer "
      "lifecycle is churn risk highest? (iii) Which customers should be prioritised, and with which interventions, to maximise retained value?")

    # ---------------- 2. Data & methodology ----------------
    H("2. Data and Methodology")
    H("2.1 Data", 2)
    n_raw = len([c for c in _g("ORIGINAL_COLS", []) if c not in (TARGET, ID_COL)])
    P(f"The dataset contains {N_TR:,} customers{' (plus ' + format(len(FINAL_TEST), ',') + ' customers to be scored)' if len(FINAL_TEST) else ''} "
      f"and {n_raw} original variables covering [EDIT: demographics, policy characteristics, premium & payment behaviour, claims and service interactions]. "
      f"The target is {TARGET} (churn = {', '.join(map(str, POS_ORIGINAL))}); the overall churn rate is {pct(CHURN_RATE)}.")
    H("2.2 Data preparation", 2)
    prep = (f"Data preparation included automatic type correction (e.g. numbers stored as text and date parsing), harmonisation of inconsistent "
            f"category labels, removal of {_g('N_DUP_REMOVED', 0)} duplicate records and treatment of impossible values. ")
    if FE_NEW:
        prep += (f"We engineered {len(FE_NEW)} domain features, including "
                 f"{', '.join([f for f in FE_NEW if not f.endswith(('_year', '_month'))][:6])}. ")
    if LEAKY_COLS:
        prep += f"Variables recorded only after churn ({', '.join(LEAKY_COLS)}) were excluded from modelling to prevent target leakage."
    P(prep)
    H("2.3 Analytical framework", 2)
    models_used = [m for m in RESULTS if m != "dummy"]
    P("Our framework has four layers (Figure 1): (1) univariate driver analysis with χ² and Mann-Whitney tests, effect sizes (Cramér's V, "
      "rank-biserial r), Information Value and Benjamini-Hochberg correction; (2) survival analysis (Kaplan-Meier, log-rank tests and Cox "
      "proportional hazards) to capture when churn occurs; (3) a multivariate logistic regression for interpretable odds ratios; and (4) predictive "
      f"machine learning ({', '.join(sorted({RESULTS[m]['model'] for m in models_used}))}) evaluated with stratified {CFG.N_FOLDS}-fold cross-validation"
      f"{', tuned with Bayesian optimisation (Optuna)' if TUNED_PARAMS else ''} and combined via ensemble selection. Model behaviour was explained "
      "with SHAP values and partial dependence, and translated into business value with lift analysis, customer lifetime value, a risk–value "
      "matrix, a retention-campaign ROI simulation and what-if scenarios.")
    P("[EDIT: sisipkan diagram alur (Figure 1) — Data → Cleaning & Feature Engineering → Driver analysis → Survival → Interpretable model → "
      "ML & explainability → Business simulation → Recommendations. Bisa dibuat di PowerPoint/Word SmartArt.]")
    H("2.4 Validation", 2)
    P(f"All performance figures are out-of-fold estimates. Target-dependent encodings were fitted inside each training fold. Sanity checks "
      f"confirmed the absence of leakage (ROC-AUC with shuffled labels ≈ 0.5) "
      f"{'and similar train/test distributions (adversarial validation AUC = ' + format(ADV_AUC, '.2f') + ')' if not np.isnan(_g('ADV_AUC', np.nan)) else ''}. "
      f"Uncertainty is reported with bootstrap 95% confidence intervals.")

    # ---------------- 3. Findings ----------------
    H("3. Findings")
    H("3.1 Churn overview", 2)
    P(f"{pct(CHURN_RATE)} of customers churned, i.e. a [EDIT: moderate/severe] class imbalance; we therefore evaluate models with ROC-AUC and "
      f"precision–recall rather than accuracy.")
    H("3.2 Key churn drivers", 2)
    for t in ins("Drivers")[:5]:
        P(t)
    FIG("driver_churn_rate_1", "Churn rate by the strongest drivers (bars: churn rate with 95% CI; dashed line: overall churn rate).")
    if len(ev):
        cols = [c for c in ["feature", "highest_risk_level", "highest_risk_rate", "highest_risk_lift", "iv", "odds_ratio", "hazard_ratio", "shap_rank", "actionable"] if c in ev]
        t = ev[cols].head(8).copy()
        if "highest_risk_rate" in t:
            t["highest_risk_rate"] = (100 * t["highest_risk_rate"]).round(1)
        t = t.rename(columns={"feature": "Driver", "highest_risk_level": "High-risk group", "highest_risk_rate": "Churn % (group)",
                              "highest_risk_lift": "Lift", "iv": "IV", "odds_ratio": "OR", "hazard_ratio": "HR", "shap_rank": "SHAP rank",
                              "actionable": "Actionable"})
        TAB(t.round(2), "Driver evidence matrix (univariate, multivariate, survival and SHAP evidence).")
    P("[EDIT: jelaskan 2–3 driver utama dengan bahasa bisnis — kenapa masuk akal di industri asuransi (lihat PLAYBOOK §D2).]")
    if ins("Interpretable model"):
        H("Multivariate evidence", 3)
        seen_terms = set()
        for t in ins("Interpretable model"):
            m_ = re.search(r"constant, (.+?) \(", t)
            base_ = (m_.group(1).split(" = ")[0] if m_ else t)
            if base_ in seen_terms:
                continue
            seen_terms.add(base_)
            P(t)
            if len(seen_terms) >= 4:
                break
    if SURV_AVAILABLE:
        H("3.3 When do customers churn?", 2)
        for t in ins("Survival")[:4]:
            P(t)
        FIG("km_by_group" if find_fig("km_by_group") else "km_overall", "Kaplan-Meier survival curves by key customer groups (log-rank p-values in titles).")
        P("[EDIT: interpretasi — kapan momen kritis (onboarding? renewal pertama?) dan implikasinya untuk timing intervensi.]")
    H("3.4 High-risk segments", 2)
    for t in ins("Segments")[:3]:
        P(t)
    if len(SEGMENT_RULES):
        sr = SEGMENT_RULES.head(3)
        sr = pd.concat([sr, SEGMENT_RULES.tail(1)])
        sr = sr[["segment_rule", "n_customers", "churn_rate", "lift"]].copy()
        sr["churn_rate"] = (100 * sr["churn_rate"]).round(1)
        TAB(sr.rename(columns={"segment_rule": "Segment (decision-tree rule)", "n_customers": "Customers", "churn_rate": "Churn %", "lift": "Lift"}).round(2),
            "Highest-risk and most loyal segments identified by a shallow decision tree.")
    H("3.5 Predictive model", 2)
    for t in ins("Model")[:4]:
        P(t)
    LBr = leaderboard()
    lbt = LBr[LBr["model"] != "dummy"].head(6)[["model", f"oof_{CFG.METRIC}", "cv_std", "oof_auc", "pr_auc"]].round(4)
    TAB(lbt.rename(columns={"model": "Model", f"oof_{CFG.METRIC}": f"OOF {CFG.METRIC}", "cv_std": "CV std", "oof_auc": "ROC-AUC", "pr_auc": "PR-AUC"}),
        "Cross-validated model comparison.")
    FIG("gains_and_lift", "Cumulative gains and lift by risk decile: the model concentrates churners in the highest-risk deciles.")
    H("3.6 Explaining churn risk", 2)
    FIG("shap_beeswarm", "SHAP summary: each dot is a customer; positive values increase churn risk (colour = feature value).", 12)
    for t in ins("Explainability")[:3]:
        P(t)
    P("[EDIT: hubungkan SHAP dengan temuan statistik — driver mana yang konsisten di semua metode (robust).]")

    # ---------------- 4. Business ----------------
    H("4. Business Impact and Recommendations")
    H("4.1 Value at risk and prioritisation", 2)
    for t in ins("Business")[:4]:
        P(t)
    FIG("risk_value_matrix", "Risk × value matrix: 'Priority Save' customers combine high churn risk with above-median value.")
    H("4.2 Retention campaign simulation", 2)
    for t in [t for t in ins("Business") if "campaign" in t.lower()]:
        P(t)
    FIG("campaign_profit_curve", "Net profit of a retention campaign by share of customers targeted (model vs random targeting).")
    P(f"[EDIT: sebutkan asumsi bisnis — margin {CFG.PROFIT_MARGIN:.0%}, cost per contact {fmt_num(CFG.RETENTION_COST)} {CFG.CURRENCY}, "
      f"success rate {CFG.RETENTION_SUCCESS_RATE:.0%} — dan sumbernya (soal / asumsi wajar); rujuk tabel sensitivity di Appendix.]")
    if len(wi):
        H("4.3 What-if scenarios", 2)
        for t in [t for t in ins("Business") if t.startswith("What-if")][:3]:
            P(t)
        FIG("what_if_scenarios", "Model-based simulation of interventions (associational, to be validated by pilots).")
    if ins("Personas"):
        H("4.4 Churn personas", 2)
        for t in ins("Personas"):
            P(t)
        P("[EDIT: beri nama tiap persona (misal 'Price-shocked renewers', 'Service-frustrated claimants') + 1 intervensi khusus per persona.]")
    H("4.5 Recommendations", 2)
    if len(recs):
        rt = recs[["priority", "recommendation", "evidence", "estimated_impact", "kpi"]].copy()
        TAB(rt.rename(columns={"priority": "#", "recommendation": "Recommendation", "evidence": "Evidence", "estimated_impact": "Expected impact", "kpi": "KPI"}),
            "Evidence-based retention recommendations.")
        for _, r in recs.head(5).iterrows():
            P(f"{int(r['priority'])}. {r['recommendation']}. {r['action']}")
    P("[EDIT: tambahkan roadmap — Quick wins (0–3 bulan), Medium (3–6 bulan), Long term (6–12 bulan) — dan rencana pengukuran (A/B test dengan control group).]")

    # ---------------- 5. Conclusion ----------------
    H("5. Conclusion and Limitations")
    P(f"Churn at {company} is driven primarily by {', '.join(act) if act else '[EDIT]'}, and is concentrated in identifiable segments and "
      f"lifecycle stages. A validated predictive model ({auc_txt}) enables proactive, value-based retention targeting. "
      "[EDIT: 1–2 kalimat jawaban langsung untuk 3 pertanyaan riset.]")
    P("Limitations: (i) the data are a single snapshot, so relationships are associational rather than causal; (ii) the ROI simulation relies on "
      "assumed costs and success rates (see sensitivity analysis); (iii) potentially important factors such as competitor offers and life events "
      "are not observed. Future work should validate interventions through randomised pilots and enrich the data with [EDIT: interaction logs / "
      "stated reasons for leaving].")

    H("References")
    for ref in REFERENCES:
        P(ref)

    # ---------------- Appendix ----------------
    B.append(("pagebreak", None))
    H("Appendix (not counted in the page limit)")
    used = set(p[0] for t, p in B if t == "fig")
    appendix_keys = [("executive_summary", "Executive summary dashboard."), ("roc_pr_curves", "ROC and precision–recall curves (out-of-fold)."),
                     ("confusion_matrices", "Confusion matrices at 0.5 and at the optimised threshold."),
                     ("information_value_ranking", "Information Value of each feature."), ("logit_odds_ratios", "Logistic regression odds ratios (95% CI)."),
                     ("cox_hazard_ratios", "Cox proportional hazards: hazard ratios (95% CI)."), ("hazard_by_tenure", "Churn hazard by tenure interval."),
                     ("km_overall", "Overall Kaplan-Meier curve."), ("interaction_heatmaps", "Two-way churn-rate heatmaps."),
                     ("segment_tree", "Churn segmentation tree."), ("shap_dependence", "SHAP dependence plots."),
                     ("partial_dependence", "Partial dependence plots."), ("driver_heterogeneity", "Driver importance by segment."),
                     ("churn_personas_heatmap", "Churn personas (SHAP clustering)."), ("churn_reasons", "Stated churn reasons."),
                     ("calibration_reliability", "Calibration (reliability) diagram."), ("learning_curve", "Learning curve."),
                     ("model_leaderboard", "Model comparison chart.")]
    for k, cap in appendix_keys:
        if k not in used and find_fig(k):
            FIG(k, cap, 13)
    for name, cap in [("risk_tiers", "Risk tiers."), ("campaign_sensitivity", "Campaign sensitivity analysis."),
                      ("what_if_scenarios", "What-if scenario results."), ("churn_personas", "Churn persona profiles."),
                      ("bootstrap_model_comparison", "Paired bootstrap comparison of models.")]:
        if name in TABLES:
            TAB(TABLES[name].head(12).round(3) if hasattr(TABLES[name], "round") else TABLES[name].head(12), cap)
    return B


REFERENCES = [
    "Akiba, T., Sano, S., Yanase, T., Ohta, T., & Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. Proceedings of the 25th ACM SIGKDD Conference, 2623–2631.",
    "Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate. Journal of the Royal Statistical Society: Series B, 57(1), 289–300.",
    "Caruana, R., Niculescu-Mizil, A., Crew, G., & Ksikes, A. (2004). Ensemble selection from libraries of models. Proceedings of ICML 2004.",
    "Cox, D. R. (1972). Regression models and life-tables. Journal of the Royal Statistical Society: Series B, 34(2), 187–220.",
    "Kaplan, E. L., & Meier, P. (1958). Nonparametric estimation from incomplete observations. Journal of the American Statistical Association, 53(282), 457–481.",
    "Ke, G., et al. (2017). LightGBM: A highly efficient gradient boosting decision tree. Advances in Neural Information Processing Systems, 30.",
    "Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. Advances in Neural Information Processing Systems, 30.",
    "Prokhorenkova, L., Gusev, G., Vorobev, A., Dorogush, A. V., & Gulin, A. (2018). CatBoost: Unbiased boosting with categorical features. Advances in Neural Information Processing Systems, 31.",
    "Verbeke, W., Dejaeger, K., Martens, D., Hur, J., & Baesens, B. (2012). New insights into churn prediction in the telecommunication sector: A profit driven data mining approach. European Journal of Operational Research, 218(1), 211–229.",
]


def write_report_md(blocks, path):
    fig_n, tab_n = 0, 0
    lines = [f"# {CFG.REPORT_TITLE}", "", f"*{CFG.TEAM_NAME} — Final Stage 1*", ""]
    for kind, p in blocks:
        if kind == "h":
            t, lvl = p
            lines += ["", "#" * (lvl + 1) + " " + t, ""]
        elif kind == "p":
            lines += [p, ""]
        elif kind == "fig":
            key, cap, _ = p
            f = find_fig(key)
            if f:
                fig_n += 1
                lines += [f"![Figure {fig_n}](figures/{os.path.basename(f)})", "", f"*Figure {fig_n}. {cap}*", ""]
        elif kind == "tab":
            df, cap = p
            tab_n += 1
            lines += [f"*Table {tab_n}. {cap}*", ""]
            fmt = lambda v: f"{v:,.2f}"
            try:
                lines += [df.to_markdown(index=False, floatfmt=",.2f"), ""]
            except Exception:
                lines += ["```", df.to_string(index=False, float_format=fmt), "```", ""]
        elif kind == "pagebreak":
            lines += ["", "---", ""]
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))


def write_report_docx(blocks, path):
    from docx import Document
    from docx.shared import Pt, Cm, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_COLOR_INDEX, WD_BREAK
    from docx.enum.section import WD_SECTION
    from docx.oxml.ns import qn

    doc = Document()

    def fmt_section(sec):
        sec.page_width, sec.page_height = Cm(21.0), Cm(29.7)
        sec.top_margin = sec.bottom_margin = Cm(4)
        sec.left_margin = sec.right_margin = Cm(3)

    def style_font(st, size, bold=None, italic=None):
        st.font.name = "Times New Roman"
        st.font.size = Pt(size)
        st.font.color.rgb = RGBColor(0, 0, 0)
        if bold is not None:
            st.font.bold = bold
        if italic is not None:
            st.font.italic = italic
        rpr = st.element.get_or_add_rPr()
        rf = rpr.find(qn("w:rFonts"))
        if rf is None:
            rf = rpr.makeelement(qn("w:rFonts"), {})
            rpr.append(rf)
        for a in ("w:ascii", "w:hAnsi", "w:eastAsia", "w:cs"):
            rf.set(qn(a), "Times New Roman")
        st.paragraph_format.line_spacing = 1.15

    fmt_section(doc.sections[0])
    style_font(doc.styles["Normal"], 12)
    doc.styles["Normal"].paragraph_format.space_after = Pt(6)
    doc.styles["Normal"].paragraph_format.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    for nm, sz in [("Title", 16), ("Heading 1", 12), ("Heading 2", 12), ("Heading 3", 12)]:
        style_font(doc.styles[nm], sz, bold=True, italic=(nm == "Heading 3"))
        doc.styles[nm].paragraph_format.space_before = Pt(8)
        doc.styles[nm].paragraph_format.space_after = Pt(4)
    style_font(doc.styles["Caption"], 10, bold=False, italic=True)

    def rich(par, text):
        for part in re.split(r"(\[EDIT[^\]]*\])", text):
            if not part:
                continue
            r = par.add_run(part)
            if part.startswith("[EDIT"):
                r.font.highlight_color = WD_COLOR_INDEX.YELLOW
                r.bold = True

    # cover
    for _ in range(6):
        doc.add_paragraph()
    p = doc.add_paragraph(style="Title")
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.add_run(CFG.REPORT_TITLE)
    for t in ["Final Stage 1 — Data Analytics Competition", "", f"Team: {CFG.TEAM_NAME}", "[EDIT: Member 1 — Member 2]", "[EDIT: University]"]:
        q = doc.add_paragraph()
        q.alignment = WD_ALIGN_PARAGRAPH.CENTER
        rich(q, t)
    sec = doc.add_section(WD_SECTION.NEW_PAGE)
    fmt_section(sec)

    fig_n, tab_n = 0, 0
    for kind, pl in blocks:
        if kind == "h":
            t, lvl = pl
            doc.add_paragraph(t, style=f"Heading {min(lvl, 3)}")
        elif kind == "p":
            rich(doc.add_paragraph(), pl)
        elif kind == "fig":
            key, cap, w = pl
            f = find_fig(key)
            if not f:
                continue
            fig_n += 1
            doc.add_picture(f, width=Cm(w))
            doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
            c = doc.add_paragraph(f"Figure {fig_n}. {cap}", style="Caption")
            c.alignment = WD_ALIGN_PARAGRAPH.CENTER
        elif kind == "tab":
            df, cap = pl
            tab_n += 1
            c = doc.add_paragraph(f"Table {tab_n}. {cap}", style="Caption")
            df = df.copy()
            tbl = doc.add_table(rows=1, cols=len(df.columns))
            tbl.style = "Table Grid"
            for j, col in enumerate(df.columns):
                cell = tbl.rows[0].cells[j]
                cell.text = ""
                r = cell.paragraphs[0].add_run(str(col))
                r.bold = True
                r.font.size = Pt(9)
            for _, row in df.iterrows():
                cells = tbl.add_row().cells
                for j, v in enumerate(row.values):
                    txt = (f"{v:,.2f}" if abs(v) < 1e6 else fmt_num(v)) if isinstance(v, (float, np.floating)) and not np.isnan(v) else ("" if isinstance(v, float) else str(v))
                    cells[j].text = ""
                    r = cells[j].paragraphs[0].add_run(txt[:300])
                    r.font.size = Pt(9)
            doc.add_paragraph()
        elif kind == "pagebreak":
            s2 = doc.add_section(WD_SECTION.NEW_PAGE)
            fmt_section(s2)
    doc.save(path)
    return fig_n, tab_n


with safe_section("Auto report draft"):
    if not CFG.MAKE_REPORT_DRAFT:
        print("⏭️ MAKE_REPORT_DRAFT=False")
    else:
        # refresh figure index (termasuk executive summary)
        with open(os.path.join(CFG.OUTPUT_DIR, "figure_index.md"), "w", encoding="utf-8") as f:
            f.write("# Figure index\n\n| File | Title |\n|---|---|\n")
            for fn, t in FIGURE_INDEX:
                f.write(f"| {fn} | {t} |\n")
        blocks = build_report_content()
        md_path = os.path.join(CFG.OUTPUT_DIR, "REPORT_DRAFT.md")
        write_report_md(blocks, md_path)
        print("Markdown draft :", md_path)
        try:
            n_f, n_t = write_report_docx(blocks, os.path.join(CFG.OUTPUT_DIR, "REPORT_DRAFT.docx"))
            print(f"Word draft     : {os.path.join(CFG.OUTPUT_DIR, 'REPORT_DRAFT.docx')} ({n_f} figures, {n_t} tables)")
        except ImportError:
            print("python-docx tidak tersedia → hanya Markdown. (pip install python-docx, lalu run ulang cell ini)")

        # TODO list
        edits = [(i, p) for i, (k, p) in enumerate(blocks) if k == "p" and "[EDIT" in p]
        current_h = ""
        todo = [f"# REPORT TODO — {CFG.TEAM_NAME}", "",
                "Kerjakan semua item ini di `REPORT_DRAFT.docx` (bagian kuning = wajib diedit).", "",
                "## A. Bagian [EDIT] di draft", ""]
        for i, (k, p) in enumerate(blocks):
            if k == "h":
                current_h = p[0]
            if k == "p" and "[EDIT" in p:
                for m in re.findall(r"\[EDIT[^\]]*\]", p):
                    todo.append(f"- [ ] **{current_h}** — {m[6:-1].strip(': ')}")
        todo += ["", "## B. Verifikasi angka & narasi", "",
                 "- [ ] Semua angka di draft sama dengan `insights.md` / `tables/` (run terakhir, `RUN_MODE='full'`).",
                 "- [ ] Mapping target benar (churn = 1) dan tidak ada fitur leak di model (Section 5.3).",
                 f"- [ ] Parameter bisnis (margin {CFG.PROFIT_MARGIN:.0%}, cost {fmt_num(CFG.RETENTION_COST)}, success {CFG.RETENTION_SUCCESS_RATE:.0%}) sesuai soal / dinyatakan sebagai asumsi.",
                 "- [ ] Kalimat otomatis diparafrase dengan gaya tim (jangan copy mentah).",
                 "- [ ] Nama fitur di teks sudah otomatis dibuat lebih ramah (pretty labels) — cek & perbaiki; untuk label custom isi `CFG.FEATURE_LABELS` lalu run ulang cell 24.3. Figure masih memakai nama kolom asli → sebutkan di caption bila perlu.",
                 "- [ ] Driver yang dibahas = yang konsisten di evidence matrix (bukan satu metode saja).",
                 "- [ ] Rekomendasi spesifik: siapa (segmen + ukuran), apa, kapan, dampak, KPI.",
                 "- [ ] Persona diberi nama & intervensi khusus.",
                 "- [ ] Keterbatasan & asumsi disebut.",
                 "", "## C. Format (aturan soal)", "",
                 "- [ ] ≤ 10 halaman (tanpa cover & appendix) — kalau lebih: pindahkan figure ke Appendix, ringkas 3.x.",
                 "- [ ] A4, Times New Roman 12, spasi 1.15, margin atas/bawah 4 cm, kiri/kanan 3 cm (sudah di draft — jangan diubah).",
                 "- [ ] Figure/tabel bernomor urut & dirujuk di teks; caption menyatakan insight.",
                 "- [ ] Figure versi `FIG_DPI = 300`.",
                 f"- [ ] Save As PDF: `{CFG.TEAM_NAME}_Final Stage 1.pdf`.",
                 f"- [ ] ZIP: `{CFG.TEAM_NAME}_Final Stage 1.zip` berisi PDF + notebook (.ipynb) + file pendukung (cell 24.4).",
                 "", "## D. Sumber setiap bagian laporan", "",
                 "| Bagian | File output |", "|---|---|",
                 "| Data & cleaning | tables/data_quality_report.csv, tables/leakage_screen.csv |",
                 "| Drivers | tables/driver_table.csv, tables/driver_evidence_matrix.csv, figures/*driver_churn_rate* |",
                 "| Survival | tables/hazard_by_tenure_interval.csv, tables/cox_hazard_ratios.csv, figures/*km_* |",
                 "| Segments | tables/segment_rules.csv, figures/*interaction_heatmaps*, *segment_tree* |",
                 "| Interpretable model | tables/logit_odds_ratios.csv, figures/*logit_odds_ratios* |",
                 "| Model | tables/leaderboard_all.csv, tables/bootstrap_ci_final_model.csv, figures/*roc_pr*, *gains_and_lift* |",
                 "| Explainability | tables/shap_importance.csv, figures/*shap_* , *partial_dependence* |",
                 "| Business | tables/risk_tiers.csv, risk_value_quadrants.csv, campaign_optimum.csv, campaign_sensitivity.csv, what_if_scenarios.csv, churn_personas.csv |",
                 "| Recommendations | tables/recommendations.csv |",
                 "| Semua kalimat insight | insights.md |"]
        with open(os.path.join(CFG.OUTPUT_DIR, "REPORT_TODO.md"), "w", encoding="utf-8") as f:
            f.write("\n".join(todo))
        n_edit = sum(1 for line in todo if line.startswith("- [ ] **"))
        print(f"TODO list      : {os.path.join(CFG.OUTPUT_DIR, 'REPORT_TODO.md')} ({n_edit} bagian [EDIT])")
        print("\n👉 Langkah berikut: buka REPORT_TODO.md, lalu edit REPORT_DRAFT.docx di Word.")

## 24.4 Package Deliverables (ZIP sesuai aturan soal)
Setelah artikel PDF final selesai: set `CFG.ARTICLE_PDF_PATH = "path/ke/TeamName_Final Stage 1.pdf"`, `CFG.MAKE_ZIP = True`, lalu jalankan cell ini saja.
ZIP berisi: artikel PDF, notebook `.ipynb` (syntax), dan folder `supporting/` (figures, tables, insights, submission).

**Jika X → Y:** soal meminta submission terpisah → tetap upload `submission.csv` sesuai instruksi; ZIP tetap memuatnya di `supporting/`.

In [ ]:
import zipfile, glob

with safe_section("Package deliverables"):
    base = f"{CFG.TEAM_NAME}_Final Stage 1"
    if not CFG.MAKE_ZIP:
        print(f"⏭️ MAKE_ZIP=False. Setelah PDF siap: CFG.ARTICLE_PDF_PATH = '...pdf'; CFG.MAKE_ZIP = True → run cell ini.")
        print(f"   Nama file yang benar: '{base}.pdf' dan '{base}.zip'")
    else:
        zpath = os.path.join(CFG.OUTPUT_DIR, f"{base}.zip")
        nb_paths = [CFG.NOTEBOOK_PATH] if CFG.NOTEBOOK_PATH else [p for p in glob.glob("*.ipynb") if "checkpoint" not in p]
        with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
            if CFG.ARTICLE_PDF_PATH and os.path.exists(CFG.ARTICLE_PDF_PATH):
                z.write(CFG.ARTICLE_PDF_PATH, f"{base}.pdf")
                print("  + artikel:", CFG.ARTICLE_PDF_PATH)
            else:
                print("  ⚠️ ARTICLE_PDF_PATH belum diset / file tidak ada → ZIP tanpa artikel!")
            for nbp in nb_paths:
                if nbp and os.path.exists(nbp):
                    z.write(nbp, os.path.basename(nbp))
                    print("  + notebook:", nbp)
            for root, _, files in os.walk(CFG.OUTPUT_DIR):
                if os.path.abspath(root).startswith(os.path.abspath(CFG.MODEL_DIR)):
                    continue
                for fn in files:
                    full = os.path.join(root, fn)
                    if fn.endswith(".zip") or fn.startswith("REPORT_"):
                        continue
                    z.write(full, os.path.join("supporting", os.path.relpath(full, CFG.OUTPUT_DIR)))
        print(f"✅ ZIP dibuat: {zpath} ({os.path.getsize(zpath) / 1e6:.1f} MB)")

# 25. Troubleshooting, Recipes & Checklist

## 25.1 Troubleshooting (Error → Penyebab → Solusi)
| Error / Gejala | Penyebab | Solusi |
|---|---|---|
| `FileNotFoundError` | path salah | `os.listdir("./data")`, cek nama/ekstensi; path Windows pakai `r"C:\..."` atau `/` |
| Data terbaca 1 kolom | separator `;`/`|` | `smart_read(path, sep=";")` |
| `UnicodeDecodeError` | encoding | `smart_read(path, encoding="latin-1")` |
| `KeyError: TARGET_COL` | nama kolom beda (spasi/kapital) | `print(raw_train.columns.tolist())`, set `CFG.TARGET_COL` persis |
| "Target punya N nilai unik" | target bukan biner | set `CFG.POSITIVE_LABEL` (nilai/list nilai churn) |
| Churn rate terbalik (≈80%) | mapping positive salah | set `CFG.POSITIVE_LABEL` ke nilai churn |
| Kolom angka terbaca kategori | ada teks/simbol aneh | `CFG.FORCE_NUMERIC = ["kolom"]` |
| Kode wilayah dianggap numeric | memang angka | `CFG.FORCE_CATEGORICAL = ["kolom"]` |
| Role (tenure/premium) salah/tidak terdeteksi | nama kolom tidak umum | `CFG.COLUMN_ROLES = {"tenure": "...", "premium": "..."}` |
| AUC > 0.97 / "terlalu bagus" | **leakage** | cek Section 5.3, buang kolom post-event (`CFG.DROP_COLS`) |
| `LightGBMError: Do not support special JSON characters` | nama kolom | sudah otomatis disanitasi; kalau masih, rename manual |
| XGBoost `enable_categorical` / categorical error | versi lama | notebook fallback ke kode kategori otomatis; atau `pip install -U xgboost` |
| CatBoost `Invalid type for cat_feature` / NaN | NaN di kategori | sudah diisi "NA"; cek kolom baru di `custom_feature_engineering` |
| `ValueError: Input contains NaN` (sklearn) | model tidak support NaN | pakai pipeline bawaan (imputer) — jangan bypass `fit_predict` |
| `MemoryError` / sangat lambat | data besar | `RUN_MODE="fast"`, kurangi `MODELS`, `N_FOLDS=3`, `SHAP_SAMPLE=1000`, matikan RF/ET |
| Optuna lama | trial banyak | `OPTUNA_TIMEOUT=300` atau `USE_OPTUNA=False` |
| `Singular matrix` di Logit/Cox | fitur kolinear / separasi sempurna | otomatis kurangi fitur; atau buang fitur VIF tinggi |
| Cox / KM di-skip | tidak ada tenure atau statsmodels | set role tenure; `pip install statsmodels` |
| SHAP error | versi shap / model | contribs native LightGBM/XGB/CatBoost dipakai dulu; beeswarm manual fallback |
| Submission `ID mismatch` / NaN | ID beda tipe | ID dibandingkan sebagai string; cek `ID_COL` |
| Skor LB jauh < CV | format salah / drift / leakage | cek mode probability vs label, adversarial AUC, kolom leak |
| Figure tidak muncul | backend | `%matplotlib inline` di cell pertama |
| Setelah install package masih error import | kernel belum restart | Kernel → Restart, run ulang |
| Excel export gagal | openpyxl tidak ada | `pip install openpyxl` (CSV tetap tersimpan) |

## 25.2 Experiment recipes
```python
# (A) Cek cepat semua jalan
CFG.RUN_MODE = "fast"
# (B) Final kuat (waktu cukup ~30-60 menit)
CFG.RUN_MODE = "full"; CFG.USE_OPTUNA = True; CFG.OPTUNA_TRIALS = 50; CFG.N_REPEATS = 2
# (C) Waktu mepet tapi tetap kuat
CFG.RUN_MODE = "full"; CFG.USE_OPTUNA = False; CFG.MODELS = ["logreg", "lgbm", "catboost"]
# (D) Metric F1 / accuracy
CFG.METRIC = "f1"; CFG.SUBMISSION_MODE = "label"; CFG.IMBALANCE = "class_weight"
# (E) Metric logloss
CFG.METRIC = "logloss"; CFG.CALIBRATION = "auto"
# (F) Data besar (>200k baris)
CFG.MODELS = ["lgbm", "xgb"]; CFG.N_FOLDS = 3; CFG.SHAP_SAMPLE = 2000; CFG.MAX_EDA_PLOTS = 12
# (G) Ada household/agent grouping
CFG.GROUP_COL = "household_id"
# (H) GPU tersedia
CFG.USE_GPU = True
```

## 25.3 Checklist sebelum submit
- [ ] Target mapping benar (churn = 1) & churn rate masuk akal
- [ ] Tidak ada fitur leak (Section 5.3) & shuffled-target AUC ≈ 0.5
- [ ] Adversarial AUC wajar (< 0.7) atau drift sudah dijelaskan
- [ ] Final run dengan `RUN_MODE="full"` (bukan fast)
- [ ] Format submission sesuai sample (probability vs label!), validator semua ✅
- [ ] File dinamai sesuai aturan lomba (misal `TeamName_Final Stage 1`)
- [ ] Figure untuk laporan pakai `CFG.FIG_DPI = 300`
- [ ] Notebook di-run ulang dari atas (Restart & Run All) tanpa error → simpan .ipynb + output untuk lampiran
- [ ] ZIP: artikel PDF + notebook + file pendukung